# NeuralSS temporal super-resolution for games (Kaggle GPU T4 x2)

**Goal.** Train and evaluate the NeuralSS temporal upscaler: a recurrent, motion-compensated network that reconstructs native-quality frames from jittered low-resolution game renders (primary 3x, for example 640x360 to 1920x1080; also 2x), stable under heavy motion and converging to near-native detail in static scenes, small enough for integrated GPUs.

**Approach.**

1. **Data.** Uses the leakage-safe manifest from the `neural-data-pipeline` notebook when attached (otherwise builds a smaller one inline with the same code). Two training streams:
   * *Synthetic game-like sequences* rendered on the GPU from cached high-resolution stills (game frames, DIV2K, Flickr2K, GameIR, video frames): layered 2.5D scenes with independent camera and object motion, thin wires and fences, particles that write no motion vectors, world-space text, exposure changes. Low-resolution inputs are rendered like a game renders at reduced resolution (one jittered sample per pixel, Halton 2,3, no mip filtering, so thin geometry aliases); targets are 4-sample anti-aliased. Motion vectors, disocclusions and depth are exact.
   * *Real sequences*: TartanAir and Sintel with exact optical flow (played backwards so forward flow becomes the engine-style motion vector) and depth; REDS, Vimeo-90K, Vimeo-1080p and GameIR video with RAFT teacher motion (marked as estimated).
2. **Model.** Per frame, at low resolution: jittered colour, depth and derived normals, motion vectors, a depth disocclusion cue, reactive mask, exposure, jitter offset, availability flags, the previous output warped by the upsampled motion vectors and folded to low resolution (space-to-depth), and a warped recurrent hidden state. A RepConv trunk (with an optional half-resolution U-Net level) predicts a per-pixel history weight and a residual, unfolded by pixel shuffle: `out = a * warped_history + (1 - a) * jitter-aware upsample + residual`. RepConv branches fold into plain 3x3 convolutions for deployment. Tiers: `igpu`, `low`, `mid`, `high`, and a `teacher` used for distillation.
3. **Training.** DDP on both GPUs, FP16, EMA, wall-clock cosine schedule. Loss per frame: Charbonnier + gradient + Fourier magnitude + LPIPS (perceptual) + temporal-change loss along exact motion; students also match the teacher's outputs (knowledge distillation).
4. **Evaluation.** Spatial quality (PSNR / SSIM on Y, LPIPS, FLIP), temporal stability (warping error and temporal-change error along exact motion, tOF with RAFT), region metrics (disocclusions, fine detail, particles), static-scene convergence, motion-speed classes, a native-render generalisation test (GameIR 720p to 1440p), against bilinear, bicubic and an analytic TAA upscaler. Ablations: no history, no G-buffers, bicubic degradation, no temporal loss, no distillation. Latency of fused tiers at 1080p and 4K outputs, ONNX export.

**Required Kaggle settings.** Accelerator **GPU T4 x2**; **Internet on** (pip packages, Hugging Face data in the fallback path, RAFT / AlexNet / DINOv2 weights). Attach the output of the `neural-data-pipeline` notebook (**Add Input > Notebook output**) and the same Kaggle datasets it lists (`soumikrakshit/div2k-high-resolution-images`, `daehoyang/flickr2k`, `amithkesavmrajagiri/reds-dataset`, `wangsally/vimeo-90k-7`, `chenshu123/vimeo-triplet`, `artemmmtry/mpi-sintel-dataset`, `uom200647r/vid4-dataset`). Optional secret `HF_TOKEN`.

**Budget** (enforced; numbers in the configuration cell): caches about 25 min, teacher 100 min, main student 80 min, parallel `igpu` 3x and `low` 2x students 50 min, three parallel ablation pairs 20 min each, evaluation and export about 45 min: about 6.5 hours of a 9.5-hour ceiling. `QUICK_RUN = True` gives a 40-minute smoke test of every cell.

**Outputs** in `/kaggle/working/`:

```text
code/          library and training scripts written by this notebook, run configs
weights/       *_best.pt / *_last.pt per run, fused students (*.pt), ONNX models
plots/         every figure as PNG
metrics/       JSON / CSV: hardware, caches, histories, per-frame and aggregated results, ablations, latency, summary
logs/          notebook log and one log per training run
predictions/   sample output frames and temporal profiles
outputs.zip    everything above (the training caches are deleted before zipping)
```

## 1. Library files

The first cell creates `/kaggle/working/code`; the shared NeuralSS library and the DDP training script are then written into it (the repository's `src/neuralss` is their single source). Training runs as separate processes that import the same files.

In [ ]:
import os

os.makedirs(
    "/kaggle/working/code", exist_ok=True
)  # %%writefile does not create folders

**`nss_common.py`**: run folders, logger, stage timer, figure and metric saving, hardware report, pip helper and the training-process launcher.

In [ ]:
%%writefile /kaggle/working/code/nss_common.py
"""Shared notebook runtime: output folders, logging, stage timing, figure saving, hardware report, pip helper."""

import importlib
import json
import logging
import os
import shutil
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import psutil
import torch

WORK = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
SUBDIRS = ["code", "weights", "plots", "metrics", "logs", "predictions"]


class Run:
    """Holds the run clock, output folders, logger and per-stage timings of one notebook run."""

    def __init__(self, name, work=WORK):
        self.start = time.time()
        self.work = Path(work)
        self.dirs = {d: self.work / d for d in SUBDIRS}
        for folder in self.dirs.values():
            folder.mkdir(parents=True, exist_ok=True)
        self.log = logging.getLogger(name)
        self.log.setLevel(logging.INFO)
        self.log.handlers.clear()
        self.log.propagate = False
        for handler in (
            logging.StreamHandler(sys.stdout),
            logging.FileHandler(self.dirs["logs"] / "notebook.log"),
        ):
            handler.setFormatter(
                logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")
            )
            self.log.addHandler(handler)
        self.stage_times = {}

    def hours(self):
        return (time.time() - self.start) / 3600

    @contextmanager
    def stage(self, name):
        t0 = time.time()
        self.log.info("start: %s", name)
        try:
            yield
        finally:
            self.stage_times[name] = time.time() - t0
            self.log.info(
                "done: %s in %.1f min (run total %.2f h)",
                name,
                self.stage_times[name] / 60,
                self.hours(),
            )

    def show(self, fig, name):
        from IPython.display import display

        fig.savefig(self.dirs["plots"] / f"{name}.png", dpi=120, bbox_inches="tight")
        display(fig)
        plt.close(fig)

    def save_json(self, obj, name):
        with open(self.dirs["metrics"] / name, "w") as f:
            json.dump(obj, f, indent=2, default=str)

    def save_csv(self, df, name, **kwargs):
        df.to_csv(self.dirs["metrics"] / name, **kwargs)


def hardware_report(work=WORK):
    """GPU, CPU, RAM and disk detected at run time (nothing hard-coded)."""
    gpus = []
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        gpus.append(
            {
                "gpu": i,
                "name": p.name,
                "vram_gb": round(p.total_memory / 2**30, 2),
                "sm": p.multi_processor_count,
                "cc": f"{p.major}.{p.minor}",
            }
        )
    vm = psutil.virtual_memory()
    report = {
        "gpus": len(gpus),
        "gpu_list": gpus,
        "cpu_count": os.cpu_count(),
        "ram_total_gb": round(vm.total / 2**30, 1),
        "ram_available_gb": round(vm.available / 2**30, 1),
        "disk_free_gb_working": round(shutil.disk_usage(work).free / 2**30, 1),
        "disk_free_gb_tmp": (
            round(shutil.disk_usage("/tmp").free / 2**30, 1)
            if Path("/tmp").exists()
            else None
        ),
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
    }
    return report


def pip_install(packages, log=None):
    """Install missing packages quietly; returns {package: importable}. Needs Kaggle Internet on."""
    status = {}
    for pip_name, module in packages.items():
        try:
            importlib.import_module(module)
            status[pip_name] = True
            continue
        except ImportError:
            pass
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            capture_output=True,
            text=True,
        )
        try:
            importlib.invalidate_caches()
            importlib.import_module(module)
            status[pip_name] = True
        except ImportError:
            status[pip_name] = False
            if log:
                log.warning(
                    "could not install %s: %s", pip_name, result.stderr.strip()[-300:]
                )
    return status


def gpu_utilisation():
    try:
        out = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=10,
        ).stdout
        return [
            [float(v) for v in line.split(",")] for line in out.strip().splitlines()
        ]
    except Exception:
        return []


def stream_process(cmd, env, on_line, log_path):
    """Run a subprocess, call on_line for every stdout line, mirror everything into log_path; returns exit code."""
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in proc.stdout:
            logf.write(line)
            logf.flush()
            on_line(line.rstrip("\n"))
        return proc.wait()


def launch_training(
    script, cfg, cfg_path, log_path, gpus, desc, port=29500, on_round=None
):
    """Run a training script on the given GPUs (torch.distributed.run for several GPUs, a plain process for one),
    write its config, mirror its output to log_path, drive a tqdm bar from PROGRESS lines and collect ROUND rows.
    Returns (rounds, done). Raises with the log tail if the process fails."""
    from tqdm.auto import tqdm

    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    env = os.environ.copy()
    env.update(
        {
            "NCCL_P2P_DISABLE": "1",
            "OMP_NUM_THREADS": "1",
            "PYTHONUNBUFFERED": "1",
            "CUDA_VISIBLE_DEVICES": ",".join(str(g) for g in gpus),
        }
    )
    if len(gpus) > 1:
        cmd = [
            sys.executable,
            "-m",
            "torch.distributed.run",
            f"--nproc_per_node={len(gpus)}",
            f"--master_port={port}",
            str(script),
            "--config",
            str(cfg_path),
        ]
    else:
        env["CUDA_DEVICE"] = "0"
        cmd = [sys.executable, str(script), "--config", str(cfg_path)]
    rounds, done, tail = [], {}, []
    bar = tqdm(
        total=100,
        desc=desc,
        unit="%",
        bar_format="{l_bar}{bar}| {n:.0f}/{total:.0f}% [{elapsed}<{remaining}] {postfix}",
    )

    def on_line(line):
        tail.append(line)
        del tail[:-40]
        kind, _, payload = line.partition(" ")
        if kind in ("PROGRESS", "ROUND", "DONE", "LOG"):
            try:
                data = json.loads(payload)
            except ValueError:
                return
            if kind == "PROGRESS":
                bar.n = round(100 * data["progress"], 1)
                bar.set_postfix(
                    loss=f"{data['loss']:.4f}",
                    sps=f"{data['samples_per_s']:.0f}",
                    lr=f"{data['lr']:.1e}",
                    mem=data["mem_gb"],
                )
                bar.refresh()
            elif kind == "ROUND":
                rounds.append(data)
                if on_round:
                    on_round(rounds)
            elif kind == "DONE":
                done.update(data)
            else:
                print(f"[{desc}] {data.get('msg')}", flush=True)
        elif "Error" in line or "Traceback" in line:
            print(f"[{desc}] {line}", flush=True)

    code = stream_process(cmd, env, on_line, log_path)
    bar.n = 100
    bar.close()
    if code != 0:
        raise RuntimeError(
            f"{desc} failed with exit code {code}; last lines:\n"
            + "\n".join(tail[-15:])
        )
    return rounds, done

**`nss_data.py`**: source registry (Kaggle and Hugging Face), capped discovery, HTTP range-request access to remote archives, Hugging Face materialisation, decoding of colour / depth / flow / masks, window and crop workers, manifest I/O.

In [ ]:
%%writefile /kaggle/working/code/nss_data.py
"""Unified dataset layer: source registry (Kaggle + Hugging Face), discovery, filtering, materialisation of
Hugging Face subsets, sequence building, decoding of colour / depth / flow / masks, and window extraction.

URI scheme used in the manifest (so it stays valid across notebooks and machines):
  kaggle:<owner/slug>|<path inside the dataset>   resolved under /kaggle/input
  data:<relative path>                            resolved under the data root (materialised Hugging Face subsets)
  mp4:<relative path>#<frame index>               a frame of a video file under the data root
"""

import hashlib
import io
import json
import os
import random
import re
import tarfile
import time
import urllib.parse
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff", ".ppm"}
# Path tokens that mark degraded copies or non-colour passes; such files are never used as targets.
LOW_QUALITY_TOKEN = re.compile(
    r"^(lr\d*|lq|low|lowres|bicubic|bix\d|bdx\d|lrbi|lrbd|x\d|blur|blurred|noisy|compressed|jpeg)$"
)
NON_COLOUR_TOKENS = {
    "occlusions",
    "occlusion",
    "invalid",
    "flow",
    "flow_viz",
    "viz",
    "mask",
    "masks",
    "depth",
    "albedo",
    "shading",
    "disparities",
    "segmentation",
    "camdata",
    "camdata_left",
    "edges",
    "normals",
    "seg",
}
HR_TOKENS = {"hr", "gt", "original", "groundtruth"}
HF = "https://huggingface.co"

# ---------------------------------------------------------------------------------------------------------------
# Source registry. Every source states where it lives, what it provides and how it may be used.
# role: train sources are split into train/val by leakage group; test sources (or test parts) are pinned to test.
# ---------------------------------------------------------------------------------------------------------------
SOURCES = [
    {
        "name": "div2k",
        "host": "kaggle",
        "ref": "soumikrakshit/div2k-high-resolution-images",
        "kind": "still",
        "test_dirs": ["DIV2K_valid_HR"],
        "licence": "DIV2K: academic research use",
        "content": "photo",
    },
    {
        "name": "flickr2k",
        "host": "kaggle",
        "ref": "daehoyang/flickr2k",
        "kind": "still",
        "licence": "Flickr2K: research use (Flickr images, original licences)",
        "content": "photo",
    },
    {
        "name": "reds",
        "host": "kaggle",
        "ref": "amithkesavmrajagiri/reds-dataset",
        "kind": "video",
        "group_level": 1,
        "licence": "REDS: CC BY 4.0",
        "content": "video",
    },
    {
        "name": "vimeo_septuplet",
        "host": "kaggle",
        "ref": "wangsally/vimeo-90k-7",
        "kind": "video",
        "group_level": 2,
        "licence": "Vimeo-90K: research use (Vimeo uploads)",
        "content": "video",
    },
    {
        "name": "vimeo_triplet",
        "host": "kaggle",
        "ref": "chenshu123/vimeo-triplet",
        "kind": "video",
        "group_level": 2,
        "test_list": "tri_testlist.txt",
        "test_only": True,
        "list_cap": 6000,
        "max_items": 1000,
        "licence": "Vimeo-90K: research use",
        "content": "video",
    },
    {
        "name": "sintel",
        "host": "kaggle",
        "ref": "artemmmtry/mpi-sintel-dataset",
        "kind": "render_seq",
        "test_groups": ["market", "cave"],
        "licence": "MPI Sintel: CC BY 3.0 (Kaggle mirror)",
        "content": "rendered",
    },
    {
        "name": "vid4",
        "host": "kaggle",
        "ref": "uom200647r/vid4-dataset",
        "kind": "video",
        "group_level": 1,
        "test_only": True,
        "licence": "Vid4: research benchmark (Kaggle mirror Apache 2.0)",
        "content": "video",
    },
    {
        "name": "game_stills",
        "host": "hf",
        "repo": "ericphann/video-game-super-resolution",
        "kind": "still",
        "train_dir": "training-hr",
        "test_dir": "test-hr",
        "licence": "HF tag Apache 2.0 (game captures; check per use)",
        "content": "game",
    },
    {
        "name": "tartanair",
        "host": "hf",
        "repo": "theairlabcmu/tartanair",
        "kind": "render_seq",
        "train_envs": [
            "abandonedfactory",
            "amusement",
            "carwelding",
            "endofworld",
            "gascola",
            "hospital",
            "neighborhood",
            "oldtown",
            "seasonsforest",
            "soulcity",
            "westerndesert",
            "ocean",
        ],
        "test_envs": ["japanesealley", "seasidetown"],
        "licence": "TartanAir: BSD-3-Clause (HF tag and tools)",
        "content": "rendered",
    },
    {
        "name": "gameir",
        "host": "hf",
        "repo": "LLLebin/GameIR",
        "kind": "render_pairs",
        "train_tars": ["mini_dataset/train/GameIR-SR/GameIR-SR-000000.tar"],
        "test_tars": ["mini_dataset/test/GameIR-SR/GameIR-SR-000000.tar"],
        "licence": "GameIR: MIT",
        "content": "game",
    },
    {
        "name": "vimeo1080p",
        "host": "hf",
        "repo": "danjacobellis/vimeo1080p",
        "kind": "video_file",
        "train_shards": [
            "data/train-00000-of-00045.parquet",
            "data/train-00001-of-00045.parquet",
        ],
        "val_shards": ["data/validation-00000-of-00003.parquet"],
        "licence": "unspecified on HF (Vimeo uploads); research use",
        "content": "video",
    },
]


def source(name):
    return next(s for s in SOURCES if s["name"] == name)


# ---------------------------------------------------------------------------------------------------------------
# Path utilities (Kaggle mounts)
# ---------------------------------------------------------------------------------------------------------------
def tokens(text):
    return [t for t in re.split(r"[_\-\s.]+", text.lower()) if t]


def path_tokens(path, root):
    rel = Path(path).relative_to(root)
    parts = list(rel.parts[:-1]) + [Path(rel.parts[-1]).stem]
    return [t for part in parts for t in tokens(part)]


def is_clean_colour(path, root):
    return not any(
        LOW_QUALITY_TOKEN.match(t) or t in NON_COLOUR_TOKENS
        for t in path_tokens(path, root)
    )


def find_dataset_root(input_root, owner_slug):
    """Mount folder of a Kaggle dataset under either supported layout, or None."""
    owner, slug = owner_slug.split("/")
    input_root = Path(input_root)
    for candidate in (
        input_root / slug,
        input_root / "datasets" / owner / slug,
        input_root / "datasets" / slug,
    ):
        if candidate.is_dir():
            return candidate
    for depth_one in input_root.iterdir() if input_root.is_dir() else []:
        if depth_one.is_dir():
            for depth_two in [depth_one] + [
                p for p in depth_one.iterdir() if p.is_dir()
            ]:
                if depth_two.name == slug:
                    return depth_two
    return None


def sampled_walk(root, max_files, seed, exts=IMG_EXTS):
    """List files, expanding a random directory from the frontier each step, so a capped listing spans many
    sequences. Each directory is listed completely, so a listed sequence folder is never cut in half.
    """
    rng = random.Random(seed)
    frontier, files, dirs_listed = [str(root)], [], 0
    while frontier and len(files) < max_files:
        i = rng.randrange(len(frontier))
        frontier[i], frontier[-1] = frontier[-1], frontier[i]
        current = frontier.pop()
        try:
            entries = list(os.scandir(current))
        except OSError:
            continue
        dirs_listed += 1
        for entry in entries:
            if entry.name.startswith("."):
                continue
            if entry.is_dir(follow_symlinks=False):
                frontier.append(entry.path)
            elif os.path.splitext(entry.name)[1].lower() in exts:
                files.append(entry.path)
    return sorted(files), dirs_listed, len(frontier) == 0


def natural_key(path):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(path).lower())]


def stable_hash(text):
    return int(hashlib.md5(text.encode()).hexdigest(), 16)


# ---------------------------------------------------------------------------------------------------------------
# Hugging Face access: plain HTTPS with retries and range requests (no local Hub cache, works for Xet-backed files)
# ---------------------------------------------------------------------------------------------------------------
def _headers():
    token = os.environ.get("HF_TOKEN")
    return {"Authorization": f"Bearer {token}"} if token else {}


def hf_url(repo, path):
    return f"{HF}/datasets/{repo}/resolve/main/{urllib.parse.quote(path)}"


def http_get(url, headers=None, retries=4, timeout=120):
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={**_headers(), **(headers or {})})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read(), r.headers
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2**attempt)


def http_download(url, dest, retries=4):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=_headers())
            with urllib.request.urlopen(req, timeout=300) as r, open(tmp, "wb") as f:
                while True:
                    chunk = r.read(1 << 22)
                    if not chunk:
                        break
                    f.write(chunk)
            tmp.rename(dest)
            return dest
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2**attempt)


def hf_list(repo, path=""):
    """All entries under a dataset folder (follows pagination)."""
    url = f"{HF}/api/datasets/{repo}/tree/main/{urllib.parse.quote(path)}"
    out = []
    while url:
        body, headers = http_get(url)
        out.extend(json.loads(body))
        link = headers.get("Link") or ""
        m = re.search(r'<([^>]+)>;\s*rel="next"', link)
        url = m.group(1) if m else None
    return out


class HttpRangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests, so zip members can be read without downloading archives."""

    def __init__(self, url):
        self.url, self.pos = url, 0
        req = urllib.request.Request(url, method="HEAD", headers=_headers())
        with urllib.request.urlopen(req, timeout=60) as r:
            self.size = int(r.headers["Content-Length"])
            self.url = r.url

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        data, _ = http_get(
            self.url, headers={"Range": f"bytes={self.pos}-{self.pos + n - 1}"}
        )
        self.pos += len(data)
        return data

    def readinto(self, b):
        data = self.read(len(b))
        b[: len(data)] = data
        return len(data)


def remote_zip(url):
    return zipfile.ZipFile(io.BufferedReader(HttpRangeFile(url), buffer_size=1 << 20))


# ---------------------------------------------------------------------------------------------------------------
# Decoding
# ---------------------------------------------------------------------------------------------------------------
class Resolver:
    """Maps manifest URIs to local paths."""

    def __init__(self, data_root, input_root="/kaggle/input"):
        self.data_root, self.input_root, self.roots = (
            Path(data_root),
            Path(input_root),
            {},
        )

    def kaggle_root(self, ref):
        if ref not in self.roots:
            self.roots[ref] = find_dataset_root(self.input_root, ref)
        return self.roots[ref]

    def path(self, uri):
        if uri.startswith("kaggle:"):
            ref, rel = uri[7:].split("|", 1)
            root = self.kaggle_root(ref)
            if root is None:
                raise FileNotFoundError(f"Kaggle dataset {ref} is not attached")
            return root / rel
        if uri.startswith("data:"):
            return self.data_root / uri[5:]
        if uri.startswith("mp4:"):
            return self.data_root / uri[4:].split("#")[0]
        raise ValueError(uri)


def kaggle_uri(ref, root, path):
    return f"kaggle:{ref}|{Path(path).relative_to(root).as_posix()}"


def read_video_frames(path, indices):
    import cv2

    cap = cv2.VideoCapture(str(path))
    frames, want = {}, sorted(set(indices))
    if want:
        cap.set(cv2.CAP_PROP_POS_FRAMES, want[0])
        idx = want[0]
        while idx <= want[-1]:
            ok, frame = cap.read()
            if not ok:
                break
            if idx in want:
                frames[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            idx += 1
    cap.release()
    missing = [i for i in indices if i not in frames]
    if missing:
        raise IOError(f"could not decode frames {missing[:3]} of {path}")
    return [frames[i] for i in indices]


def video_frame_count(path):
    import cv2

    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


def load_rgb(resolver, uri):
    if uri.startswith("mp4:"):
        return read_video_frames(resolver.path(uri), [int(uri.split("#")[1])])[0]
    with Image.open(resolver.path(uri)) as im:
        return np.asarray(im.convert("RGB"))


def load_rgb_many(resolver, uris):
    """Decode a list of frames, reading each video file once."""
    if (
        uris
        and all(u.startswith("mp4:") for u in uris)
        and len({u.split("#")[0] for u in uris}) == 1
    ):
        return read_video_frames(
            resolver.path(uris[0]), [int(u.split("#")[1]) for u in uris]
        )
    return [load_rgb(resolver, u) for u in uris]


def read_flo(path):
    """Middlebury .flo (Sintel): float32 (H, W, 2)."""
    with open(path, "rb") as f:
        magic = np.frombuffer(f.read(4), np.float32)[0]
        assert abs(magic - 202021.25) < 1e-3, f"bad .flo magic in {path}"
        w, h = np.frombuffer(f.read(8), np.int32)
        return np.frombuffer(f.read(), np.float32).reshape(h, w, 2).copy()


def load_flow(resolver, uri):
    p = resolver.path(uri)
    return read_flo(p) if p.suffix == ".flo" else np.load(p).astype(np.float32)


def load_valid(resolver, uri, kind):
    """Validity of the motion vector of each pixel (1 = the pixel is visible in the previous frame)."""
    p = resolver.path(uri)
    with Image.open(p) as im:
        a = np.asarray(im.convert("L"))
    return (
        (a < 128).astype(np.uint8)
        if kind == "occlusion"
        else (a >= 128).astype(np.uint8)
    )


def decode_carla_depth(rgba):
    """CARLA depth PNG (R + 256 G + 65536 B) / (2^24 - 1) * 1000 m."""
    rgb = rgba[..., :3].astype(np.float64)
    norm = (rgb[..., 0] + rgb[..., 1] * 256.0 + rgb[..., 2] * 65536.0) / (256.0**3 - 1)
    return (norm * 1000.0).astype(np.float32)


def load_depth(resolver, uri):
    p = resolver.path(uri)
    if p.suffix == ".npy":
        return np.load(p).astype(np.float32)
    with Image.open(p) as im:
        return decode_carla_depth(np.asarray(im.convert("RGBA")))


def normalise_depth(d):
    """Unit-free inverse-depth style encoding in [0, 1] (1 = near): 1 / (1 + d / median)."""
    d = np.where(np.isfinite(d) & (d > 0), d, np.nan)
    med = np.nanmedian(d) if np.isfinite(d).any() else 1.0
    out = 1.0 / (1.0 + np.nan_to_num(d, nan=1e9) / max(med, 1e-6))
    return out.astype(np.float32)


# ---------------------------------------------------------------------------------------------------------------
# Catalogue builders: each returns a list of item dicts (one per still image or per sequence)
# ---------------------------------------------------------------------------------------------------------------
def _item(src, kind, group, frames, role, **extra):
    item_id = (
        f"{src['name']}/{group}/{hashlib.md5(frames[0].encode()).hexdigest()[:10]}"
    )
    return {
        "item_id": item_id,
        "source": src["name"],
        "kind": kind,
        "group": f"{src['name']}:{group}",
        "role": role,
        "frames": frames,
        "n_frames": len(frames),
        "content": src.get("content", ""),
        **extra,
    }


def catalogue_kaggle(src, input_root, cap, seed):
    """Stills and frame folders of a Kaggle-mounted source, with the degraded-copy filter."""
    root = find_dataset_root(input_root, src["ref"])
    if root is None:
        return [], {"found": False}
    files, dirs_listed, complete = sampled_walk(root, src.get("list_cap", cap), seed)
    kept = [f for f in files if is_clean_colour(f, root)]
    if src["kind"] in ("video",) and any(
        any(t in HR_TOKENS for t in path_tokens(f, root)) for f in kept
    ):
        kept = [f for f in kept if any(t in HR_TOKENS for t in path_tokens(f, root))]
    items = []
    if src["kind"] == "still":
        for f in kept:
            rel = Path(f).relative_to(root)
            role = (
                "test"
                if any(d in rel.parts for d in src.get("test_dirs", []))
                else "train"
            )
            items.append(
                _item(
                    src, "still", Path(f).stem, [kaggle_uri(src["ref"], root, f)], role
                )
            )
    elif src["kind"] == "video":
        test_set = set()
        if src.get("test_list"):
            lists = sorted(Path(root).rglob(src["test_list"]))
            if lists:
                test_set = {
                    line.strip()
                    for line in lists[0].read_text().splitlines()
                    if line.strip()
                }
        folders = {}
        for f in kept:
            folders.setdefault(str(Path(f).parent), []).append(f)
        for folder, frames in folders.items():
            frames = sorted(frames, key=natural_key)
            if len(frames) < 3:
                continue
            p = Path(folder)
            if p.name.lower() in HR_TOKENS:
                p = p.parent
            level = src.get("group_level", 1)
            group = p.name if level == 1 else p.parent.name
            clip_key = f"{p.parent.name}/{p.name}"
            if src.get("test_only"):
                if test_set and clip_key not in test_set:
                    continue
                role = "test"
            else:
                role = "train"
            items.append(
                _item(
                    src,
                    "seq",
                    group,
                    [kaggle_uri(src["ref"], root, f) for f in frames],
                    role,
                    mv="none",
                )
            )
    elif src["name"] == "sintel":
        items = catalogue_sintel(src, root)
    if src.get("max_items") and len(items) > src["max_items"]:
        items = random.Random(seed).sample(items, src["max_items"])
    return items, {
        "found": True,
        "listed": len(files),
        "kept_files": len(kept),
        "dirs_listed": dirs_listed,
        "listing_complete": complete,
    }


def catalogue_sintel(src, root):
    """Sintel training scenes with exact forward flow and occlusion masks. The leakage group is the scene prefix
    (alley, ambush, market, ...) because numbered scenes share sets and characters."""
    items = []
    for pass_dir in sorted(Path(root).rglob("training/clean")) + sorted(
        Path(root).rglob("training/final")
    ):
        train_root = pass_dir.parent
        for scene in sorted(p for p in pass_dir.iterdir() if p.is_dir()):
            frames = sorted(scene.glob("*.png"), key=natural_key)
            flows = [
                train_root / "flow" / scene.name / (f.stem + ".flo") for f in frames
            ]
            occ = [train_root / "occlusions" / scene.name / f.name for f in frames]
            if len(frames) < 4 or not flows[0].exists():
                continue
            prefix = scene.name.split("_")[0]
            role = "test" if prefix in src.get("test_groups", []) else "train"
            n = len(frames) - 1  # the last frame has no forward flow
            items.append(
                _item(
                    src,
                    "seq",
                    prefix,
                    [kaggle_uri(src["ref"], root, f) for f in frames[:n]],
                    role,
                    mv="exact_forward",
                    flows=[kaggle_uri(src["ref"], root, f) for f in flows[:n]],
                    valid=[kaggle_uri(src["ref"], root, f) for f in occ[:n]],
                    valid_kind="occlusion",
                    variant=pass_dir.name,
                )
            )
    return items


# ---------------------------------------------------------------------------------------------------------------
# Hugging Face materialisation (download only what is needed into the data root)
# ---------------------------------------------------------------------------------------------------------------
def materialise_game_stills(src, data_root, n_train, n_test, seed, log=print):
    rng = random.Random(seed)
    items = []
    for role, folder, n in (
        ("train", src["train_dir"], n_train),
        ("test", src["test_dir"], n_test),
    ):
        names = sorted(
            e["path"]
            for e in hf_list(src["repo"], folder)
            if e["type"] == "file" and e["path"].endswith(".png")
        )
        chosen = sorted(rng.sample(names, min(n, len(names))))
        dests = [Path(data_root) / "game_stills" / p for p in chosen]
        with ThreadPoolExecutor(8) as pool:
            list(
                pool.map(
                    lambda pd: http_download(hf_url(src["repo"], pd[0]), pd[1]),
                    zip(chosen, dests),
                )
            )
        for p in chosen:
            items.append(
                _item(src, "still", Path(p).stem, [f"data:game_stills/{p}"], role)
            )
        log(f"game_stills {role}: {len(chosen)} of {len(names)} images")
    return items


def _tartanair_env(src, data_root, env, role, frames_per_env, seq_len, difficulty):
    """One environment: read a contiguous run of frames of the first trajectory from the remote zips (range
    requests only) and store RGB PNG, depth (fp16 npy), forward flow (fp16 npy) and validity (PNG, 255 = valid).
    TartanAir mask values: 0 = valid correspondence, non-zero = occluded or out of view.
    """
    base = f"{env}/{difficulty}"
    zips = {
        k: remote_zip(hf_url(src["repo"], f"{base}/{k}.zip"))
        for k in ("image_left", "depth_left", "flow_flow", "flow_mask")
    }
    trajs = sorted(
        {n.split("/")[2] for n in zips["image_left"].namelist() if n.count("/") >= 4}
    )
    traj = trajs[0]
    names = {
        k: {
            Path(n).name: n
            for n in z.namelist()
            if f"/{traj}/" in n and not n.endswith("/")
        }
        for k, z in zips.items()
    }
    frame_ids = sorted(
        int(n[:6]) for n in names["image_left"] if n.endswith("_left.png")
    )
    start = (
        frame_ids[len(frame_ids) // 4]
        if len(frame_ids) > 2 * frames_per_env
        else frame_ids[0]
    )
    run = [i for i in frame_ids if start <= i < start + frames_per_env]
    out_dir = Path(data_root) / "tartanair" / env / traj
    for sub in ("image", "depth", "flow"):
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    have = []
    for i in run:
        fid, nid = f"{i:06d}", f"{i + 1:06d}"
        f_key, m_key = f"{fid}_{nid}_flow.npy", f"{fid}_{nid}_mask.npy"
        if f_key not in names["flow_flow"] or m_key not in names["flow_mask"]:
            break  # keep the run contiguous
        if not (out_dir / "flow" / f"{fid}_valid.png").exists():
            (out_dir / "image" / f"{fid}.png").write_bytes(
                zips["image_left"].read(names["image_left"][f"{fid}_left.png"])
            )
            d = np.load(
                io.BytesIO(
                    zips["depth_left"].read(
                        names["depth_left"][f"{fid}_left_depth.npy"]
                    )
                )
            )
            np.save(
                out_dir / "depth" / f"{fid}.npy", np.minimum(d, 1e4).astype(np.float16)
            )
            fl = np.load(io.BytesIO(zips["flow_flow"].read(names["flow_flow"][f_key])))
            np.save(out_dir / "flow" / f"{fid}.npy", fl.astype(np.float16))
            m = np.load(io.BytesIO(zips["flow_mask"].read(names["flow_mask"][m_key])))
            Image.fromarray(np.where(m == 0, 255, 0).astype(np.uint8)).save(
                out_dir / "flow" / f"{fid}_valid.png"
            )
        have.append(i)
    items, rel = [], f"tartanair/{env}/{traj}"
    for k in range(0, len(have) - seq_len + 1, seq_len):
        seg = have[k : k + seq_len]
        items.append(
            _item(
                src,
                "seq",
                env,
                [f"data:{rel}/image/{i:06d}.png" for i in seg],
                role,
                mv="exact_forward",
                flows=[f"data:{rel}/flow/{i:06d}.npy" for i in seg],
                valid=[f"data:{rel}/flow/{i:06d}_valid.png" for i in seg],
                valid_kind="valid",
                depth=[f"data:{rel}/depth/{i:06d}.npy" for i in seg],
            )
        )
    return items, len(have), traj


def materialise_tartanair(
    src,
    data_root,
    frames_per_env,
    envs_train,
    envs_test,
    seq_len,
    difficulty="Easy",
    workers=6,
    log=print,
):
    """Environments are fetched in parallel threads (one zip set per thread; zip readers are not shared)."""
    jobs = [
        (env, "test" if env in envs_test else "train") for env in envs_train + envs_test
    ]

    def run(job):
        env, role = job
        try:
            return (
                env,
                role,
                _tartanair_env(
                    src, data_root, env, role, frames_per_env, seq_len, difficulty
                ),
                None,
            )
        except Exception as exc:
            return env, role, None, exc

    items = []
    with ThreadPoolExecutor(workers) as pool:
        for env, role, result, exc in pool.map(run, jobs):
            if exc is not None:
                log(f"tartanair {env}: skipped ({exc})")
                continue
            env_items, n, traj = result
            items.extend(env_items)
            log(
                f"tartanair {env} ({role}): {n} frames from {traj}, {len(env_items)} sequences"
            )
    return items


def materialise_gameir(src, data_root, max_clips_train, max_clips_test, log=print):
    """Stream GameIR-SR tars (sequential read, stops early) keeping native 720p and 1440p RGB and depth.
    Frames of a clip are sampled every 10 rendered frames; the leakage group is the town.
    """
    items = []
    for role, tars, cap in (
        ("train", src["train_tars"], max_clips_train),
        ("test", src["test_tars"], max_clips_test),
    ):
        clips = {}
        for tar_path in tars:
            try:
                stream = io.BufferedReader(
                    HttpRangeFile(hf_url(src["repo"], tar_path)), buffer_size=1 << 22
                )
                tf = tarfile.open(fileobj=stream, mode="r|")
            except Exception as exc:
                log(f"gameir {tar_path}: skipped ({exc})")
                continue
            for m in tf:
                parts = m.name.split("/")
                if (
                    len(parts) < 5
                    or not m.isfile()
                    or not (
                        m.name.endswith(".rgb.png") or m.name.endswith(".depth.png")
                    )
                ):
                    continue
                town, clip, res = parts[1], parts[2], parts[3]
                key = (town, clip)
                if key not in clips and len(clips) >= cap:
                    break
                clips.setdefault(key, set()).add(res)
                dst = Path(data_root) / "gameir" / town / clip / res / parts[4]
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    dst.write_bytes(tf.extractfile(m).read())
            tf.close()
        for (town, clip), res in sorted(clips.items()):
            folder = Path(data_root) / "gameir" / town / clip
            hr = sorted((folder / "1440p").glob("*.rgb.png"), key=natural_key)
            lr = sorted((folder / "720p").glob("*.rgb.png"), key=natural_key)
            if len(hr) < 3 or len(lr) != len(hr):
                continue
            rel = f"gameir/{town}/{clip}"
            items.append(
                _item(
                    src,
                    "seq",
                    town.split("_")[-1],
                    [f"data:{rel}/1440p/{p.name}" for p in hr],
                    role,
                    mv="none",
                    native_lr=[f"data:{rel}/720p/{p.name}" for p in lr],
                    depth=[
                        f"data:{rel}/1440p/{p.name.replace('.rgb.png', '.depth.png')}"
                        for p in hr
                    ],
                    frame_stride=10,
                )
            )
        log(f"gameir {role}: {sum(1 for i in items if i['role'] == role)} clips")
    return items


def materialise_vimeo1080p(src, data_root, n_train, n_val, log=print):
    """Download parquet shards, write each embedded MP4 to disk and register it as a sequence of frames."""
    import pyarrow.parquet as pq

    items = []
    for role, shards, cap in (
        ("train", src["train_shards"], n_train),
        ("val_pinned", src["val_shards"], n_val),
    ):
        written = 0
        for shard in shards:
            if written >= cap:
                break
            local = http_download(
                hf_url(src["repo"], shard), Path(data_root) / "_downloads" / shard
            )
            table = pq.ParquetFile(local)
            for batch in table.iter_batches(
                batch_size=16, columns=["video_file", "video"]
            ):
                for name, blob in zip(
                    batch.column(0).to_pylist(), batch.column(1).to_pylist()
                ):
                    if written >= cap:
                        break
                    stem = Path(name).stem
                    rel = f"vimeo1080p/{role}/{stem}.mp4"
                    dst = Path(data_root) / rel
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    dst.write_bytes(blob)
                    n = video_frame_count(dst)
                    if n < 8:
                        dst.unlink()
                        continue
                    items.append(
                        _item(
                            src,
                            "seq",
                            stem,
                            [f"mp4:{rel}#{i}" for i in range(n)],
                            "test" if role == "val_pinned" else "train",
                            mv="none",
                        )
                    )
                    written += 1
            local.unlink()  # the shard is only a container
        log(f"vimeo1080p {role}: {written} videos")
    return items


# ---------------------------------------------------------------------------------------------------------------
# Item statistics for filtering (run in worker processes)
# ---------------------------------------------------------------------------------------------------------------
def keyframes(item):
    """Frames used to represent an item in deduplication: the still itself, the middle frame of a short clip,
    or first, middle and last frame of a long sequence (scene content can change within it).
    """
    frames, n = item["frames"], item["n_frames"]
    if n == 1:
        return [frames[0]]
    if n >= 24:
        return [frames[0], frames[n // 2], frames[-1]]
    return [frames[n // 2]]


def thumb_and_stats(task):
    """Worker: decode one frame, return a 256x256 thumbnail and quality statistics."""
    data_root, uri = task
    resolver = Resolver(data_root)
    try:
        img = load_rgb(resolver, uri)
    except Exception as exc:
        return uri, None, {"error": str(exc)[:200]}
    h, w = img.shape[:2]
    grey = img.mean(axis=2)
    rows = grey.mean(axis=1)
    dark = rows < 8
    letterbox = (
        (np.argmax(~dark) + np.argmax(~dark[::-1])) / h if (~dark).any() else 1.0
    )
    small = np.asarray(Image.fromarray(img).resize((256, 256), Image.BILINEAR))
    g = small.mean(axis=2).astype(np.float32)
    lap = g[1:-1, 1:-1] * 4 - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    rg, yb = (
        small[..., 0].astype(np.float32) - small[..., 1],
        0.5 * (small[..., 0].astype(np.float32) + small[..., 1]) - small[..., 2],
    )
    stats = {
        "width": w,
        "height": h,
        "mean": float(grey.mean()),
        "std": float(grey.std()),
        "sharpness": float(lap.var()),
        "letterbox": float(letterbox),
        "colourfulness": float(
            np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())
        ),
    }
    return uri, small, stats


# ---------------------------------------------------------------------------------------------------------------
# Window extraction for training caches (run in worker processes)
# ---------------------------------------------------------------------------------------------------------------
def _resize(a, size, mode):
    """Resize an array to size = (W, H). Modes: 'image' (uint8 HxWx3, Lanczos), 'nearest' (uint8 HxW),
    'depth' (float HxW, bilinear), 'flow' (float HxWx2, bilinear, vectors rescaled to the new pixel units).
    """
    h, w = a.shape[:2]
    W, H = size
    if (W, H) == (w, h):
        return a
    if mode == "image":
        return np.asarray(Image.fromarray(a).resize((W, H), Image.LANCZOS))
    if mode == "nearest":
        return np.asarray(Image.fromarray(a).resize((W, H), Image.NEAREST))
    if mode == "depth":
        return np.asarray(
            Image.fromarray(a.astype(np.float32), mode="F").resize(
                (W, H), Image.BILINEAR
            )
        )
    if mode == "flow":
        chans = [
            np.asarray(
                Image.fromarray(a[..., c].astype(np.float32), mode="F").resize(
                    (W, H), Image.BILINEAR
                )
            )
            for c in range(2)
        ]
        return np.stack(chans, -1) * np.array([W / w, H / h], np.float32)
    raise ValueError(mode)


def extract_window(task):
    """Worker: decode a window of T frames (plus exact motion, validity and depth when the source has them),
    down-scale by a random factor, take one P x P crop at the same place in all frames, and return motion,
    validity and depth at half resolution.

    Convention: mv[k] is, for every pixel of network frame k, the displacement to its position in network frame
    k-1, in full-resolution (P-grid) pixels (a game engine's motion vector). Sources with exact forward flow
    (TartanAir, Sintel) are played backwards so that their forward flow becomes exactly this vector.
    """
    data_root, item, start, T, P, scale_range, seed = task
    rng = np.random.default_rng(seed)
    resolver = Resolver(data_root)
    exact = item.get("mv") == "exact_forward"
    try:
        idx = list(range(start, start + T))
        frames = load_rgb_many(resolver, [item["frames"][i] for i in idx])
        h, w = frames[0].shape[:2]
        if any(f.shape[:2] != (h, w) for f in frames):
            return seed, None
        flows = [load_flow(resolver, item["flows"][i]) for i in idx] if exact else None
        valids = (
            [
                load_valid(resolver, item["valid"][i], item.get("valid_kind", "valid"))
                for i in idx
            ]
            if exact
            else None
        )
        depths = (
            [load_depth(resolver, item["depth"][i]) for i in idx]
            if item.get("depth")
            else None
        )
    except Exception:
        return seed, None
    r = min(1.0, max(float(rng.uniform(*scale_range)), P / min(h, w) * 1.02))
    W, H = int(round(w * r)), int(round(h * r))
    if min(W, H) < P:
        return seed, None
    frames = [_resize(f, (W, H), "image") for f in frames]
    y, x = int(rng.integers(0, H - P + 1)), int(rng.integers(0, W - P + 1))
    crop = (slice(y, y + P), slice(x, x + P))
    half = (P // 2, P // 2)
    out = {
        "rgb": np.stack([f[crop] for f in frames]),
        "mv": np.zeros((T, P // 2, P // 2, 2), np.float16),
        "valid": np.zeros((T, P // 2, P // 2), np.uint8),
        "depth": np.zeros((T, P // 2, P // 2), np.float16),
        "has_depth": depths is not None,
        "mv_quality": 0.0,
    }
    if depths is not None:
        depths = [_resize(d, (W, H), "depth") for d in depths]
        out["depth"] = np.stack(
            [_resize(normalise_depth(d[crop]), half, "depth") for d in depths]
        ).astype(np.float16)
    if exact:
        flows = [_resize(f, (W, H), "flow")[crop] for f in flows]  # P-grid pixel units
        valids = [_resize(v, (W, H), "nearest")[crop] for v in valids]
        # resampling to half resolution must keep P-grid units, so rescale the half-res vectors back by 2
        mv = np.stack([_resize(f, half, "flow") * 2.0 for f in flows])
        va = np.stack([_resize(v, half, "nearest") for v in valids])
        # reverse time: network frame k holds source frame T-1-k; its previous network frame (k-1) is source
        # frame T-k, which is exactly where the forward flow of source frame T-1-k points
        out["rgb"], out["depth"] = out["rgb"][::-1].copy(), out["depth"][::-1].copy()
        mv, va = mv[::-1].copy(), va[::-1].copy()
        mv[0], va[0] = 0.0, 0  # network frame 0 has no previous frame in the window
        out["mv"], out["valid"], out["mv_quality"] = mv.astype(np.float16), va, 1.0
    return seed, out


def plan_windows(items, T, n, seed, stride=None):
    """Sample n (item, start) windows, spread evenly across items (round-robin over shuffled items)."""
    rng = random.Random(seed)
    usable = [it for it in items if it["n_frames"] >= T]
    rng.shuffle(usable)
    plans = []
    while usable and len(plans) < n:
        for it in usable:
            starts = range(0, it["n_frames"] - T + 1, stride or max(1, T // 2))
            plans.append((it, rng.choice(list(starts))))
            if len(plans) >= n:
                break
    return plans


def still_crop(task):
    """Worker: decode one still or frame and return k random S x S crops after a random down-scale."""
    data_root, uri, k, S, scale_range, seed, min_std = task
    rng = np.random.default_rng(seed)
    try:
        img = load_rgb(Resolver(data_root), uri)
    except Exception:
        return seed, None
    h, w = img.shape[:2]
    r = max(float(rng.uniform(*scale_range)), S / min(h, w) * 1.02)
    if r < 1.0:
        img = _resize(img, (int(round(w * r)), int(round(h * r))), "image")
    h, w = img.shape[:2]
    if min(h, w) < S:
        return seed, None
    crops = []
    for _ in range(k * 3):
        y, x = int(rng.integers(0, h - S + 1)), int(rng.integers(0, w - S + 1))
        c = img[y : y + S, x : x + S]
        if c.mean(axis=2).std() >= min_std:
            crops.append(c)
            if len(crops) == k:
                break
    return seed, np.stack(crops) if crops else None


# ---------------------------------------------------------------------------------------------------------------
# Manifest I/O and discovery of an attached data-pipeline output
# ---------------------------------------------------------------------------------------------------------------
LIST_COLUMNS = ["frames", "flows", "valid", "depth", "native_lr"]


def save_manifest(items, path):
    import pandas as pd

    df = pd.DataFrame(items)
    for c in LIST_COLUMNS:
        if c in df:
            df[c] = df[c].map(lambda v: json.dumps(v) if isinstance(v, list) else None)
    df.to_parquet(path, index=False)
    return df


def load_manifest(path):
    import pandas as pd

    df = pd.read_parquet(path)
    for c in LIST_COLUMNS:
        if c in df:
            df[c] = df[c].map(lambda v: json.loads(v) if isinstance(v, str) else None)
    return df


def items_from_frame(df):
    return [
        {k: v for k, v in row.items() if not (isinstance(v, float) and np.isnan(v))}
        for row in df.to_dict("records")
    ]


def locate_data_root(
    input_root="/kaggle/input", name="nss_manifest.parquet", max_depth=5
):
    """Folder holding the manifest produced by the data-pipeline notebook, if that output is attached."""
    input_root = Path(input_root)
    if not input_root.exists():
        return None
    frontier = [(input_root, 0)]
    while frontier:
        folder, depth = frontier.pop()
        if (folder / name).exists():
            return folder
        if depth < max_depth:
            try:
                frontier.extend((p, depth + 1) for p in folder.iterdir() if p.is_dir())
            except OSError:
                pass
    return None

**`nss_dedup.py`**: pHash / dHash, DINOv2 embeddings on both GPUs, exhaustive GPU neighbour search, threshold calibration with synthetic near-duplicates, union-find leakage-safe split.

In [ ]:
%%writefile /kaggle/working/code/nss_dedup.py
"""Perceptual near-duplicate detection across the combined dataset and leakage-safe splitting.

Pipeline: 64-bit DCT perceptual hash (pHash) and difference hash (dHash) on thumbnails, DINOv2 image embeddings,
exhaustive GPU nearest-neighbour search (chunked matrix products, both GPUs), thresholds calibrated on synthetic
positive pairs (crops, rescales, JPEG, colour shifts, flips, text overlays) against negative pairs, union-find
clustering of near-duplicate edges together with structural edges (same sequence / scene / environment), and a
split assigned per connected component so no component spans train, validation and test.
"""

import hashlib
import io

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFilter

# ---------------------------------------------------------------------------------------------------------------
# Hashes
# ---------------------------------------------------------------------------------------------------------------
_DCT32 = None


def _dct_matrix(n=32):
    k = np.arange(n)[:, None]
    i = np.arange(n)[None, :]
    m = np.cos(np.pi * (2 * i + 1) * k / (2 * n)) * np.sqrt(2.0 / n)
    m[0] /= np.sqrt(2.0)
    return m.astype(np.float32)


def phash_bits(thumbs):
    """thumbs: uint8 (N, h, w, 3) -> bool (N, 64). DCT of a 32x32 grey image, 8x8 low frequencies (DC excluded
    from the median), each bit = coefficient above the median."""
    global _DCT32
    if _DCT32 is None:
        _DCT32 = _dct_matrix(32)
    grey = np.stack(
        [
            np.asarray(
                Image.fromarray(t).convert("L").resize((32, 32), Image.BILINEAR),
                np.float32,
            )
            for t in thumbs
        ]
    )
    coeffs = _DCT32 @ grey @ _DCT32.T
    low = coeffs[:, :8, :8].reshape(len(thumbs), 64)
    med = np.median(low[:, 1:], axis=1, keepdims=True)
    return low > med


def dhash_bits(thumbs):
    """Horizontal gradient-sign hash on a 9x8 grey image -> bool (N, 64)."""
    grey = np.stack(
        [
            np.asarray(
                Image.fromarray(t).convert("L").resize((9, 8), Image.BILINEAR),
                np.float32,
            )
            for t in thumbs
        ]
    )
    return (grey[:, :, 1:] > grey[:, :, :-1]).reshape(len(thumbs), 64)


def bits_to_hex(bits):
    return [
        "".join(
            f"{int(''.join('1' if b else '0' for b in row[i:i + 4]), 2):x}"
            for i in range(0, 64, 4)
        )
        for row in bits
    ]


# ---------------------------------------------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------------------------------------------
def load_embedder(device, log=print):
    """DINOv2-small (Apache 2.0) from the Hugging Face Hub; returns (callable, name) or (None, None) offline."""
    try:
        from transformers import AutoModel

        model = (
            AutoModel.from_pretrained("facebook/dinov2-small").to(device).eval().half()
        )

        @torch.no_grad()
        def embed(x):  # x: float (N, 3, 224, 224) in [0, 1]
            mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
            out = model(pixel_values=((x - mean) / std).half())
            feat = (
                out.pooler_output
                if getattr(out, "pooler_output", None) is not None
                else out.last_hidden_state[:, 0]
            )
            return F.normalize(feat.float(), dim=1)

        return embed, "dinov2-small"
    except Exception as exc:
        log(f"DINOv2 unavailable ({exc}); embeddings skipped, hashes only")
        return None, None


@torch.no_grad()
def embed_thumbs(thumbs, embedders, devices, batch=256):
    """Embed uint8 thumbnails (N, 256, 256, 3), splitting batches across devices (threads, one per GPU)."""
    from concurrent.futures import ThreadPoolExecutor

    chunks = [(i, thumbs[i : i + batch]) for i in range(0, len(thumbs), batch)]
    out = [None] * len(chunks)

    def work(rank):
        for k in range(rank, len(chunks), len(devices)):
            _, t = chunks[k]
            x = (
                torch.from_numpy(np.ascontiguousarray(t))
                .to(devices[rank])
                .permute(0, 3, 1, 2)
                .float()
                / 255.0
            )
            x = F.interpolate(
                x, size=(224, 224), mode="bilinear", antialias=True, align_corners=False
            )
            out[k] = embedders[rank](x).cpu()

    with ThreadPoolExecutor(len(devices)) as pool:
        list(pool.map(work, range(len(devices))))
    return torch.cat(out).numpy()


# ---------------------------------------------------------------------------------------------------------------
# Exhaustive neighbour search on GPU (cosine for embeddings, Hamming for hashes via +-1 dot products)
# ---------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def pairs_above(feats, threshold, device, chunk=4096):
    """All pairs (i < j) with dot(feats_i, feats_j) >= threshold. Returns int64 (M, 2) and float32 scores."""
    x = torch.from_numpy(np.ascontiguousarray(feats)).to(device).half()
    n = x.shape[0]
    ii, jj, ss = [], [], []
    for a in range(0, n, chunk):
        sim = (x[a : a + chunk] @ x.T).float()
        rows = torch.arange(a, min(a + chunk, n), device=device)[:, None]
        cols = torch.arange(n, device=device)[None, :]
        mask = (sim >= threshold) & (cols > rows)
        r, c = mask.nonzero(as_tuple=True)
        ii.append((r + a).cpu())
        jj.append(c.cpu())
        ss.append(sim[r, c].cpu())
    if not ii:
        return np.zeros((0, 2), np.int64), np.zeros(0, np.float32)
    return torch.stack([torch.cat(ii), torch.cat(jj)], 1).numpy(), torch.cat(ss).numpy()


def bits_to_pm1(bits):
    return np.where(bits, 1.0, -1.0).astype(np.float32)


def hamming_threshold_to_dot(h, nbits=64):
    """Hamming distance <= h  <=>  dot of +-1 vectors >= nbits - 2h."""
    return float(nbits - 2 * h)


# ---------------------------------------------------------------------------------------------------------------
# Calibration with synthetic positives and random negatives
# ---------------------------------------------------------------------------------------------------------------
def augment_copy(img, rng):
    """A near-duplicate of img as it might appear in another dataset: re-crop, rescale, recompress, recolour,
    mirror, small rotation, blur or a text/watermark overlay. img: PIL RGB."""
    w, h = img.size
    a = rng.uniform(0.6, 0.95)
    cw, ch = int(w * np.sqrt(a)), int(h * np.sqrt(a))
    x, y = rng.integers(0, w - cw + 1), rng.integers(0, h - ch + 1)
    out = img.crop((x, y, x + cw, y + ch)).resize((256, 256), Image.BILINEAR)
    if rng.random() < 0.5:
        out = out.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.3:
        out = out.rotate(float(rng.uniform(-4, 4)), resample=Image.BILINEAR)
    if rng.random() < 0.5:
        arr = np.asarray(out, np.float32) * rng.uniform(0.8, 1.2) + rng.uniform(-20, 20)
        out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if rng.random() < 0.3:
        out = out.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.5, 1.5))))
    if rng.random() < 0.3:
        ImageDraw.Draw(out).text(
            (int(rng.integers(5, 120)), int(rng.integers(5, 220))),
            "SAMPLE 1080p",
            fill=(255, 255, 255),
        )
    buf = io.BytesIO()
    out.save(buf, format="JPEG", quality=int(rng.integers(35, 92)))
    return Image.open(io.BytesIO(buf.getvalue())).convert("RGB")


def calibration_pairs(thumbs, n_pos, seed):
    """Positive pairs (thumb, augmented copy) and negative pairs (two different thumbs). Returns arrays of
    thumbnails A, B (uint8, N x 256 x 256 x 3) and labels."""
    rng = np.random.default_rng(seed)
    n = len(thumbs)
    idx = rng.choice(n, size=min(n_pos, n), replace=False)
    a_list, b_list, labels = [], [], []
    for i in idx:
        a_list.append(thumbs[i])
        b_list.append(
            np.asarray(augment_copy(Image.fromarray(thumbs[i]), rng).resize((256, 256)))
        )
        labels.append(1)
    for _ in range(len(idx) * 4):
        i, j = rng.choice(n, size=2, replace=False)
        a_list.append(thumbs[i])
        b_list.append(thumbs[j])
        labels.append(0)
    return np.stack(a_list), np.stack(b_list), np.array(labels)


def choose_threshold(scores, labels, max_fpr=0.001):
    """Lowest similarity threshold whose false-positive rate on negatives stays <= max_fpr; returns the
    threshold with precision, recall and F1 at that point, plus a ROC table."""
    order = np.argsort(-scores)
    s, y = scores[order], labels[order]
    tp, fp = np.cumsum(y == 1), np.cumsum(y == 0)
    P, N = max((labels == 1).sum(), 1), max((labels == 0).sum(), 1)
    fpr, tpr = fp / N, tp / P
    ok = np.nonzero(fpr <= max_fpr)[0]
    k = ok[-1] if len(ok) else 0
    thr = float(s[k])
    prec = tp[k] / max(tp[k] + fp[k], 1)
    rec = tpr[k]
    roc = {
        "fpr": fpr[:: max(1, len(fpr) // 400)].tolist(),
        "tpr": tpr[:: max(1, len(tpr) // 400)].tolist(),
    }
    return {
        "threshold": thr,
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(2 * prec * rec / max(prec + rec, 1e-9)),
        "roc": roc,
    }


# ---------------------------------------------------------------------------------------------------------------
# Clustering and leakage-safe split
# ---------------------------------------------------------------------------------------------------------------
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, a):
        while self.parent[a] != a:
            self.parent[a] = self.parent[self.parent[a]]
            a = self.parent[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


def assign_splits(items, dup_edges, val_fraction, seed):
    """items: dicts with 'group' and 'role' ('train' or 'test'); dup_edges: (i, j) item index pairs.
    Components = union of shared leakage groups and near-duplicate edges. A component containing any test item
    becomes test; its train-role members are dropped (they would leak). Other components go to val with
    probability val_fraction (deterministic hash), else train. Returns (split list, component id list, report).
    """
    n = len(items)
    uf = UnionFind(n)
    first_of_group = {}
    for i, it in enumerate(items):
        g = it["group"]
        if g in first_of_group:
            uf.union(i, first_of_group[g])
        else:
            first_of_group[g] = i
    for i, j in dup_edges:
        uf.union(int(i), int(j))
    comp = [uf.find(i) for i in range(n)]
    has_test = {}
    for i, it in enumerate(items):
        if it["role"] == "test":
            has_test[comp[i]] = True
    split, dropped = [], 0
    for i, it in enumerate(items):
        c = comp[i]
        if has_test.get(c):
            if it["role"] == "test":
                split.append("test")
            else:
                split.append("drop")
                dropped += 1
        else:
            digest = hashlib.md5(f"{seed}/{items[c]['item_id']}".encode()).hexdigest()
            h = (int(digest, 16) % 10_000) / 10_000.0
            split.append("val" if h < val_fraction else "train")
    sizes = np.bincount(np.unique(comp, return_inverse=True)[1])
    report = {
        "items": n,
        "components": int(len(sizes)),
        "largest_component": int(sizes.max()) if n else 0,
        "dup_edges": int(len(dup_edges)),
        "train_items_dropped_for_test_leakage": dropped,
        "split_counts": {s: split.count(s) for s in ("train", "val", "test", "drop")},
    }
    return split, comp, report

**`nss_pipeline.py`**: the data-pipeline steps (discover, materialise, statistics, filters, hashes, calibration, duplicate search, split, audit) and the end-to-end fallback.

In [ ]:
%%writefile /kaggle/working/code/nss_pipeline.py
"""Unified data pipeline steps (used cell by cell in the data-pipeline notebook, and end to end by
build_manifest() when the supersampling or frame-generation notebook runs without the pipeline output attached).

discover -> materialise Hugging Face subsets -> keyframe thumbnails and quality statistics -> filters ->
perceptual hashes and embeddings -> threshold calibration -> near-duplicate search -> leakage-safe split.
"""

import shutil
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import nss_data as D
import nss_dedup as DD

FILTER_RULES = {
    "min_side": 256,
    "min_std": 8.0,
    "max_letterbox": 0.35,
    "min_sharpness": {"still": 20.0, "seq": 5.0},
}


def discover_kaggle(input_root, cap, seed, log=print):
    items, rows = [], []
    kaggle = [s for s in D.SOURCES if s["host"] == "kaggle"]
    with ThreadPoolExecutor(len(kaggle)) as pool:
        results = list(
            pool.map(
                lambda s: (s, *D.catalogue_kaggle(s, input_root, cap, seed)), kaggle
            )
        )
    for src, its, info in results:
        items.extend(its)
        rows.append(
            {
                "source": src["name"],
                "ref": src["ref"],
                **info,
                "items": len(its),
                "test_items": sum(i["role"] == "test" for i in its),
            }
        )
        if not info.get("found"):
            log(f"{src['name']}: not attached ({src['ref']})")
    return items, pd.DataFrame(rows)


def materialise_hf(data_root, caps, seed, log=print):
    """caps: {"game_stills": (n_train, n_test), "tartanair": frames_per_env, "tartanair_seq": len,
    "gameir": (train_clips, test_clips), "vimeo1080p": (n_train, n_val)}; a cap of 0 skips the source.
    """
    items = []
    steps = [
        (
            "game_stills",
            lambda s: D.materialise_game_stills(
                s, data_root, *caps["game_stills"], seed, log=log
            ),
        ),
        (
            "tartanair",
            lambda s: D.materialise_tartanair(
                s,
                data_root,
                caps["tartanair"],
                s["train_envs"],
                s["test_envs"],
                caps["tartanair_seq"],
                log=log,
            ),
        ),
        (
            "gameir",
            lambda s: D.materialise_gameir(s, data_root, *caps["gameir"], log=log),
        ),
        (
            "vimeo1080p",
            lambda s: D.materialise_vimeo1080p(
                s, data_root, *caps["vimeo1080p"], log=log
            ),
        ),
    ]
    for name, fn in steps:
        cap = caps.get(name)
        if not cap or (isinstance(cap, tuple) and not any(cap)):
            continue
        try:
            items.extend(fn(D.source(name)))
        except Exception as exc:
            log(f"{name}: materialisation failed ({exc}); continuing without it")
    return items


def keyframe_stats(items, data_root, workers):
    """Thumbnails (K, 256, 256, 3) and statistics for every keyframe; returns thumbs, table (one row per keyframe)."""
    tasks, owner = [], []
    for i, it in enumerate(items):
        for uri in D.keyframes(it):
            tasks.append((str(data_root), uri))
            owner.append(i)
    thumbs = np.zeros((len(tasks), 256, 256, 3), np.uint8)
    rows = [None] * len(tasks)
    from tqdm.auto import tqdm

    with Pool(workers) as pool:
        for k, (uri, small, stats) in enumerate(
            tqdm(
                pool.imap(D.thumb_and_stats, tasks, chunksize=4),
                total=len(tasks),
                desc="Keyframe thumbnails and statistics",
            )
        ):
            if small is not None:
                thumbs[k] = small
            rows[k] = {"item": owner[k], "uri": uri, "ok": small is not None, **stats}
    return thumbs, pd.DataFrame(rows)


def apply_filters(items, kf, rules=FILTER_RULES):
    """Item-level keep flags and rejection reasons (an item is rejected if its keyframes fail)."""
    reasons = {}
    for i, g in kf.groupby("item"):
        kind = items[i]["kind"]
        r = []
        if not g["ok"].all():
            r.append("decode_error")
        else:
            if g[["width", "height"]].min(axis=1).min() < rules["min_side"]:
                r.append("too_small")
            if g["std"].min() < rules["min_std"]:
                r.append("flat")
            if g["letterbox"].max() > rules["max_letterbox"]:
                r.append("letterbox")
            if (
                g["sharpness"].median()
                < rules["min_sharpness"]["still" if kind == "still" else "seq"]
            ):
                r.append("blurry")
        reasons[i] = r
    keep = [not reasons.get(i, ["no_keyframe"]) for i in range(len(items))]
    return keep, reasons


def hashes(thumbs):
    return DD.phash_bits(thumbs), DD.dhash_bits(thumbs)


def calibrate(thumbs, embedders, devices, n_pos, seed, max_fpr=0.001, margin=0.02):
    """Thresholds for pHash Hamming distance and embedding cosine similarity from synthetic positive pairs.

    An all-pairs search over K keyframes tests about K^2 / 2 pairs, so a threshold tuned only to a 0.1 percent
    false-positive rate on random pairs would create far too many false edges. Each threshold is therefore the
    stricter of (a) the max_fpr point and (b) the most similar random negative plus a margin; find_duplicates
    then tightens it further if the edge count is still implausible."""
    a, b, y = DD.calibration_pairs(thumbs, n_pos, seed)
    pa, pb = DD.phash_bits(a), DD.phash_bits(b)
    ham = (pa != pb).sum(1)
    res = {"phash": DD.choose_threshold(-ham.astype(np.float32), y, max_fpr)}
    neg_min_ham = int(ham[y == 0].min()) if (y == 0).any() else 64
    res["phash"]["hamming"] = int(min(-res["phash"]["threshold"], neg_min_ham - 2))
    res["phash"]["recall_at_threshold"] = float(
        (ham[y == 1] <= res["phash"]["hamming"]).mean()
    )
    res["phash_scores"], res["labels"] = ham.tolist(), y.tolist()
    if embedders:
        ea, eb = DD.embed_thumbs(a, embedders, devices), DD.embed_thumbs(
            b, embedders, devices
        )
        cos = (ea * eb).sum(1)
        res["embedding"] = DD.choose_threshold(cos, y, max_fpr)
        res["embedding"]["threshold"] = float(
            max(res["embedding"]["threshold"], cos[y == 0].max() + margin)
        )
        res["embedding"]["recall_at_threshold"] = float(
            (cos[y == 1] >= res["embedding"]["threshold"]).mean()
        )
        res["embedding_scores"] = cos.tolist()
    return res


def find_duplicates(
    kf, phash, emb, thresholds, device, dhash=None, min_std=15.0, max_degree=2.0
):
    """Keyframe pairs judged duplicates by pHash (Hamming <= h, confirmed by dHash, only for keyframes with enough
    structure: low-contrast frames share hashes by accident) or by embeddings (cosine >= c), mapped to items
    (pairs inside one item are ignored). If the number of item edges exceeds max_degree per keyframe, the
    thresholds are tightened step by step (an implausible duplicate rate means false positives, which would merge
    unrelated data into giant components). Returns the pair table; final thresholds are written into thresholds.
    """
    n = len(kf)
    textured = kf["std"].fillna(0).values >= min_std
    h = thresholds["phash"]["hamming"]
    while True:
        rows = []
        pairs, scores = DD.pairs_above(
            DD.bits_to_pm1(phash), DD.hamming_threshold_to_dot(h), device
        )
        for (i, j), sc in zip(pairs, scores):
            if not (textured[i] and textured[j]):
                continue
            if dhash is not None and (dhash[i] != dhash[j]).sum() > 2 * h + 4:
                continue
            rows.append((i, j, "phash", (64 - sc) / 2))
        if len(rows) <= max_degree * n or h <= 0:
            break
        h -= 1
    thresholds["phash"]["hamming_final"] = int(h)
    if emb is not None and "embedding" in thresholds:
        c = thresholds["embedding"]["threshold"]
        while True:
            pairs, scores = DD.pairs_above(emb, c, device)
            if len(pairs) <= max_degree * n or c >= 0.995:
                break
            c = min(0.995, c + 0.01)
        thresholds["embedding"]["threshold_final"] = float(c)
        rows += [(i, j, "embedding", sc) for (i, j), sc in zip(pairs, scores)]
    df = pd.DataFrame(rows, columns=["kf_a", "kf_b", "method", "score"])
    if len(df):
        df["item_a"] = kf["item"].values[df["kf_a"]]
        df["item_b"] = kf["item"].values[df["kf_b"]]
        df = df[df["item_a"] != df["item_b"]].reset_index(drop=True)
    return df


def split_items(items, dup_pairs, val_fraction, seed):
    edges = (
        list(zip(dup_pairs["item_a"], dup_pairs["item_b"])) if len(dup_pairs) else []
    )
    split, comp, report = DD.assign_splits(items, edges, val_fraction, seed)
    for it, s, c in zip(items, split, comp):
        it["split"], it["component"] = s, int(c)
    return report


def cross_split_audit(kf, items, emb, phash, device, k=1):
    """Nearest neighbour of every test / val keyframe among train keyframes (cosine and Hamming)."""
    split = np.array([items[i]["split"] for i in kf["item"]])
    tr = np.nonzero(split == "train")[0]
    out = {}
    for s in ("val", "test"):
        q = np.nonzero(split == s)[0]
        if not len(q) or not len(tr):
            continue
        res = {}
        if emb is not None:
            x = torch.from_numpy(emb[tr]).to(device).half()
            best = []
            for a in range(0, len(q), 4096):
                sim = torch.from_numpy(emb[q[a : a + 4096]]).to(device).half() @ x.T
                best.append(sim.float().max(1).values.cpu())
            res["max_cosine_to_train"] = torch.cat(best).numpy()
        x = torch.from_numpy(DD.bits_to_pm1(phash[tr])).to(device).half()
        best = []
        for a in range(0, len(q), 4096):
            dot = (
                torch.from_numpy(DD.bits_to_pm1(phash[q[a : a + 4096]]))
                .to(device)
                .half()
                @ x.T
            )
            best.append(((64 - dot.float().max(1).values) / 2).cpu())
        res["min_hamming_to_train"] = torch.cat(best).numpy()
        out[s] = res
    return out


def build_manifest(data_root, input_root, cfg, devices, log=print):
    """End-to-end fallback used by the training notebooks when no pipeline output is attached."""
    Path(data_root).mkdir(parents=True, exist_ok=True)
    items, _ = discover_kaggle(input_root, cfg["list_cap_files"], cfg["seed"], log=log)
    items += materialise_hf(data_root, cfg["hf_caps"], cfg["seed"], log=log)
    thumbs, kf = keyframe_stats(items, data_root, cfg["workers"])
    keep, _ = apply_filters(items, kf)
    for it, k in zip(items, keep):
        it["filtered_out"] = not k
    ph, dh = hashes(thumbs)
    embedders = []
    for dvc in devices:
        e, _ = DD.load_embedder(dvc, log=log)
        if e is None:
            embedders = []
            break
        embedders.append(e)
    emb = DD.embed_thumbs(thumbs, embedders, devices) if embedders else None
    thr = calibrate(thumbs, embedders, devices, cfg["calibration_pairs"], cfg["seed"])
    pairs = find_duplicates(kf, ph, emb, thr, devices[0], dhash=dh)
    report = split_items(items, pairs, cfg["val_fraction"], cfg["seed"])
    for it in items:
        if it.get("filtered_out"):
            it["split"] = "filtered"
    D.save_manifest(items, Path(data_root) / "nss_manifest.parquet")
    log(f"manifest: {report['split_counts']}")
    shutil.rmtree(Path(data_root) / "_downloads", ignore_errors=True)
    return items, report

**`nss_cache.py`**: memory-mapped training caches (still crops, sequence windows, frame-generation quads) and RAFT teacher motion.

In [ ]:
%%writefile /kaggle/working/code/nss_cache.py
"""Training caches: memory-mapped uint8 still crops (textures for the synthetic renderer) and fixed-size
sequence windows with motion / validity / depth, plus RAFT teacher motion for windows without exact motion.

Arrays per window cache prefix:
  rgb (N, T, P, P, 3) uint8 | mv (N, T, P/2, P/2, 2) fp16 (P-grid pixels, frame -> previous frame)
  valid (N, T, P/2, P/2) uint8 | depth (N, T, P/2, P/2) fp16 | meta (N, 2) fp32 [has_depth, mv_quality]
Quad caches (frame generation) add flows (N, 4, P/2, P/2, 2) fp16 = teacher flows [2->0, 1->0, 1->2, 3->2].
"""

import math
import random
import time
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn.functional as F

import nss_data as D
import nss_metrics as M

# down-scale range per source before cropping (detail density; removes compression noise from large frames)
SCALE_RANGES = {
    "div2k": (0.35, 0.9),
    "flickr2k": (0.35, 0.9),
    "game_stills": (0.5, 1.0),
    "gameir": (0.4, 0.8),
    "vimeo1080p": (0.35, 0.7),
    "reds": (0.5, 1.0),
    "vimeo_septuplet": (1.0, 1.0),
    "tartanair": (0.6, 1.0),
    "sintel": (0.6, 1.0),
    "vid4": (1.0, 1.0),
    "vimeo_triplet": (1.0, 1.0),
}


def items_for(manifest, split, sources=None, kinds=None, min_frames=1):
    df = manifest[manifest["split"] == split]
    if sources is not None:
        df = df[df["source"].isin(sources)]
    if kinds is not None:
        df = df[df["kind"].isin(kinds)]
    df = df[df["n_frames"] >= min_frames]
    return D.items_from_frame(df)


def still_tasks(items, data_root, capacity, S, crops_per_image, seed, min_std=8.0):
    """One decode task per (item, frame): stills directly, sequences contribute a few random frames."""
    rng = random.Random(seed)
    tasks = []
    for it in items:
        frames = (
            it["frames"]
            if it["n_frames"] == 1
            else rng.sample(it["frames"], min(3, it["n_frames"]))
        )
        for uri in frames:
            tasks.append(
                (
                    str(data_root),
                    uri,
                    crops_per_image,
                    S,
                    SCALE_RANGES.get(it["source"], (0.5, 1.0)),
                    rng.randrange(1 << 30),
                    min_std,
                )
            )
    rng.shuffle(tasks)
    need = math.ceil(capacity / crops_per_image) * 2
    return tasks[:need]


def build_still_cache(tasks, path, capacity, S, workers, deadline_s, log=print):
    from tqdm.auto import tqdm

    arr = np.lib.format.open_memmap(
        path, mode="w+", dtype=np.uint8, shape=(capacity, S, S, 3)
    )
    filled, failed, t0 = 0, 0, time.time()
    bar = tqdm(total=capacity, desc=f"Still crops -> {path.name}", unit="crop")
    with Pool(workers) as pool:
        for _, crops in pool.imap_unordered(D.still_crop, tasks, chunksize=2):
            if crops is None:
                failed += 1
                continue
            take = min(len(crops), capacity - filled)
            arr[filled : filled + take] = crops[:take]
            filled += take
            bar.update(take)
            if filled >= capacity or time.time() - t0 > deadline_s:
                break
    bar.close()
    arr.flush()
    del arr
    log(f"{path.name}: {filled} crops ({failed} decode failures)")
    return filled


def open_window_arrays(prefix, capacity, T, P, flows=False):
    half = P // 2
    spec = {
        "rgb": ((capacity, T, P, P, 3), np.uint8),
        "mv": ((capacity, T, half, half, 2), np.float16),
        "valid": ((capacity, T, half, half), np.uint8),
        "depth": ((capacity, T, half, half), np.float16),
        "meta": ((capacity, 2), np.float32),
    }
    if flows:
        spec["flows"] = ((capacity, 4, half, half, 2), np.float16)
    return {
        k: np.lib.format.open_memmap(
            f"{prefix}_{k}.npy", mode="w+", dtype=dt, shape=shape
        )
        for k, (shape, dt) in spec.items()
    }


def build_window_cache(
    items,
    data_root,
    prefix,
    capacity,
    T,
    P,
    workers,
    deadline_s,
    seed,
    flows=False,
    log=print,
):
    """Decode capacity windows of T frames into memory maps (round-robin over items so the cache spans many
    sequences). Returns (count, per-source counts, indices of windows that still need estimated motion).
    """
    from tqdm.auto import tqdm

    plans = D.plan_windows(items, T, int(capacity * 1.3), seed)
    tasks = [
        (
            str(data_root),
            it,
            start,
            T,
            P,
            SCALE_RANGES.get(it["source"], (0.5, 1.0)),
            seed * 7 + k,
        )
        for k, (it, start) in enumerate(plans)
    ]
    src_of = {seed * 7 + k: it["source"] for k, (it, _) in enumerate(plans)}
    arrs = open_window_arrays(prefix, capacity, T, P, flows=flows)
    filled, failed, need_raft, counts, t0 = 0, 0, [], {}, time.time()
    bar = tqdm(total=capacity, desc=f"Windows -> {prefix.name}", unit="win")
    with Pool(workers) as pool:
        for key, w in pool.imap_unordered(D.extract_window, tasks, chunksize=1):
            if w is None:
                failed += 1
                continue
            i = filled
            arrs["rgb"][i], arrs["mv"][i], arrs["valid"][i], arrs["depth"][i] = (
                w["rgb"],
                w["mv"],
                w["valid"],
                w["depth"],
            )
            arrs["meta"][i] = (
                (0.0, 0.0) if flows else (float(w["has_depth"]), float(w["mv_quality"]))
            )
            if w["mv_quality"] < 1.0 or flows:
                need_raft.append(i)
            counts[src_of[key]] = counts.get(src_of[key], 0) + 1
            filled += 1
            bar.update(1)
            if filled >= capacity or time.time() - t0 > deadline_s:
                break
    bar.close()
    for a in arrs.values():
        a.flush()
    log(f"{prefix.name}: {filled} windows ({failed} failures)")
    return filled, counts, need_raft


def fill_estimated_motion(
    prefix, indices, rafts, devices, batch=16, quads=False, log=print
):
    """RAFT teacher motion for cached windows: motion vectors frame -> previous frame with forward-backward
    validity (windows without exact motion), or the four frame-generation flows of each quad. Work is split
    across GPUs with one thread per GPU; each thread writes disjoint rows of the memory maps.
    """
    from tqdm.auto import tqdm

    if not indices or not rafts:
        if indices:
            log(
                f"{prefix.name}: no RAFT; {len(indices)} windows keep zero motion (mv_quality 0)"
            )
        return
    rgb = np.load(f"{prefix}_rgb.npy", mmap_mode="r")
    mv = np.load(f"{prefix}_mv.npy", mmap_mode="r+")
    valid = np.load(f"{prefix}_valid.npy", mmap_mode="r+")
    meta = np.load(f"{prefix}_meta.npy", mmap_mode="r+")
    fl = np.load(f"{prefix}_flows.npy", mmap_mode="r+") if quads else None
    chunks = [indices[i : i + batch] for i in range(0, len(indices), batch)]
    bar = tqdm(total=len(indices), desc=f"RAFT motion -> {prefix.name}", unit="win")

    def half(x):
        return F.avg_pool2d(x, 2)  # values stay in P-grid pixels

    def work(rank):
        dev, net = devices[rank], rafts[rank]
        for c in chunks[rank :: len(devices)]:
            idx = np.array(c)
            x = (
                torch.from_numpy(np.ascontiguousarray(rgb[idx]))
                .to(dev)
                .permute(0, 1, 4, 2, 3)
                .float()
                / 255.0
            )
            n, T = x.shape[:2]
            if quads:
                pairs = [(2, 0), (1, 0), (1, 2), (3, 2)]
                out = [half(M.raft_flow(net, x[:, a], x[:, b])) for a, b in pairs]
                fl[idx] = (
                    torch.stack(out, 1).permute(0, 1, 3, 4, 2).half().cpu().numpy()
                )
                meta[idx, 0] = 1.0  # quad caches: meta = [flows available, unused]
            else:
                m_all = np.zeros((n, T) + mv.shape[2:], np.float16)
                v_all = np.zeros((n, T) + valid.shape[2:], np.uint8)
                for t in range(1, T):
                    m, v = M.raft_mv_and_valid(net, x[:, t], x[:, t - 1])
                    m_all[:, t] = half(m).permute(0, 2, 3, 1).half().cpu().numpy()
                    v_all[:, t] = (half(v)[:, 0] > 0.5).to(torch.uint8).cpu().numpy()
                mv[idx], valid[idx] = m_all, v_all
                meta[idx, 1] = 0.5
            bar.update(len(c))

    with ThreadPoolExecutor(len(devices)) as pool:
        list(pool.map(work, range(len(devices))))
    bar.close()
    for a in (mv, valid, meta) + ((fl,) if quads else ()):
        a.flush()


def cache_spec(prefix, count, keys):
    return {"arrays": {k: f"{prefix}_{k}.npy" for k in keys}, "count": int(count)}

**`nss_synth.py`**: GPU renderer of layered game-like scenes with exact motion, disocclusion, thin geometry, particles, HUD / menus / cuts, jittered low-resolution rendering; batch builders for real sequences.

In [ ]:
%%writefile /kaggle/working/code/nss_synth.py
"""GPU renderer of synthetic game-like sequences with exact ground truth.

A scene is a 2.5D layered world built from cached high-resolution stills: a background plane and a foreground
layer with a procedural alpha mask (solid blobs, thin wires and fences), each moving with its own continuous
affine camera trajectory, plus alpha-blended particles that write no motion vectors (as in real engines),
optional world-space text stamped into the background, and optional screen-space HUD/UI overlays.

Because every pixel's layer and trajectory are known, the renderer returns exact motion vectors, disocclusion
masks, per-layer depth, intermediate-time optical flow, reactive (transparency) masks and UI masks.

Low-resolution frames are rendered like a game renders at reduced resolution: one sample per pixel at a sub-pixel
jittered position (Halton 2,3), sampling the full-resolution texture without mip filtering, which produces real
aliasing on thin geometry and fine texture. Targets are rendered at output resolution with 4 samples per pixel
(rotated grid), an anti-aliased "native" reference.

Coordinates: screen positions are normalised to [-1, 1] (pixel centres at (2i + 1) / W - 1); texture
coordinates are normalised to [-1, 1]; a layer maps screen to texture with tex = s R(theta) x + o.
"""

import math

import numpy as np
import torch
import torch.nn.functional as F

SSAA_OFFSETS = [
    (-0.375, -0.125),
    (0.125, -0.375),
    (0.375, 0.125),
    (-0.125, 0.375),
]  # rotated grid, pixel units


def halton(index, base):
    f, r = 1.0, 0.0
    while index > 0:
        f /= base
        r += f * (index % base)
        index //= base
    return r


def jitter_sequence(n, phase_count=16):
    """Halton(2,3) sub-pixel offsets in [-0.5, 0.5) for frames 0..n-1 (cycle of phase_count)."""
    return torch.tensor(
        [
            [halton(i % phase_count + 1, 2) - 0.5, halton(i % phase_count + 1, 3) - 0.5]
            for i in range(n)
        ]
    )


def pixel_grid(n, h, w, device, offset=None):
    """Normalised screen positions of pixel centres, (n, h, w, 2) as (x, y). offset: (n, 2) in pixels."""
    ys = (torch.arange(h, device=device, dtype=torch.float32) + 0.5) * (2.0 / h) - 1.0
    xs = (torch.arange(w, device=device, dtype=torch.float32) + 0.5) * (2.0 / w) - 1.0
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    grid = torch.stack([gx, gy], -1)[None].expand(n, h, w, 2)
    if offset is not None:
        grid = (
            grid
            + (offset * torch.tensor([2.0 / w, 2.0 / h], device=device))[
                :, None, None, :
            ]
        )
    return grid


def srgb_to_linear(x):
    return torch.where(
        x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).clamp_min(0) ** 2.4
    )


def linear_to_srgb(x):
    x = x.clamp(0, 1)
    return torch.where(
        x <= 0.0031308, x * 12.92, 1.055 * x.clamp_min(1e-8) ** (1 / 2.4) - 0.055
    )


# ---------------------------------------------------------------------------------------------------------------
# Layer trajectories
# ---------------------------------------------------------------------------------------------------------------
def layer_matrix(p, t):
    """p: dict of (N, ...) tensors; t: (N,) times. Returns (N, 2, 3) screen -> texture affine."""
    s = p["s0"] * torch.exp(p["zeta"] * t)
    th = p["th0"] + p["omega"] * t
    flip = torch.relu(t - p["t_flip"])[:, None] * p["flip"][:, None]
    o = (
        p["o0"]
        + p["u"] * t[:, None]
        + 0.5 * p["a"] * (t**2)[:, None]
        - 2.0 * p["u"] * flip
    )
    c, s_ = torch.cos(th), torch.sin(th)
    row0 = torch.stack([s * c, -s * s_, o[:, 0]], -1)
    row1 = torch.stack([s * s_, s * c, o[:, 1]], -1)
    return torch.stack([row0, row1], 1)


def apply_affine(m, pts):
    return torch.einsum("nij,nhwj->nhwi", m[:, :, :2], pts) + m[:, None, None, :, 2]


def invert_affine(m):
    a_inv = torch.linalg.inv(m[:, :, :2])
    return torch.cat([a_inv, -(a_inv @ m[:, :, 2:])], 2)


def _speed_px(n, g, profile, device):
    """Screen speed in output pixels per frame from a motion profile: static / slow / medium / fast mixture."""
    probs = torch.tensor(profile["probs"], device=device)
    k = torch.multinomial(probs, n, replacement=True, generator=g)
    lo = torch.tensor([b[0] for b in profile["bins"]], device=device)[k]
    hi = torch.tensor([b[1] for b in profile["bins"]], device=device)[k]
    return lo + (hi - lo) * torch.rand(n, device=device, generator=g), k


MOTION_PROFILE = {
    "names": ["static", "slow", "medium", "fast"],
    "probs": [0.15, 0.45, 0.25, 0.15],
    "bins": [(0.0, 0.0), (0.3, 3.0), (3.0, 10.0), (10.0, 28.0)],
}


def random_layer(
    n, g, device, out_px, tex_px, duration, profile, centre_screen=None, rot_scale=1.0
):
    """Trajectory parameters for n layers whose view of out_px pixels samples a texture of tex_px texels.
    The path is centred so the view stays inside the texture for the whole duration where possible.
    """
    rnd = lambda *shape: torch.rand(*shape, device=device, generator=g)  # noqa: E731
    k = 1.0 + 0.45 * rnd(
        n
    )  # texels per output pixel (> 1: minification, aliasing at low resolution)
    s0 = out_px * k / tex_px
    speed, cls = _speed_px(n, g, profile, device)
    ang = 2 * math.pi * rnd(n)
    u_tex = speed * (2.0 / out_px) * s0  # texture units per frame
    room = (1.0 - 1.42 * s0 - 0.02).clamp_min(0.0)
    u_tex = torch.minimum(u_tex, 2 * room / max(duration, 1e-3))
    u = torch.stack([torch.cos(ang), torch.sin(ang)], -1) * u_tex[:, None]
    o0 = (
        -u * duration / 2
        + (rnd(n, 2) * 2 - 1) * (room - u_tex * duration / 2).clamp_min(0)[:, None]
    )
    p = {
        "s0": s0,
        "zeta": torch.randn(n, device=device, generator=g) * 0.006 * (cls > 0),
        "th0": (rnd(n) * 2 - 1) * 0.2 * rot_scale,
        "omega": torch.randn(n, device=device, generator=g)
        * 0.008
        * rot_scale
        * (cls > 0),
        "o0": o0,
        "u": u,
        "a": torch.randn(n, 2, device=device, generator=g) * 0.1 * u_tex[:, None],
        "flip": torch.zeros(n, device=device),
        "t_flip": torch.full((n,), 1e9, device=device),
        "speed_class": cls,
    }
    if (
        centre_screen is not None
    ):  # place a texture point (the object centre) at a screen position at t = 0
        m0 = layer_matrix(p, torch.zeros(n, device=device))
        tex_at = (
            torch.einsum("nij,nj->ni", m0[:, :, :2], centre_screen[1]) + m0[:, :, 2]
        )
        p["o0"] = p["o0"] + centre_screen[0] - tex_at
    return p


# ---------------------------------------------------------------------------------------------------------------
# Procedural foreground masks and text atlas
# ---------------------------------------------------------------------------------------------------------------
def foreground_alpha(n, size, g, device, centres):
    """Binary-ish alpha (n, 1, S, S): one superellipse blob around centres (n, 2) plus thin wires and, sometimes,
    a fence of thin bars (1-2 texels wide) to create sub-pixel geometry."""
    rnd = lambda *shape: torch.rand(*shape, device=device, generator=g)  # noqa: E731
    coords = (torch.arange(size, device=device, dtype=torch.float32) + 0.5) * (
        2.0 / size
    ) - 1.0
    v, u = torch.meshgrid(coords, coords, indexing="ij")
    u, v = u[None], v[None]
    ang = (rnd(n) * math.pi)[:, None, None]
    du, dv = u - centres[:, 0, None, None], v - centres[:, 1, None, None]
    ru, rv = du * torch.cos(ang) + dv * torch.sin(ang), -du * torch.sin(
        ang
    ) + dv * torch.cos(ang)
    rx, ry = (0.12 + 0.3 * rnd(n))[:, None, None], (0.12 + 0.3 * rnd(n))[:, None, None]
    pw = (1.5 + 2.5 * rnd(n))[:, None, None]
    alpha = ((ru / rx).abs() ** pw + (rv / ry).abs() ** pw < 1.0).float()
    texel = 2.0 / size
    for _ in range(4):  # thin wires, present with probability 0.6 each
        on = (rnd(n) < 0.6)[:, None, None]
        px, py = (rnd(n) * 1.6 - 0.8)[:, None, None], (rnd(n) * 1.6 - 0.8)[
            :, None, None
        ]
        phi = (rnd(n) * math.pi)[:, None, None]
        width = ((0.6 + 1.4 * rnd(n)) * texel)[:, None, None]
        dist = ((u - px) * torch.sin(phi) - (v - py) * torch.cos(phi)).abs()
        alpha = torch.maximum(alpha, ((dist < width / 2) & on).float())
    fence = (rnd(n) < 0.35)[:, None, None]
    period = ((6 + 14 * rnd(n)) * texel)[:, None, None]
    bar = ((1.0 + rnd(n)) * texel)[:, None, None]
    phi = (rnd(n) * math.pi)[:, None, None]
    proj = u * torch.cos(phi) + v * torch.sin(phi)
    frac = torch.remainder(proj, period)
    region = ((u - centres[:, 0, None, None]).abs() < 0.6) & (
        (v - centres[:, 1, None, None]).abs() < 0.6
    )
    alpha = torch.maximum(alpha, ((frac < bar) & fence & region).float())
    return alpha[:, None]


def make_text_bank(n, seed, height=40, width=320):
    """RGBA uint8 tiles (n, 4, height, width) of HUD-like text, bars and icons rendered with PIL (CPU, once)."""
    import matplotlib.font_manager as fm
    from PIL import Image, ImageDraw, ImageFont

    rng = np.random.default_rng(seed)
    font_paths = sorted(
        {
            fm.findfont("DejaVu Sans"),
            *[f for f in fm.findSystemFonts() if f.lower().endswith(".ttf")],
        }
    )
    words = [
        "HP",
        "AMMO",
        "SCORE",
        "LVL",
        "QUEST",
        "Objective:",
        "Press E to interact",
        "Paused",
        "Settings",
        "Resume",
        "Mission complete",
        "Reload",
        "Map",
        "FPS",
        "Health",
        "Shield",
        "XP",
        "Gold",
        "Wave",
        "Where are you going?",
        "We have to move now.",
        "Checkpoint reached",
        "Inventory full",
    ]
    tiles = []
    for _ in range(n):
        img = Image.new("RGBA", (width, height), (0, 0, 0, 0))
        d = ImageDraw.Draw(img)
        kind = rng.integers(0, 4)
        col = tuple(int(c) for c in rng.integers(150, 256, 3)) + (255,)
        if (
            kind == 0 or kind == 3
        ):  # text, optionally with a translucent plate (subtitle style)
            if kind == 3:
                d.rectangle(
                    [0, 0, width - 1, height - 1],
                    fill=(0, 0, 0, int(rng.integers(90, 180))),
                )
            text = " ".join(rng.choice(words, size=int(rng.integers(1, 4))))
            if rng.random() < 0.5:
                text += f" {int(rng.integers(0, 999))}"
            fp = font_paths[int(rng.integers(0, len(font_paths)))]
            try:
                font = (
                    ImageFont.truetype(fp, int(rng.integers(14, 30)))
                    if fp
                    else ImageFont.load_default()
                )
            except Exception:
                font = ImageFont.load_default()
            d.text(
                (6, 4),
                text,
                font=font,
                fill=col,
                stroke_width=1,
                stroke_fill=(0, 0, 0, 255),
            )
        elif kind == 1:  # health / progress bar
            frac = rng.uniform(0.1, 1.0)
            d.rectangle(
                [4, 10, width - 5, height - 11], outline=(255, 255, 255, 255), width=2
            )
            d.rectangle([7, 13, 7 + int((width - 15) * frac), height - 14], fill=col)
        else:  # crosshair / minimap frame
            cx, cy = width // 2, height // 2
            d.line([cx - 14, cy, cx + 14, cy], fill=col, width=2)
            d.line([cx, cy - 14, cx, cy + 14], fill=col, width=2)
            d.ellipse([cx - 5, cy - 5, cx + 5, cy + 5], outline=col, width=1)
        tiles.append(np.asarray(img).transpose(2, 0, 1))
    return np.stack(tiles)


def stamp_tiles(canvas, alpha, tiles, idx, xy):
    """Alpha-blend tiles (K, 4, th, tw) float [0,1] into canvas (N, 3, H, W) / alpha (N, 1, H, W) at integer
    positions xy (N, 2) for tile indices idx (N,). Rows with idx < 0 are skipped."""
    th, tw = tiles.shape[-2:]
    H, W = canvas.shape[-2:]
    for i in range(canvas.shape[0]):
        k = int(idx[i])
        if k < 0:
            continue
        x, y = int(xy[i, 0]), int(xy[i, 1])
        x0, y0, x1, y1 = max(x, 0), max(y, 0), min(x + tw, W), min(y + th, H)
        if x1 <= x0 or y1 <= y0:
            continue
        t = tiles[k, :, y0 - y : y1 - y, x0 - x : x1 - x]
        a = t[3:4]
        canvas[i, :, y0:y1, x0:x1] = canvas[i, :, y0:y1, x0:x1] * (1 - a) + t[:3] * a
        alpha[i, :, y0:y1, x0:x1] = alpha[i, :, y0:y1, x0:x1] * (1 - a) + a


# ---------------------------------------------------------------------------------------------------------------
# Scenes
# ---------------------------------------------------------------------------------------------------------------
class Scene:
    """A batch of layered scenes. stills: uint8 (N, S, S, 3) tensor on the target device."""

    def __init__(
        self,
        stills,
        out_px,
        duration,
        g,
        profile=MOTION_PROFILE,
        particles=True,
        text_bank=None,
        world_text_prob=0.3,
        fg_prob=0.85,
    ):
        dev = stills.device
        n, S = stills.shape[0], stills.shape[1]
        rnd = lambda *shape: torch.rand(*shape, device=dev, generator=g)  # noqa: E731
        tex = stills.permute(0, 3, 1, 2).float() / 255.0
        self.n, self.S, self.device, self.out_px = n, S, dev, out_px
        self.bg = tex
        perm = torch.roll(torch.arange(n, device=dev), 1)
        flat = (rnd(n) < 0.25)[:, None, None, None]
        flat_col = rnd(n, 3, 1, 1) * (0.85 + 0.3 * rnd(n, 1, S, S)).clamp(0, 1)
        self.fg = torch.where(flat, flat_col.expand(-1, -1, S, S), tex[perm])
        centre_tex = rnd(n, 2) * 1.0 - 0.5
        self.fg_alpha = (
            foreground_alpha(n, S, g, dev, centre_tex)
            * (rnd(n) < fg_prob).float()[:, None, None, None]
        )
        if (
            text_bank is not None and world_text_prob > 0
        ):  # world-space text moves with the background
            tiles = text_bank.to(dev).float() / 255.0
            canvas, a = self.bg.clone(), torch.zeros(n, 1, S, S, device=dev)
            idx = torch.where(
                rnd(n) < world_text_prob,
                torch.randint(0, len(tiles), (n,), device=dev, generator=g),
                torch.full((n,), -1, device=dev),
            )
            xy = torch.stack(
                [
                    torch.randint(
                        0, max(1, S - tiles.shape[-1]), (n,), device=dev, generator=g
                    ),
                    torch.randint(
                        0, max(1, S - tiles.shape[-2]), (n,), device=dev, generator=g
                    ),
                ],
                1,
            )
            stamp_tiles(canvas, a, tiles, idx, xy)
            self.bg = canvas
        self.cam = random_layer(n, g, dev, out_px, S, duration, profile)
        target_screen = rnd(n, 2) * 1.2 - 0.6
        self.obj = random_layer(
            n,
            g,
            dev,
            out_px,
            S,
            duration,
            profile,
            centre_screen=(centre_tex, target_screen),
            rot_scale=2.0,
        )
        self.d_bg = 0.6 + 0.4 * rnd(n)
        self.d_fg = 0.15 + 0.35 * rnd(n)
        self.exposure = torch.exp2((rnd(n) * 2 - 1) * 0.8)
        k = 6
        self.p_on = (rnd(n, k) < (0.5 if particles else 0.0)).float()
        self.p_pos = rnd(n, k, 2) * 2 - 1
        self.p_vel = (rnd(n, k, 2) * 2 - 1) * 0.06
        self.p_rad = (2 + 10 * rnd(n, k)) * (2.0 / out_px)
        self.p_col = 0.6 + 0.4 * rnd(n, k, 3)
        self.p_amp = 0.2 + 0.5 * rnd(n, k)

    def matrices(self, t):
        t = torch.as_tensor(t, device=self.device, dtype=torch.float32).expand(self.n)
        return layer_matrix(self.cam, t), layer_matrix(self.obj, t)

    def render(self, t, pts):
        """Sample the scene at normalised screen positions pts (N, h, w, 2) at time t.
        Returns colour (N, 3, h, w) in display sRGB, fg coverage (N, 1, h, w), depth, reactive mask.
        """
        m_bg, m_fg = self.matrices(t)
        bg = F.grid_sample(
            self.bg,
            apply_affine(m_bg, pts),
            mode="bilinear",
            padding_mode="reflection",
            align_corners=False,
        )
        g_fg = apply_affine(m_fg, pts)
        fg = F.grid_sample(
            self.fg, g_fg, mode="bilinear", padding_mode="zeros", align_corners=False
        )
        a = F.grid_sample(
            self.fg_alpha,
            g_fg,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )
        col = a * fg + (1 - a) * bg
        tex_bg = apply_affine(m_bg, pts)
        d_bg = self.d_bg[:, None, None, None] * (
            1 + 0.3 * tex_bg[..., 1:2].permute(0, 3, 1, 2)
        )
        depth = torch.where(a > 0.5, self.d_fg[:, None, None, None].expand_as(a), d_bg)
        tt = (
            torch.as_tensor(t, device=self.device, dtype=torch.float32)
            .expand(self.n)
            .view(self.n, 1, 1)
        )
        pos = self.p_pos + self.p_vel * tt  # (N, K, 2)
        d2 = ((pts[:, None] - pos[:, :, None, None, :]) ** 2).sum(-1)  # (N, K, h, w)
        gk = (
            self.p_on[..., None, None]
            * self.p_amp[..., None, None]
            * torch.exp(-d2 / (2 * self.p_rad[..., None, None] ** 2))
        )
        reactive = 1 - torch.prod(1 - gk, dim=1, keepdim=True)
        for k in range(gk.shape[1]):
            col = (
                col * (1 - gk[:, k : k + 1])
                + self.p_col[:, k, :, None, None] * gk[:, k : k + 1]
            )
        lin = srgb_to_linear(col.clamp(0, 1)) * self.exposure[:, None, None, None]
        return linear_to_srgb(lin), a, depth, reactive

    def frame(self, t, h, w, jitter=None, ssaa=False):
        """Render a full frame of h x w pixels. jitter: (N, 2) sub-pixel offsets (pixels) or None."""
        if not ssaa:
            return self.render(t, pixel_grid(self.n, h, w, self.device, jitter))
        acc = None
        for ox, oy in SSAA_OFFSETS:
            off = torch.tensor([[ox, oy]], device=self.device).expand(self.n, 2)
            col, _, _, _ = self.render(t, pixel_grid(self.n, h, w, self.device, off))
            acc = col if acc is None else acc + col
        _, a, depth, reactive = self.render(t, pixel_grid(self.n, h, w, self.device))
        return acc / len(SSAA_OFFSETS), a, depth, reactive

    def flow(self, t_from, t_to, h, w, jitter=None):
        """For every pixel of the frame at t_from: displacement (pixels of an h x w grid) to the same surface point
        at t_to, of the visible layer, and validity (1 = visible at t_to, inside the view, not occluded).
        """
        pts = pixel_grid(self.n, h, w, self.device, jitter)
        bg_f, fg_f = self.matrices(t_from)
        bg_t, fg_t = self.matrices(t_to)
        a_from = F.grid_sample(
            self.fg_alpha,
            apply_affine(fg_f, pts),
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )
        to_bg = apply_affine(invert_affine(bg_t), apply_affine(bg_f, pts))
        to_fg = apply_affine(invert_affine(fg_t), apply_affine(fg_f, pts))
        on_fg = (a_from > 0.5).permute(0, 2, 3, 1)
        dest = torch.where(on_fg, to_fg, to_bg)
        a_to = F.grid_sample(
            self.fg_alpha,
            apply_affine(fg_t, to_bg),
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )
        inside = (dest.abs() <= 1.0).all(-1, keepdim=True).permute(0, 3, 1, 2)
        visible = torch.where(
            on_fg.permute(0, 3, 1, 2),
            torch.ones_like(a_to, dtype=torch.bool),
            a_to < 0.5,
        )
        disp = (dest - pts) * torch.tensor([w / 2.0, h / 2.0], device=self.device)
        return disp.permute(0, 3, 1, 2), (inside & visible).float()


# ---------------------------------------------------------------------------------------------------------------
# Batches for the two tasks
# ---------------------------------------------------------------------------------------------------------------
def sr_batch(
    stills,
    scale,
    T,
    P,
    g,
    jitter_phases=16,
    profile=MOTION_PROFILE,
    text_bank=None,
    room_frames=None,
):
    """Temporal super-resolution sequence batch. Output-resolution P x P targets with 4-sample anti-aliasing,
    jittered single-sample low-resolution inputs of (P / scale)^2, exact motion vectors at low resolution
    (current -> previous frame, low-resolution pixels), validity, depth, reactive masks and exposure.
    room_frames: number of frames the camera path must stay inside the texture (default T); a smaller value keeps
    fast motion in long sequences, after which the view continues onto the mirrored texture (still exact).
    """
    n, dev = stills.shape[0], stills.device
    h = P // scale
    scene = Scene(stills, P, room_frames or T, g, profile=profile, text_bank=text_bank)
    phase = torch.randint(0, jitter_phases, (n,), device=dev, generator=g)
    seq = jitter_sequence(T + jitter_phases, jitter_phases).to(dev)
    out = {
        k: []
        for k in (
            "lr",
            "hr",
            "mv_lr",
            "valid_lr",
            "mv_hr",
            "valid_hr",
            "depth_lr",
            "reactive_lr",
            "jitter",
        )
    }
    for t in range(T):
        jit = seq[(phase + t) % jitter_phases]
        lr, _, depth, reactive = scene.frame(float(t), h, h, jitter=jit)
        hr, _, _, _ = scene.frame(float(t), P, P, ssaa=True)
        if t == 0:
            mv_lr, va_lr = torch.zeros(n, 2, h, h, device=dev), torch.zeros(
                n, 1, h, h, device=dev
            )
            mv_hr, va_hr = torch.zeros(n, 2, P, P, device=dev), torch.zeros(
                n, 1, P, P, device=dev
            )
        else:
            mv_lr, va_lr = scene.flow(float(t), float(t - 1), h, h, jitter=jit)
            mv_hr, va_hr = scene.flow(float(t), float(t - 1), P, P)
        for k, v in zip(
            out, (lr, hr, mv_lr, va_lr, mv_hr, va_hr, depth, reactive, jit)
        ):
            out[k].append(v)
    batch = {k: torch.stack(v, 1) for k, v in out.items()}
    batch["depth_lr"] = 1.0 / (
        1.0
        + batch["depth_lr"]
        / batch["depth_lr"].flatten(1).median(1).values.view(n, 1, 1, 1, 1)
    )
    batch["exposure"] = torch.log2(scene.exposure)[:, None].expand(n, T)
    batch["has_depth"] = torch.ones(n, device=dev)
    batch["mv_quality"] = torch.ones(n, device=dev)
    batch["speed_class"] = scene.cam["speed_class"]
    return batch


def ui_layer(n, H, W, tiles, g, device, n_slots=4, menu=None):
    """Screen-space HUD for n samples in two states (before and after a subtitle / HUD change at t = 1).
    Returns premultiplied colour [(N, 3, H, W)] * 2 and alpha [(N, 1, H, W)] * 2; composite as
    frame * (1 - alpha) + colour. tiles: float (K, 4, th, tw). menu: bool (N,) adds a full-screen panel.
    """
    rgb = [torch.zeros(n, 3, H, W, device=device) for _ in range(2)]
    alp = [torch.zeros(n, 1, H, W, device=device) for _ in range(2)]
    if menu is not None:
        m = menu[:, None, None, None].float()
        for s in range(2):
            rgb[s] = rgb[s] + 0.6 * 0.05 * m
            alp[s] = alp[s] + 0.6 * m
    K, _, th, tw = tiles.shape
    for slot in range(n_slots):
        on = torch.rand(n, device=device, generator=g) < 0.7
        idx = torch.where(
            on,
            torch.randint(0, K, (n,), device=device, generator=g),
            torch.full((n,), -1, device=device),
        )
        xy = torch.stack(
            [
                torch.randint(0, max(1, W - tw // 2), (n,), device=device, generator=g),
                torch.randint(0, max(1, H - th // 2), (n,), device=device, generator=g),
            ],
            1,
        )
        change = (torch.rand(n, device=device, generator=g) < 0.3) & on
        idx2 = torch.where(
            change, torch.randint(0, K, (n,), device=device, generator=g), idx
        )
        stamp_tiles(rgb[0], alp[0], tiles, idx, xy)
        stamp_tiles(rgb[1], alp[1], tiles, idx2, xy)
    return rgb, alp


def fg_batch(
    stills,
    P,
    g,
    tiles=None,
    p_extrap=0.5,
    p_ui=0.5,
    p_cut=0.03,
    p_menu=0.02,
    p_flip=0.1,
    profile=MOTION_PROFILE,
):
    """Frame-generation batch at P x P. Real frames I0 (t=0) and I1 (t=1); target at tau = 0.5 (interpolation)
    or tau = 1.5 (extrapolation). Returns frames, the engine motion vector of I1 (I1 -> I0), depth of I1, the
    camera-only flow from the target to I1 (known from the latest input, available to extrapolation), exact
    target -> I0 / I1 flows, UI composites and masks, and flags for cuts and menus."""
    n, dev = stills.shape[0], stills.device
    scene = Scene(stills, P, 2.0, g, profile=profile)
    flip = (
        torch.rand(n, device=dev, generator=g) < p_flip
    )  # rapid input change: camera reverses after t = 1
    scene.cam["flip"], scene.cam["t_flip"] = flip.float(), torch.ones(n, device=dev)
    extrap = torch.rand(n, device=dev, generator=g) < p_extrap
    tau = torch.where(
        extrap, torch.full((n,), 1.5, device=dev), torch.full((n,), 0.5, device=dev)
    )
    i0, _, _, _ = scene.frame(0.0, P, P, ssaa=True)
    i1, _, d1, _ = scene.frame(1.0, P, P, ssaa=True)
    tgt, _, _, _ = scene.frame(tau, P, P, ssaa=True)
    mv1, mv1_valid = scene.flow(1.0, 0.0, P, P)
    f_t0, v_t0 = scene.flow(tau, 0.0, P, P)
    f_t1, v_t1 = scene.flow(tau, 1.0, P, P)
    # camera-only prior: map every target pixel through the background (camera) layer to t = 1
    pts = pixel_grid(n, P, P, dev)
    m_bg, bg1 = layer_matrix(scene.cam, tau), layer_matrix(
        scene.cam, torch.ones(n, device=dev)
    )
    to1 = apply_affine(invert_affine(bg1), apply_affine(m_bg, pts))
    cam_prior = ((to1 - pts) * (P / 2.0)).permute(0, 3, 1, 2)
    batch = {
        "i0": i0,
        "i1": i1,
        "target": tgt,
        "tau": tau,
        "extrap": extrap.float(),
        "mv1": mv1,
        "mv1_valid": mv1_valid,
        "depth1": 1.0 / (1.0 + d1 / d1.flatten(1).median(1).values.view(n, 1, 1, 1)),
        "cam_prior": cam_prior,
        "flow_t0": f_t0,
        "flow_t1": f_t1,
        "valid_t0": v_t0,
        "valid_t1": v_t1,
        "speed_class": scene.cam["speed_class"],
        "flip": flip.float(),
    }
    # scene cuts: I1 and the target come from a different scene; the correct output is I1 (repeat)
    cut = torch.rand(n, device=dev, generator=g) < p_cut
    if cut.any():
        perm = torch.roll(torch.arange(n, device=dev), 1)
        for k in ("i1", "mv1", "depth1"):
            batch[k] = torch.where(cut[:, None, None, None], batch[k][perm], batch[k])
        batch["target"] = torch.where(
            cut[:, None, None, None], batch["i1"], batch["target"]
        )
        for k in ("mv1_valid", "valid_t0", "valid_t1"):
            batch[k] = torch.where(
                cut[:, None, None, None], torch.zeros_like(batch[k]), batch[k]
            )
    batch["cut"] = cut.float()
    # HUD / subtitles / menus composited in screen space after rendering (static, may change at t = 1)
    batch["ui_mask"] = torch.zeros(n, 1, P, P, device=dev)
    batch["menu"] = torch.zeros(n, device=dev)
    if tiles is not None:
        use_ui = torch.rand(n, device=dev, generator=g) < p_ui
        menu = (torch.rand(n, device=dev, generator=g) < p_menu) & use_ui
        rgb, alp = ui_layer(n, P, P, tiles, g, dev, menu=menu)
        u = use_ui[:, None, None, None].float()
        a0, a1, c0, c1 = alp[0] * u, alp[1] * u, rgb[0] * u, rgb[1] * u
        batch["hudless_i0"], batch["hudless_i1"], batch["hudless_target"] = (
            batch["i0"],
            batch["i1"],
            batch["target"],
        )
        batch["ui_rgb"], batch["ui_alpha"] = c1, a1
        batch["i0"] = batch["i0"] * (1 - a0) + c0
        batch["i1"] = batch["i1"] * (1 - a1) + c1
        # the UI shown in a generated frame is the UI of the newest real frame (I1): never interpolated
        batch["target"] = batch["target"] * (1 - a1) + c1
        batch["ui_mask"] = torch.maximum(a0, a1)
        batch["menu"] = menu.float()
    return batch


# ---------------------------------------------------------------------------------------------------------------
# Batches from cached real sequences (TartanAir, Sintel, video), built on the GPU
# ---------------------------------------------------------------------------------------------------------------
def _flip_fields(x, fx, fy, vec=False):
    """Flip (N, T, C, H, W) or (N, C, H, W) tensors per sample; vector fields also negate their components."""
    shape = (-1,) + (1,) * (x.dim() - 1)
    x = torch.where(fx.view(shape), x.flip(-1), x)
    x = torch.where(fy.view(shape), x.flip(-2), x)
    if vec:
        cdim = x.dim() - 3
        sx = torch.where(fx, -1.0, 1.0).view(shape)
        sy = torch.where(fy, -1.0, 1.0).view(shape)
        x = torch.cat([x.narrow(cdim, 0, 1) * sx, x.narrow(cdim, 1, 1) * sy], cdim)
    return x


def sr_batch_from_frames(
    rgb,
    mv_half,
    valid_half,
    depth_half,
    has_depth,
    mv_quality,
    scale,
    g,
    jitter_phases=16,
):
    """Temporal SR batch from cached real frames: rgb uint8 (N, T, P, P, 3); mv_half (N, T, P/2, P/2, 2) in P-grid
    pixels (current -> previous); valid_half (N, T, P/2, P/2); depth_half (N, T, P/2, P/2). The low-resolution
    input is a jittered single-sample read of the full-resolution frame (aliased, like a low-resolution render).
    """
    n, T, P = rgb.shape[0], rgb.shape[1], rgb.shape[2]
    dev, h = rgb.device, P // scale
    hr = rgb.permute(0, 1, 4, 2, 3).float() / 255.0
    mv = mv_half.permute(0, 1, 4, 2, 3).float()
    valid = valid_half[:, :, None].float()
    depth = depth_half[:, :, None].float()
    fx = torch.rand(n, device=dev, generator=g) < 0.5
    fy = torch.rand(n, device=dev, generator=g) < 0.5
    hr, valid, depth = (
        _flip_fields(hr, fx, fy),
        _flip_fields(valid, fx, fy),
        _flip_fields(depth, fx, fy),
    )
    mv = _flip_fields(mv, fx, fy, vec=True)
    gain = torch.exp2((torch.rand(n, device=dev, generator=g) * 2 - 1) * 0.8)
    hr = linear_to_srgb(srgb_to_linear(hr) * gain.view(n, 1, 1, 1, 1))
    phase = torch.randint(0, jitter_phases, (n,), device=dev, generator=g)
    seq = jitter_sequence(T + jitter_phases, jitter_phases).to(dev)
    keys = (
        "lr",
        "hr",
        "mv_lr",
        "valid_lr",
        "mv_hr",
        "valid_hr",
        "depth_lr",
        "reactive_lr",
        "jitter",
    )
    out = {k: [] for k in keys}
    for t in range(T):
        jit = seq[(phase + t) % jitter_phases]
        pts = pixel_grid(n, h, h, dev, jit)
        lr = F.grid_sample(
            hr[:, t], pts, mode="bilinear", padding_mode="border", align_corners=False
        )
        mv_lr = (
            F.grid_sample(
                mv[:, t],
                pts,
                mode="bilinear",
                padding_mode="border",
                align_corners=False,
            )
            / scale
        )
        va_lr = F.grid_sample(
            valid[:, t], pts, mode="nearest", padding_mode="border", align_corners=False
        )
        d_lr = F.grid_sample(
            depth[:, t],
            pts,
            mode="bilinear",
            padding_mode="border",
            align_corners=False,
        )
        mv_hr = F.interpolate(
            mv[:, t], scale_factor=2, mode="bilinear", align_corners=False
        )
        va_hr = F.interpolate(valid[:, t], scale_factor=2, mode="nearest")
        for k, v in zip(
            keys,
            (
                lr,
                hr[:, t],
                mv_lr,
                va_lr,
                mv_hr,
                va_hr,
                d_lr,
                torch.zeros_like(va_lr),
                jit,
            ),
        ):
            out[k].append(v)
    batch = {k: torch.stack(v, 1) for k, v in out.items()}
    batch["exposure"] = torch.log2(gain)[:, None].expand(n, T)
    batch["has_depth"], batch["mv_quality"] = has_depth.float(), mv_quality.float()
    batch["speed_class"] = torch.full((n,), -1, device=dev, dtype=torch.long)
    return batch


def fg_batch_from_frames(
    rgb, flows_half, flow_ok, g, tiles=None, p_extrap=0.5, p_ui=0.5
):
    """Frame-generation batch from cached real quads: rgb uint8 (N, 4, P, P, 3) = frames k..k+3; flows_half
    (N, 4, P/2, P/2, 2) in P-grid pixels = [2->0, 1->0, 1->2, 3->2] (teacher optical flow); flow_ok (N,).
    Interpolation: I0 = k, I1 = k+2, target k+1. Extrapolation: I0 = k, I1 = k+2, target k+3 (tau = 1.5).
    """
    n, P, dev = rgb.shape[0], rgb.shape[2], rgb.device
    fr = rgb.permute(0, 1, 4, 2, 3).float() / 255.0
    fl = flows_half.permute(0, 1, 4, 2, 3).float()
    fx = torch.rand(n, device=dev, generator=g) < 0.5
    fy = torch.rand(n, device=dev, generator=g) < 0.5
    fr, fl = _flip_fields(fr, fx, fy), _flip_fields(fl, fx, fy, vec=True)
    up = lambda x: F.interpolate(
        x, scale_factor=2, mode="bilinear", align_corners=False
    )  # noqa: E731
    extrap = torch.rand(n, device=dev, generator=g) < p_extrap
    e4 = extrap.view(n, 1, 1, 1)
    ok = flow_ok.float().view(n, 1, 1, 1).expand(n, 1, P, P)
    batch = {
        "i0": fr[:, 0],
        "i1": fr[:, 2],
        "target": torch.where(e4, fr[:, 3], fr[:, 1]),
        "tau": torch.where(extrap, 1.5, 0.5).float(),
        "extrap": extrap.float(),
        "mv1": up(fl[:, 0]),
        "mv1_valid": ok,
        "depth1": torch.zeros(n, 1, P, P, device=dev),
        "cam_prior": torch.zeros(n, 2, P, P, device=dev),
        "flow_t0": up(fl[:, 1]),
        "valid_t0": ok * (~e4).float(),
        "flow_t1": torch.where(e4, up(fl[:, 3]), up(fl[:, 2])),
        "valid_t1": ok,
        "speed_class": torch.full((n,), -1, device=dev, dtype=torch.long),
        "flip": torch.zeros(n, device=dev),
        "cut": torch.zeros(n, device=dev),
        "menu": torch.zeros(n, device=dev),
        "ui_mask": torch.zeros(n, 1, P, P, device=dev),
    }
    if tiles is not None:
        use_ui = torch.rand(n, device=dev, generator=g) < p_ui
        rgb_ui, alp = ui_layer(n, P, P, tiles, g, dev)
        u = use_ui[:, None, None, None].float()
        a0, a1, c0, c1 = alp[0] * u, alp[1] * u, rgb_ui[0] * u, rgb_ui[1] * u
        batch["hudless_i0"], batch["hudless_i1"], batch["hudless_target"] = (
            batch["i0"],
            batch["i1"],
            batch["target"],
        )
        batch["ui_rgb"], batch["ui_alpha"] = c1, a1
        batch["i0"] = batch["i0"] * (1 - a0) + c0
        batch["i1"] = batch["i1"] * (1 - a1) + c1
        batch["target"] = batch["target"] * (1 - a1) + c1
        batch["ui_mask"] = torch.maximum(a0, a1)
    return batch

**`nss_models.py`**: RepConv blocks with online re-parameterisation, the temporal super-resolution network, the frame generator, tiers, fusion and ONNX wrappers.

In [ ]:
%%writefile /kaggle/working/code/nss_models.py
"""Models: NeuralSS temporal super-resolution (TSR) and low-latency frame generation (FG).

Both are built from RepConv blocks: during training each block is a sum of linear branches (3x3, 1x1, identity
and fixed Sobel-x, Sobel-y, Laplacian filters with a learnable 1x1 projection, after ECBSR). The branches are
folded into one 3x3 kernel on every forward pass ("online re-parameterisation"), so training costs one
convolution per block and the deployed network is a plain chain of 3x3 convolutions, which maps to any GPU API
(DirectML, Vulkan / compute shaders, Metal, ONNX Runtime).
"""

import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

SOBEL_X = [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]
SOBEL_Y = [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]]
LAPLACIAN = [[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]]


class RepConv(nn.Module):
    """3x3 + 1x1 + identity + fixed edge filters, folded into one 3x3 convolution on every call."""

    def __init__(self, cin, cout, act=True):
        super().__init__()
        self.cin, self.cout = cin, cout
        self.conv3 = nn.Conv2d(cin, cout, 3, padding=1)
        self.conv1 = nn.Conv2d(cin, cout, 1)
        self.edge_pw = nn.Conv2d(3 * cin, cout, 1)
        self.register_buffer(
            "edges", torch.tensor([SOBEL_X, SOBEL_Y, LAPLACIAN]), persistent=False
        )
        self.act = nn.PReLU(cout) if act else nn.Identity()

    def weight_bias(self):
        w = self.conv3.weight + F.pad(self.conv1.weight, [1, 1, 1, 1])
        pw = self.edge_pw.weight[:, :, 0, 0].reshape(self.cout, self.cin, 3)
        w = w + torch.einsum("oik,khw->oihw", pw, self.edges.to(pw.dtype))
        if self.cin == self.cout:
            w = w + torch.eye(self.cin, device=w.device, dtype=w.dtype)[
                :, :, None, None
            ] * F.pad(
                torch.ones(1, 1, 1, 1, device=w.device, dtype=w.dtype), [1, 1, 1, 1]
            )
        return w, self.conv3.bias + self.conv1.bias + self.edge_pw.bias

    def forward(self, x):
        w, b = self.weight_bias()
        return self.act(F.conv2d(x, w, b, padding=1))

    @torch.no_grad()
    def fused(self):
        w, b = self.weight_bias()
        conv = nn.Conv2d(self.cin, self.cout, 3, padding=1).to(w.device, w.dtype)
        conv.weight.copy_(w)
        conv.bias.copy_(b)
        return nn.Sequential(conv, copy.deepcopy(self.act))


def fuse_model(model):
    """Deep copy with every RepConv replaced by its fused Conv2d (+ activation)."""
    fused = copy.deepcopy(model).eval()

    def swap(module):
        for name, child in module.named_children():
            if isinstance(child, RepConv):
                setattr(module, name, child.fused())
            else:
                swap(child)

    swap(fused)
    return fused


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def backward_warp(img, flow):
    """Sample img at (x + flow) for every output pixel; flow (N, 2, H, W) in pixels of img."""
    n, _, h, w = img.shape
    ys = torch.arange(h, device=img.device, dtype=torch.float32)
    xs = torch.arange(w, device=img.device, dtype=torch.float32)
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    fx, fy = flow[:, 0].float(), flow[:, 1].float()
    grid = torch.stack(
        [
            (gx[None] + fx + 0.5) * (2.0 / w) - 1.0,
            (gy[None] + fy + 0.5) * (2.0 / h) - 1.0,
        ],
        -1,
    )
    return F.grid_sample(
        img.float(), grid, mode="bilinear", padding_mode="border", align_corners=False
    ).to(img.dtype)


def upsample_flow(flow, scale):
    return (
        F.interpolate(
            flow.float(), scale_factor=scale, mode="bilinear", align_corners=False
        )
        * scale
    )


def pad_to(x, multiple):
    h, w = x.shape[-2:]
    ph, pw = (-h) % multiple, (-w) % multiple
    return (F.pad(x, (0, pw, 0, ph), mode="replicate") if ph or pw else x), (h, w)


class Trunk(nn.Module):
    """Plain RepConv chain with an optional half-resolution U-Net level (strided conv down, pixel-shuffle up)."""

    def __init__(self, cin, width, blocks, unet_blocks):
        super().__init__()
        self.head = RepConv(cin, width)
        n_a = blocks // 2
        self.a = nn.Sequential(*[RepConv(width, width) for _ in range(n_a)])
        self.unet = unet_blocks > 0
        if self.unet:
            self.down = nn.Sequential(
                nn.Conv2d(width, 2 * width, 3, stride=2, padding=1), nn.PReLU(2 * width)
            )
            self.mid = nn.Sequential(
                *[RepConv(2 * width, 2 * width) for _ in range(unet_blocks)]
            )
            self.up = nn.Sequential(
                nn.Conv2d(2 * width, 4 * width, 1), nn.PixelShuffle(2)
            )
        self.b = nn.Sequential(*[RepConv(width, width) for _ in range(blocks - n_a)])

    def forward(self, x):
        x = self.a(self.head(x))
        if self.unet:
            x = x + self.up(self.mid(self.down(x)))
        return self.b(x)


# ---------------------------------------------------------------------------------------------------------------
# Temporal super-resolution
# ---------------------------------------------------------------------------------------------------------------
TSR_TIERS = {
    "igpu": {"width": 16, "blocks": 3, "unet_blocks": 0, "hidden": 8},
    "low": {"width": 24, "blocks": 4, "unet_blocks": 2, "hidden": 8},
    "mid": {"width": 32, "blocks": 4, "unet_blocks": 3, "hidden": 16},
    "high": {"width": 48, "blocks": 6, "unet_blocks": 4, "hidden": 16},
    "teacher": {"width": 64, "blocks": 8, "unet_blocks": 6, "hidden": 32},
}


def normals_from_depth(d):
    """View-space normal estimate from normalised depth (N, 1, h, w) by central differences."""
    dp = F.pad(d, (1, 1, 1, 1), mode="replicate")
    gx = (dp[..., 1:-1, 2:] - dp[..., 1:-1, :-2]) * 8.0
    gy = (dp[..., 2:, 1:-1] - dp[..., :-2, 1:-1]) * 8.0
    n = torch.cat([-gx, -gy, torch.ones_like(d)], 1)
    return n / n.norm(dim=1, keepdim=True)


class TSRNet(nn.Module):
    """Recurrent temporal super-resolution at low resolution.

    Per frame inputs (low resolution h x w): jittered colour, depth (+ normals derived from it), motion vectors
    (current -> previous, low-res pixels), a depth-based disocclusion cue, reactive mask, exposure, jitter offset
    and availability flags; plus the previous output warped to the current frame at full resolution and folded to
    low resolution (space-to-depth), and a warped recurrent hidden state.

    Output: full-resolution colour = a * warped_history + (1 - a) * jitter-aware upsample + residual, where the
    per-pixel history weight a and the residual are predicted at low resolution and unfolded (pixel shuffle).
    """

    def __init__(self, scale, width, blocks, unet_blocks, hidden):
        super().__init__()
        self.scale, self.hidden = scale, hidden
        s2 = scale * scale
        cin = 3 + 1 + 3 + 2 + 1 + 1 + 1 + 2 + 2 + 3 * s2 + 3 + hidden
        self.trunk = Trunk(cin, width, blocks, unet_blocks)
        self.out = nn.Conv2d(width, 4 * s2, 3, padding=1)
        self.to_hidden = nn.Conv2d(width, hidden, 3, padding=1)
        self.shuffle = nn.PixelShuffle(scale)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def base_upsample(self, color, jitter):
        """Bilinear reconstruction at output pixel centres that undoes the sub-pixel jitter of the samples."""
        n, _, h, w = color.shape
        s = self.scale
        H, W = h * s, w * s
        ys = (torch.arange(H, device=color.device, dtype=torch.float32) + 0.5) / s
        xs = (torch.arange(W, device=color.device, dtype=torch.float32) + 0.5) / s
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        jx, jy = jitter[:, 0, None, None].float(), jitter[:, 1, None, None].float()
        grid = torch.stack(
            [(gx[None] - jx) * (2.0 / w) - 1.0, (gy[None] - jy) * (2.0 / h) - 1.0], -1
        )
        return F.grid_sample(
            color.float(),
            grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=False,
        )

    def forward(self, f, state=None, use_history=True):
        """f: dict with color (N,3,h,w), depth (N,1,h,w), mv (N,2,h,w), reactive (N,1,h,w), exposure (N,),
        jitter (N,2), has_depth (N,), mv_quality (N,). Returns (output (N,3,H,W), new state).
        """
        color, depth, mv, jitter = f["color"], f["depth"], f["mv"], f["jitter"]
        n, _, h, w = color.shape
        s = self.scale
        base = self.base_upsample(color, jitter)
        if state is None or not use_history:
            hist, hidden, d_prev, first = (
                base,
                color.new_zeros(n, self.hidden, h, w),
                depth,
                True,
            )
        else:
            mv_hr = upsample_flow(mv, s)
            hist = backward_warp(state["out"], mv_hr)
            hidden = backward_warp(state["hidden"], mv)
            d_prev, first = backward_warp(state["depth"], mv), False
        has_d = f["has_depth"].view(n, 1, 1, 1).to(color.dtype)
        disocc = ((depth - d_prev).abs() / (depth.abs() + 1e-3)).clamp(0, 1) * has_d
        hist_lr = F.avg_pool2d(hist, s)
        feats = [
            color,
            depth * has_d,
            normals_from_depth(depth) * has_d,
            mv * 0.125,
            disocc,
            f["reactive"],
            f["exposure"].view(n, 1, 1, 1).expand(n, 1, h, w).to(color.dtype),
            jitter.view(n, 2, 1, 1).expand(n, 2, h, w).to(color.dtype),
            has_d.expand(n, 1, h, w),
            f["mv_quality"].view(n, 1, 1, 1).expand(n, 1, h, w).to(color.dtype),
            F.pixel_unshuffle(hist.to(color.dtype), s) * (0.0 if first else 1.0),
            (hist_lr.to(color.dtype) - color).abs() * (0.0 if first else 1.0),
            hidden.to(color.dtype),
        ]
        x, (h0, w0) = pad_to(torch.cat(feats, 1), 2)
        y = self.trunk(x)
        o = self.out(y)[..., :h0, :w0]
        new_hidden = torch.tanh(self.to_hidden(y))[..., :h0, :w0]
        o = self.shuffle(o.float())
        residual, a = o[:, :3], torch.sigmoid(o[:, 3:4] + 2.0)
        if first:
            a = torch.zeros_like(a)
        out = a * hist + (1 - a) * base + residual
        return out, {
            "out": out,
            "hidden": new_hidden,
            "depth": depth,
            "first": False,
            "alpha": a,
        }


def build_tsr(tier, scale):
    return TSRNet(scale, **TSR_TIERS[tier])


def tsr_macs_per_lr_pixel(model, sample_hw=(64, 64)):
    """Multiply-accumulates per low-resolution pixel, counted with forward hooks on a fused copy."""
    fused = fuse_model(model).float().cpu()
    total = [0]

    def hook(mod, inp, out):
        if isinstance(mod, nn.Conv2d):
            total[0] += (
                out.numel()
                // out.shape[0]
                * mod.in_channels
                // mod.groups
                * mod.kernel_size[0]
                * mod.kernel_size[1]
            )

    hooks = [
        m.register_forward_hook(hook)
        for m in fused.modules()
        if isinstance(m, nn.Conv2d)
    ]
    h, w = sample_hw
    f = {
        "color": torch.rand(1, 3, h, w),
        "depth": torch.rand(1, 1, h, w),
        "mv": torch.zeros(1, 2, h, w),
        "reactive": torch.zeros(1, 1, h, w),
        "exposure": torch.zeros(1),
        "jitter": torch.zeros(1, 2),
        "has_depth": torch.ones(1),
        "mv_quality": torch.ones(1),
    }
    with torch.no_grad():
        fused(f, None)
    for hk in hooks:
        hk.remove()
    return total[0] / (h * w)


# ---------------------------------------------------------------------------------------------------------------
# Frame generation
# ---------------------------------------------------------------------------------------------------------------
FG_TIERS = {
    "igpu": {"width": 16, "levels": 1, "blocks": 3},
    "low": {"width": 24, "levels": 1, "blocks": 4},
    "mid": {"width": 32, "levels": 2, "blocks": 4},
    "high": {"width": 48, "levels": 2, "blocks": 5},
    "teacher": {"width": 64, "levels": 2, "blocks": 6},
}


class FGNet(nn.Module):
    """Coarse-to-fine intermediate-flow frame generator for interpolation (tau = 0.5) and extrapolation
    (tau = 1.5) from real frames I0 and I1.

    Motion priors: the engine motion vector of I1 (I1 -> I0) gives linear-motion flows from the target time,
    F(tau->0) = tau * M1 and F(tau->1) = (tau - 1) * M1; for extrapolation the camera-only flow of the latest input
    (target -> I1) is also given. Each level (1/4, then 1/2 resolution) predicts flow corrections, a blend
    weight, a residual and a UI/static-overlay probability. The output is
        out = u * I1 + (1 - u) * (m * warp(I0) + (1 - m) * warp(I1) + residual)
    so text, HUD and menus are taken from the newest real frame instead of being interpolated.
    The privileged teacher additionally sees the true target frame (training only)."""

    def __init__(self, width, levels, blocks, privileged=False):
        super().__init__()
        self.levels = [4, 2][:levels] if levels > 1 else [4]
        self.privileged = privileged
        cin = 3 * 4 + 2 * 3 + 1 + 2 + 3 + (3 if privileged else 0)
        self.nets = nn.ModuleList()
        for _ in self.levels:
            trunk = Trunk(cin + 2, width, blocks, 0)
            head = nn.Conv2d(width, 2 + 2 + 1 + 1 + 3, 3, padding=1)
            nn.init.zeros_(head.weight)
            nn.init.zeros_(head.bias)
            with torch.no_grad():
                head.bias[5] = -4.0  # UI probability starts near zero
            self.nets.append(nn.ModuleDict({"trunk": trunk, "head": head}))

    @staticmethod
    def priors(f):
        n = f["i0"].shape[0]
        tau = f["tau"].view(n, 1, 1, 1)
        m1 = f["mv1"] * f["mv_avail"].view(n, 1, 1, 1)
        f0, f1 = tau * m1, (tau - 1.0) * m1
        cam = f["cam_prior"] * f["cam_avail"].view(n, 1, 1, 1)
        use_cam = (f["cam_avail"] > 0) & (f["mv_avail"] <= 0)
        f1 = torch.where(use_cam.view(n, 1, 1, 1), cam, f1)
        return f0, f1

    def forward(self, f):
        i0, i1 = f["i0"], f["i1"]
        n, _, H, W = i0.shape
        i0p, (H0, W0) = pad_to(i0, 8)
        i1p = pad_to(i1, 8)[0]
        f0, f1 = (pad_to(x, 8)[0] for x in self.priors(f))
        cam = pad_to(f["cam_prior"] * f["cam_avail"].view(n, 1, 1, 1), 8)[0]
        depth = pad_to(f["depth1"] * f["depth_avail"].view(n, 1, 1, 1), 8)[0]
        gt = pad_to(f["target"], 8)[0] if self.privileged else None
        Hp, Wp = i0p.shape[-2:]
        blend = torch.zeros(n, 1, Hp, Wp, device=i0.device)
        ui = torch.full((n, 1, Hp, Wp), -4.0, device=i0.device)
        residual = torch.zeros(n, 3, Hp, Wp, device=i0.device)
        consts = torch.stack(
            [
                f["tau"] - 1.0,
                f["extrap"],
                f["mv_avail"],
                f["cam_avail"],
                f["depth_avail"],
            ],
            1,
        ).view(n, 5, 1, 1)
        for d, net in zip(self.levels, self.nets):
            down = lambda x: F.avg_pool2d(x.float(), d)  # noqa: E731
            w0, w1 = backward_warp(i0p, f0), backward_warp(i1p, f1)
            x = [
                down(i0p),
                down(i1p),
                down(w0),
                down(w1),
                down(f0) / (d * 8),
                down(f1) / (d * 8),
                down(cam) / (d * 8),
                down(depth),
                consts[:, :2].expand(n, 2, Hp // d, Wp // d),
                consts[:, 2:].expand(n, 3, Hp // d, Wp // d),
            ]
            if gt is not None:
                x.append(down(gt))
            x += [F.avg_pool2d(blend, d), F.avg_pool2d(ui, d)]
            y = net["head"](net["trunk"](torch.cat(x, 1))).float()
            up = lambda t: F.interpolate(
                t, scale_factor=d, mode="bilinear", align_corners=False
            )  # noqa: E731
            f0 = f0 + up(y[:, 0:2]) * d
            f1 = f1 + up(y[:, 2:4]) * d
            blend = blend + up(y[:, 4:5])
            ui = ui + up(y[:, 5:6])
            residual = residual + up(y[:, 6:9])
        w0, w1 = backward_warp(i0p, f0), backward_warp(i1p, f1)
        m, u = torch.sigmoid(blend), torch.sigmoid(ui)
        gen = m * w0 + (1 - m) * w1 + residual
        out = u * i1p.float() + (1 - u) * gen
        crop = (..., slice(0, H0), slice(0, W0))
        return out[crop], {
            "flow0": f0[crop],
            "flow1": f1[crop],
            "blend": m[crop],
            "ui": u[crop],
            "ui_logit": ui[crop],
            "gen": gen[crop],
        }


def build_fg(tier):
    spec = dict(FG_TIERS[tier])
    return FGNet(
        spec["width"], spec["levels"], spec["blocks"], privileged=(tier == "teacher")
    )


class TSRExport(nn.Module):
    """ONNX wrapper of one fused TSR step with explicit recurrent state tensors."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(
        self,
        color,
        depth,
        mv,
        reactive,
        exposure,
        jitter,
        flags,
        prev_out,
        prev_hidden,
        prev_depth,
    ):
        f = {
            "color": color,
            "depth": depth,
            "mv": mv,
            "reactive": reactive,
            "exposure": exposure,
            "jitter": jitter,
            "has_depth": flags[:, 0],
            "mv_quality": flags[:, 1],
        }
        out, st = self.net(
            f,
            {
                "out": prev_out,
                "hidden": prev_hidden,
                "depth": prev_depth,
                "first": False,
            },
        )
        return out, st["hidden"], depth


class FGExport(nn.Module):
    """ONNX wrapper of the fused frame generator."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, i0, i1, mv1, depth1, cam_prior, tau, extrap, avail):
        f = {
            "i0": i0,
            "i1": i1,
            "mv1": mv1,
            "depth1": depth1,
            "cam_prior": cam_prior,
            "tau": tau,
            "extrap": extrap,
            "mv_avail": avail[:, 0],
            "cam_avail": avail[:, 1],
            "depth_avail": avail[:, 2],
        }
        return self.net(f)[0]

**`nss_metrics.py`**: losses, PSNR / SSIM / LPIPS / FLIP, RAFT teacher, temporal metrics, analytic baselines and the frame-pacing latency model.

In [ ]:
%%writefile /kaggle/working/code/nss_metrics.py
"""Losses, image / perceptual / temporal metrics, optical-flow teacher, analytic baselines and the frame-pacing
latency simulator shared by the supersampling and frame-generation notebooks."""

import math
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nss_models import backward_warp


# ---------------------------------------------------------------------------------------------------------------
# Losses
# ---------------------------------------------------------------------------------------------------------------
def charbonnier(pred, target, mask=None, eps=1e-3):
    err = torch.sqrt((pred - target) ** 2 + eps * eps)
    if mask is None:
        return err.mean()
    return (err * mask).sum() / (mask.sum() * pred.shape[1] + 1e-6)


def gradient_loss(pred, target):
    def g(x):
        return x[..., :, 2:] - x[..., :, :-2], x[..., 2:, :] - x[..., :-2, :]

    (px, py), (tx, ty) = g(pred), g(target)
    return (px - tx).abs().mean() + (py - ty).abs().mean()


def fft_loss(pred, target):
    """L1 distance of 2D spectrum magnitudes (keeps fine texture energy; Fourier-space loss)."""
    fp = torch.fft.rfft2(pred.float(), norm="ortho")
    ft = torch.fft.rfft2(target.float(), norm="ortho")
    return (fp.abs() - ft.abs()).abs().mean()


def census_loss(pred, target, patch=7):
    """Soft ternary census distance (robust to brightness changes; as used for frame interpolation)."""

    def census(img):
        grey = 0.299 * img[:, 0:1] + 0.587 * img[:, 1:2] + 0.114 * img[:, 2:3]
        eye = torch.eye(patch * patch, device=img.device, dtype=img.dtype).reshape(
            patch * patch, 1, patch, patch
        )
        diff = F.conv2d(grey, eye, padding=patch // 2) - grey
        return diff / torch.sqrt(0.81 + diff * diff)

    d = (census(pred.float()) - census(target.float())) ** 2
    dist = (d / (0.1 + d)).mean(dim=1, keepdim=True)
    b = patch // 2
    return dist[..., b:-b, b:-b].mean()


def temporal_loss(out_t, out_prev, gt_t, gt_prev, mv_hr, valid):
    """Match the temporal change of the output to that of the ground truth along exact motion (flicker and
    ghosting penalty): |(o_t - W o_{t-1}) - (g_t - W g_{t-1})| over pixels visible in both frames.
    """
    d_out = out_t - backward_warp(out_prev, mv_hr)
    d_gt = gt_t - backward_warp(gt_prev, mv_hr)
    return ((d_out - d_gt).abs() * valid).sum() / (valid.sum() * 3 + 1e-6)


# ---------------------------------------------------------------------------------------------------------------
# Image metrics
# ---------------------------------------------------------------------------------------------------------------
def rgb_to_y(x):
    return (
        16.0 + 65.481 * x[:, 0:1] + 128.553 * x[:, 1:2] + 24.966 * x[:, 2:3]
    ) / 255.0


def quantize(x):
    return (x.clamp(0, 1) * 255.0).round() / 255.0


def psnr(a, b, mask=None, border=0):
    if border:
        a, b = (
            a[..., border:-border, border:-border],
            b[..., border:-border, border:-border],
        )
        mask = mask[..., border:-border, border:-border] if mask is not None else None
    se = (a.float() - b.float()) ** 2
    if mask is None:
        mse = se.mean(dim=(1, 2, 3))
    else:
        m = mask.float().expand_as(se)
        mse = (se * m).sum(dim=(1, 2, 3)) / m.sum(dim=(1, 2, 3)).clamp_min(1)
    return 10.0 * torch.log10(1.0 / mse.clamp_min(1e-10))


def interpolation_error(a, b):
    """Middlebury interpolation error: RMS difference on the 0-255 scale."""
    return ((a.float() - b.float()) ** 2).mean(dim=(1, 2, 3)).sqrt() * 255.0


def ssim(a, b, border=0):
    if border:
        a, b = (
            a[..., border:-border, border:-border],
            b[..., border:-border, border:-border],
        )
    a, b = a.float(), b.float()
    c = a.shape[1]
    coords = torch.arange(11, dtype=torch.float32, device=a.device) - 5
    g = torch.exp(-(coords**2) / (2 * 1.5**2))
    g = g / g.sum()
    win = (g[:, None] * g[None, :]).expand(c, 1, 11, 11).contiguous()
    mu_a, mu_b = F.conv2d(a, win, groups=c), F.conv2d(b, win, groups=c)
    va = F.conv2d(a * a, win, groups=c) - mu_a**2
    vb = F.conv2d(b * b, win, groups=c) - mu_b**2
    cov = F.conv2d(a * b, win, groups=c) - mu_a * mu_b
    c1, c2 = 0.01**2, 0.03**2
    s = ((2 * mu_a * mu_b + c1) * (2 * cov + c2)) / (
        (mu_a**2 + mu_b**2 + c1) * (va + vb + c2)
    )
    return s.mean(dim=(1, 2, 3))


class LPIPSAlex(nn.Module):
    """LPIPS (AlexNet, v0.1) built from torchvision AlexNet weights and the linear calibration weights shipped
    with the lpips package; used when lpips.LPIPS itself cannot be constructed."""

    def __init__(self, lin_path):
        super().__init__()
        from torchvision.models import AlexNet_Weights, alexnet

        feats = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1).features
        self.slices = nn.ModuleList(
            [feats[0:2], feats[2:5], feats[5:8], feats[8:10], feats[10:12]]
        )
        chans = [64, 192, 384, 256, 256]
        self.lins = nn.ModuleList([nn.Conv2d(c, 1, 1, bias=False) for c in chans])
        state = torch.load(lin_path, map_location="cpu")
        for k, lin in enumerate(self.lins):
            lin.weight.data.copy_(state[f"lin{k}.model.1.weight"])
        self.register_buffer(
            "shift", torch.tensor([-0.030, -0.088, -0.188]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "scale", torch.tensor([0.458, 0.448, 0.450]).view(1, 3, 1, 1)
        )
        self.requires_grad_(False)

    def forward(self, a, b):
        xa, xb = (2 * a - 1 - self.shift) / self.scale, (
            2 * b - 1 - self.shift
        ) / self.scale
        total = 0
        for sl, lin in zip(self.slices, self.lins):
            xa, xb = sl(xa), sl(xb)
            na = xa / (xa.pow(2).sum(1, keepdim=True).sqrt() + 1e-10)
            nb = xb / (xb.pow(2).sum(1, keepdim=True).sqrt() + 1e-10)
            total = total + lin((na - nb) ** 2).mean(dim=(2, 3))
        return total.view(-1)


def load_lpips(device, log=print):
    """Callable lpips(a, b) for images in [0, 1] (N, 3, H, W) returning (N,), or None when unavailable."""
    try:
        import lpips

        try:
            net = (
                lpips.LPIPS(net="alex", verbose=False)
                .to(device)
                .eval()
                .requires_grad_(False)
            )
            return lambda a, b: net(a.float(), b.float(), normalize=True).view(-1)
        except Exception as exc:
            log(
                f"lpips.LPIPS failed ({exc}); building LPIPS-Alex from torchvision weights"
            )
            lin = os.path.join(
                os.path.dirname(lpips.__file__), "weights", "v0.1", "alex.pth"
            )
            net = LPIPSAlex(lin).to(device).eval()
            return lambda a, b: net(a.float(), b.float())
    except Exception as exc:
        log(f"LPIPS unavailable ({exc})")
        return None


def flip_error(ref, test):
    """Mean LDR-FLIP error (NVIDIA, BSD-3) of two (3, H, W) tensors in [0, 1], or None if unavailable."""
    try:
        import flip_evaluator as flip

        r = ref.permute(1, 2, 0).float().cpu().numpy().astype(np.float32)
        t = test.permute(1, 2, 0).float().cpu().numpy().astype(np.float32)
        return float(flip.evaluate(r, t, "LDR")[1])
    except Exception:
        return None


# ---------------------------------------------------------------------------------------------------------------
# Optical-flow teacher (torchvision RAFT, BSD-3)
# ---------------------------------------------------------------------------------------------------------------
def load_raft(device, log=print, large=False):
    try:
        from torchvision.models.optical_flow import (
            Raft_Large_Weights,
            Raft_Small_Weights,
            raft_large,
            raft_small,
        )

        net = (
            raft_large(weights=Raft_Large_Weights.DEFAULT)
            if large
            else raft_small(weights=Raft_Small_Weights.DEFAULT)
        )
        return net.to(device).eval().requires_grad_(False)
    except Exception as exc:
        log(
            f"RAFT unavailable ({exc}); estimated motion vectors and RAFT metrics are skipped"
        )
        return None


@torch.no_grad()
def raft_flow(net, a, b, iters=12):
    """Flow from a to b (for each pixel of a, its displacement in b), pixels; a, b in [0, 1] (N, 3, H, W)."""
    h, w = a.shape[-2:]
    ph, pw = (-h) % 8, (-w) % 8
    pa = F.pad(a.float() * 2 - 1, (0, pw, 0, ph), mode="replicate")
    pb = F.pad(b.float() * 2 - 1, (0, pw, 0, ph), mode="replicate")
    return net(pa, pb, num_flow_updates=iters)[-1][..., :h, :w]


@torch.no_grad()
def raft_mv_and_valid(net, cur, prev, tol=1.0):
    """Motion vector cur -> prev with a forward-backward consistency validity mask."""
    fwd = raft_flow(net, cur, prev)
    bwd = raft_flow(net, prev, cur)
    back = backward_warp(bwd, fwd)
    err = (fwd + back).norm(dim=1, keepdim=True)
    valid = (err < tol + 0.05 * fwd.norm(dim=1, keepdim=True)).float()
    return fwd, valid


# ---------------------------------------------------------------------------------------------------------------
# Temporal metrics
# ---------------------------------------------------------------------------------------------------------------
def warping_error(out_t, out_prev, mv, valid):
    """Mean absolute difference between a frame and the previous frame warped by ground-truth motion."""
    diff = (out_t - backward_warp(out_prev, mv)).abs().mean(1, keepdim=True)
    return (diff * valid).sum(dim=(1, 2, 3)) / valid.sum(dim=(1, 2, 3)).clamp_min(1)


def temporal_change_error(out_t, out_prev, gt_t, gt_prev, mv, valid):
    d = (out_t - backward_warp(out_prev, mv)) - (gt_t - backward_warp(gt_prev, mv))
    diff = d.abs().mean(1, keepdim=True)
    return (diff * valid).sum(dim=(1, 2, 3)) / valid.sum(dim=(1, 2, 3)).clamp_min(1)


@torch.no_grad()
def tof(raft, out_t, out_prev, gt_t, gt_prev):
    """tOF (TecoGAN): mean |flow(out) - flow(gt)| between consecutive frames, pixels."""
    if raft is None:
        return None
    return (
        (raft_flow(raft, out_t, out_prev) - raft_flow(raft, gt_t, gt_prev))
        .norm(dim=1)
        .mean(dim=(1, 2))
    )


@torch.no_grad()
def tlp(lpips_fn, out_t, out_prev, gt_t, gt_prev):
    """tLP (TecoGAN): |LPIPS(out_t, out_prev) - LPIPS(gt_t, gt_prev)|."""
    if lpips_fn is None:
        return None
    return (lpips_fn(out_t, out_prev) - lpips_fn(gt_t, gt_prev)).abs()


# ---------------------------------------------------------------------------------------------------------------
# Analytic baselines
# ---------------------------------------------------------------------------------------------------------------
def rgb_to_ycocg(x):
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    return torch.cat(
        [
            0.25 * r + 0.5 * g + 0.25 * b,
            0.5 * r - 0.5 * b,
            -0.25 * r + 0.5 * g - 0.25 * b,
        ],
        1,
    )


def ycocg_to_rgb(x):
    y, co, cg = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    return torch.cat([y + co - cg, y + cg, y - co - cg], 1)


class AnalyticTAAU:
    """Hand-tuned temporal anti-aliasing upscaler: reproject history with upsampled motion vectors, clamp it to the
    3x3 YCoCg neighbourhood of the current jitter-aware upsample, reset on invalid history, blend 10 percent of the
    new frame. A stand-in for analytic TAAU / FSR 2-style accumulation without its proprietary heuristics.
    """

    def __init__(self, base_upsample, scale, blend=0.1):
        self.base_upsample, self.scale, self.blend = base_upsample, scale, blend

    def __call__(self, color, jitter, mv, state):
        base = self.base_upsample(color, jitter)
        if state is None:
            return base, base
        hist = backward_warp(
            state,
            F.interpolate(
                mv, scale_factor=self.scale, mode="bilinear", align_corners=False
            )
            * self.scale,
        )
        yb = rgb_to_ycocg(base)
        lo = -F.max_pool2d(-yb, 3, 1, 1)
        hi = F.max_pool2d(yb, 3, 1, 1)
        hc = ycocg_to_rgb(torch.max(torch.min(rgb_to_ycocg(hist), hi), lo))
        out = hc + self.blend * (base - hc)
        return out, out


def fg_analytic(i0, i1, f0, f1):
    """Motion-vector reprojection baseline: warp both frames along linear-motion flows and keep, per pixel, the
    warp that agrees better with the other (occlusion heuristic); falls back to I1 where both disagree strongly.
    """
    w0, w1 = backward_warp(i0, f0), backward_warp(i1, f1)
    agree = (w0 - w1).abs().mean(1, keepdim=True)
    m = torch.sigmoid((0.1 - agree) * 40.0)
    return m * 0.5 * (w0 + w1) + (1 - m) * w1


# ---------------------------------------------------------------------------------------------------------------
# Frame-pacing latency simulator
# ---------------------------------------------------------------------------------------------------------------
def simulate_latency(
    mode,
    render_ms,
    sr_ms=0.0,
    fg_ms=0.0,
    cpu_ms=4.0,
    frames=2000,
    cv=0.08,
    present_ms=1.0,
    seed=0,
):
    """Discrete-event model of a GPU-bound game loop with a just-in-time low-latency scheduler (CPU work starts so
    that the GPU never queues frames) on a variable-refresh display.

    modes: 'native' (no generation), 'interp' (generated frame between real frames k-1 and k; real frame k is held
    until after the generated one), 'extrap' (generated frame after k from k-1 and k; nothing held), and
    'extrap_late' (extrapolation whose camera is re-sampled from input just before generation).

    Returns per-mode statistics: displayed frame rate, game input latency (input sample -> first photons of a
    real frame reflecting it), camera latency (input sample -> first displayed frame, real or generated, whose
    camera reflects it) and content age (time since the newest real input in each displayed frame).
    """
    rng = np.random.default_rng(seed)
    t = 0.0
    real_lat, cam_lat, shown, ages = [], [], [], []
    for _ in range(frames):
        r = render_ms * float(rng.lognormal(0.0, cv))
        gpu = r + sr_ms + (fg_ms if mode != "native" else 0.0)
        period = max(cpu_ms, gpu)
        t_input = t
        ready = t_input + cpu_ms + r + sr_ms
        if mode == "native":
            p_real = ready + present_ms
            shown.append(p_real)
            real_lat.append(p_real - t_input)
            cam_lat.append(p_real - t_input)
            ages.append(p_real - t_input)
        elif mode == "interp":
            p_gen = ready + fg_ms + present_ms
            p_real = p_gen + period / 2
            shown += [p_gen, p_real]
            real_lat.append(p_real - t_input)
            cam_lat.append(
                p_gen - t_input
            )  # the generated frame already moves toward frame k's camera (half way)
            ages += [p_gen - t_input, p_real - t_input]
        else:
            p_real = ready + present_ms
            p_gen = p_real + period / 2
            shown += [p_real, p_gen]
            real_lat.append(p_real - t_input)
            if mode == "extrap_late":
                t_cam = (
                    p_gen - fg_ms - present_ms
                )  # camera re-sampled from input just before generation
                cam_lat += [p_real - t_input, p_gen - t_cam]
            else:
                cam_lat += [p_real - t_input, p_gen - t_input]
            ages += [p_real - t_input, p_gen - t_input]
        t += period
    shown = np.array(shown)
    gaps = np.diff(shown[len(shown) // 10 :])
    stats = lambda x: (float(np.mean(x)), float(np.percentile(x, 95)))  # noqa: E731
    rl, cl, ag = stats(real_lat), stats(cam_lat), stats(ages)
    return {
        "mode": mode,
        "render_ms": render_ms,
        "sr_ms": sr_ms,
        "fg_ms": fg_ms,
        "displayed_fps": float(1000.0 / gaps.mean()),
        "frame_gap_cv": float(gaps.std() / gaps.mean()),
        "game_latency_ms": rl[0],
        "game_latency_p95_ms": rl[1],
        "camera_latency_ms": cl[0],
        "camera_latency_p95_ms": cl[1],
        "content_age_ms": ag[0],
    }


def estimate_ms(gmac, sustained_tflops):
    """Arithmetic latency estimate: 2 FLOP per MAC at a sustained throughput (FP16 TFLOPS)."""
    return (
        2.0 * gmac / (sustained_tflops * 1e3) * 1e3
        if sustained_tflops > 0
        else math.inf
    )

**`nss_train.py`**: DDP harness: batch streaming from memory maps, wall-clock schedule, EMA, checkpoints and the PROGRESS / ROUND / DONE protocol.

In [ ]:
%%writefile /kaggle/working/code/nss_train.py
"""Shared DDP training harness: process group setup, random-batch streaming from memory-mapped caches,
wall-clock learning-rate schedule, EMA, checkpointing and the machine-readable progress protocol
(PROGRESS / ROUND / DONE JSON lines on rank 0, parsed by the notebooks)."""

import copy
import json
import math
import os
import time
from datetime import timedelta

import numpy as np
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset


def setup_ddp():
    if "RANK" in os.environ:
        dist.init_process_group("nccl", timeout=timedelta(minutes=30))
        rank, world, local = (
            dist.get_rank(),
            dist.get_world_size(),
            int(os.environ["LOCAL_RANK"]),
        )
    else:
        rank, world, local = 0, 1, int(os.environ.get("CUDA_DEVICE", 0))
    torch.cuda.set_device(local)
    torch.backends.cudnn.benchmark = True
    return rank, world, torch.device("cuda", local)


def is_dist():
    return dist.is_available() and dist.is_initialized()


class BatchStream(Dataset):
    """Each item is a whole random batch: for every cache {name: {"arrays": {key: npy path}, "count": n}} and
    per-cache batch size, sorted random indices are read from the memory maps (opened lazily per worker).
    """

    def __init__(self, caches, sizes, seed, steps=10**9):
        self.caches, self.sizes, self.seed, self.steps, self.maps = (
            caches,
            sizes,
            seed,
            steps,
            None,
        )

    def __len__(self):
        return self.steps

    def __getitem__(self, i):
        if self.maps is None:
            self.maps = {
                name: {k: np.load(p, mmap_mode="r") for k, p in c["arrays"].items()}
                for name, c in self.caches.items()
            }
        rng = np.random.default_rng(self.seed + i * 7919)
        out = {}
        for name, c in self.caches.items():
            b = self.sizes.get(name, 0)
            if b <= 0 or c["count"] <= 0:
                continue
            idx = np.sort(rng.choice(c["count"], size=b, replace=c["count"] < b))
            for k, arr in self.maps[name].items():
                out[f"{name}/{k}"] = torch.from_numpy(np.ascontiguousarray(arr[idx]))
        return out


def make_loader(caches, sizes, seed, workers):
    return DataLoader(
        BatchStream(caches, sizes, seed),
        batch_size=None,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        persistent_workers=workers > 0,
        prefetch_factor=4 if workers > 0 else None,
    )


def lr_at(progress, cfg):
    if progress < cfg["warmup_frac"]:
        return cfg["lr"] * max(progress / cfg["warmup_frac"], 0.01)
    t = min(1.0, (progress - cfg["warmup_frac"]) / (1.0 - cfg["warmup_frac"]))
    return cfg["min_lr"] + 0.5 * (cfg["lr"] - cfg["min_lr"]) * (
        1.0 + math.cos(math.pi * t)
    )


class EMA:
    def __init__(self, model, decay):
        self.model, self.decay, self.step = (
            copy.deepcopy(model).eval().requires_grad_(False),
            decay,
            0,
        )

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for e, p in zip(self.model.parameters(), model.parameters()):
            e.lerp_(p.detach(), 1.0 - d)


def emit(kind, payload):
    print(f"{kind} " + json.dumps(payload, default=float), flush=True)


def train_loop(cfg, model, step_fn, val_fn, rank, world, device, loader):
    """Generic loop. step_fn(batch, step) -> (loss tensor, logs dict of floats); val_fn(ema_model) -> metrics dict
    (rank 0 only). The schedule follows the fraction of cfg['train_seconds'] already used, agreed across ranks.
    """
    from torch.nn.parallel import DistributedDataParallel as DDP

    ddp = (
        DDP(
            model,
            device_ids=[device.index],
            broadcast_buffers=False,
            find_unused_parameters=cfg.get("find_unused", False),
        )
        if world > 1
        else model
    )
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        betas=(0.9, 0.99),
        weight_decay=cfg["weight_decay"],
    )
    scaler = torch.amp.GradScaler("cuda")
    ema = EMA(model, cfg["ema_decay"]) if rank == 0 else None
    start = last_val = last_prog = time.time()
    sync = torch.zeros(3 + world, device=device)
    acc, n_acc, seen, round_start = {}, 0, 0, time.time()
    history, best, step, done = [], -1e9, 0, False
    it = iter(loader)
    while not done:
        if step % cfg["sync_every"] == 0:
            sync.zero_()
            if rank == 0:
                el = time.time() - start
                sync[0] = el
                sync[1] = float(
                    el - (last_val - start) >= cfg["val_interval_s"]
                    or el >= cfg["train_seconds"]
                )
            sync[3 + rank] = torch.cuda.max_memory_allocated(device) / 2**30
            if world > 1:
                dist.all_reduce(sync)
            el = sync[0].item()
            progress = min(1.0, el / cfg["train_seconds"])
            lr = lr_at(progress, cfg)
            for g in opt.param_groups:
                g["lr"] = lr
            mem = [round(v, 2) for v in sync[3:].tolist()]
            if sync[1].item() > 0.5 and n_acc > 0:
                if rank == 0:
                    row = {
                        "round": len(history),
                        "elapsed_min": round(el / 60, 2),
                        "step": step,
                        "lr": lr,
                        "samples_per_s": seen
                        * world
                        / max(time.time() - round_start, 1e-6),
                        "mem_gb": mem,
                        **{f"train_{k}": float(v) / n_acc for k, v in acc.items()},
                    }
                    row.update({f"val_{k}": v for k, v in val_fn(ema.model).items()})
                    history.append(row)
                    torch.save(
                        {
                            "model": model.state_dict(),
                            "ema": ema.model.state_dict(),
                            "step": step,
                            "cfg": cfg,
                        },
                        os.path.join(cfg["weights_dir"], f"{cfg['name']}_last.pt"),
                    )
                    score = row.get(f"val_{cfg['select_metric']}", -1e9)
                    if score > best:
                        best = score
                        torch.save(
                            {
                                "ema": ema.model.state_dict(),
                                "round": row["round"],
                                "score": best,
                                "cfg": cfg,
                            },
                            os.path.join(cfg["weights_dir"], f"{cfg['name']}_best.pt"),
                        )
                    row["best"] = best
                    emit("ROUND", row)
                    last_val = time.time()
                if world > 1:
                    dist.barrier()
                acc, n_acc, seen, round_start = {}, 0, 0, time.time()
            if progress >= 1.0:
                break
            if rank == 0 and time.time() - last_prog >= cfg["progress_interval_s"]:
                emit(
                    "PROGRESS",
                    {
                        "progress": progress,
                        "step": step,
                        "elapsed_min": round(el / 60, 2),
                        "lr": lr,
                        "loss": float(acc.get("loss", 0.0)) / max(n_acc, 1),
                        "samples_per_s": seen
                        * world
                        / max(time.time() - round_start, 1e-6),
                        "mem_gb": mem,
                    },
                )
                last_prog = time.time()
        batch = {k: v.to(device, non_blocking=True) for k, v in next(it).items()}
        loss, logs, n_samples = step_fn(ddp, batch, step)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        scaler.step(opt)
        scaler.update()
        if ema is not None:
            ema.update(model)
        for k, v in logs.items():  # kept on the GPU; converted when a line is printed
            acc[k] = acc.get(k, 0.0) + (
                v.detach().float() if torch.is_tensor(v) else float(v)
            )
        n_acc += 1
        seen += n_samples
        step += 1
    if rank == 0:
        with open(
            os.path.join(cfg["metrics_dir"], f"{cfg['name']}_history.json"), "w"
        ) as f:
            json.dump(history, f, indent=2, default=float)
        emit(
            "DONE",
            {
                "steps": step,
                "train_seconds": round(time.time() - start, 1),
                "best": best,
            },
        )
    if world > 1:
        dist.barrier()

**`nss_eval.py`**: test clips, input construction, method runners and per-frame metric rows.

In [ ]:
%%writefile /kaggle/working/code/nss_eval.py
"""Evaluation helpers for both training notebooks: full-resolution test clips, input construction, method
runners (networks and analytic baselines) and per-frame metric rows."""

import numpy as np
import torch
import torch.nn.functional as F

import nss_data as D
import nss_metrics as M
import nss_synth as S
from nss_models import backward_warp


# ---------------------------------------------------------------------------------------------------------------
# Clips
# ---------------------------------------------------------------------------------------------------------------
def load_clip(data_root, item, start, T, crop_hw, max_side=None):
    """Full-resolution test clip: frames uint8 (T, H, W, 3) center-cropped to crop_hw (after an optional
    down-scale so the shorter side is at most max_side), plus exact motion vectors (T, H, W, 2) and validity
    (T, H, W) in network time when the source has exact flow, depth (T, H, W) when available, and native
    low-resolution frames when the source provides them. Exact-forward sources are played backwards.
    """
    res = D.Resolver(data_root)
    idx = list(range(start, start + T))
    frames = D.load_rgb_many(res, [item["frames"][i] for i in idx])
    h, w = frames[0].shape[:2]
    r = min(1.0, max_side / min(h, w)) if max_side else 1.0
    W, H = int(round(w * r)), int(round(h * r))
    ch, cw = crop_hw
    if H < ch or W < cw:
        return None
    y0, x0 = (H - ch) // 2, (W - cw) // 2
    sl = (slice(y0, y0 + ch), slice(x0, x0 + cw))
    out = {
        "rgb": np.stack([D._resize(f, (W, H), "image")[sl] for f in frames]),
        "exact": False,
    }
    if item.get("mv") == "exact_forward":
        flows = [
            D._resize(D.load_flow(res, item["flows"][i]), (W, H), "flow")[sl]
            for i in idx
        ]
        valid = [
            D._resize(
                D.load_valid(res, item["valid"][i], item.get("valid_kind", "valid")),
                (W, H),
                "nearest",
            )[sl]
            for i in idx
        ]
        mv, va = np.stack(flows)[::-1].copy(), np.stack(valid)[::-1].copy()
        mv[0], va[0] = 0, 0
        out.update(rgb=out["rgb"][::-1].copy(), mv=mv, valid=va, exact=True)
    if item.get("depth"):
        d = np.stack(
            [
                D.normalise_depth(
                    D._resize(D.load_depth(res, item["depth"][i]), (W, H), "depth")[sl]
                )
                for i in idx
            ]
        )
        out["depth"] = d[::-1].copy() if out["exact"] else d
    if item.get("native_lr"):
        lr = D.load_rgb_many(res, [item["native_lr"][i] for i in idx])
        f = lr[0].shape[0] / h
        lsl = (
            slice(int(y0 * f), int(y0 * f) + int(ch * f)),
            slice(int(x0 * f), int(x0 * f) + int(cw * f)),
        )
        out["native_lr"] = np.stack(
            [
                D._resize(x, (int(round(w * r * f)), int(round(h * r * f))), "image")[
                    lsl
                ]
                for x in lr
            ]
        )
    return out


def sr_inputs(clip, scale, device, raft=None, jitter=True, native=False, phases=16):
    """Batch (N = 1) for the TSR network from a clip: jittered single-sample low-resolution frames (or the
    native low-resolution render), motion vectors (exact, or RAFT on consecutive low-resolution frames),
    validity, depth, full-resolution targets."""
    hr = (
        torch.from_numpy(clip["rgb"]).to(device).permute(0, 3, 1, 2).float()[None]
        / 255.0
    )  # 1, T, 3, H, W
    _, T, _, H, W = hr.shape
    h, w = H // scale, W // scale
    seq = (
        S.jitter_sequence(T, phases).to(device)
        if jitter and not native
        else torch.zeros(T, 2, device=device)
    )
    lrs, mv_lr, va_lr, d_lr, mv_hr, va_hr = [], [], [], [], [], []
    for t in range(T):
        if native:
            lr = (
                torch.from_numpy(clip["native_lr"][t])
                .to(device)
                .permute(2, 0, 1)
                .float()[None]
                / 255.0
            )
            lr = (
                F.interpolate(
                    lr,
                    size=(h, w),
                    mode="bilinear",
                    antialias=True,
                    align_corners=False,
                )
                if lr.shape[-2:] != (h, w)
                else lr
            )
        else:
            pts = S.pixel_grid(1, h, w, device, seq[t : t + 1])
            # non-square frames: pixel_grid normalises x and y independently, which matches grid_sample
            lr = F.grid_sample(
                hr[:, t],
                pts,
                mode="bilinear",
                padding_mode="border",
                align_corners=False,
            )
        lrs.append(lr)
    lr_all = torch.stack(lrs, 1)
    quality = 0.0
    for t in range(T):
        if clip.get("exact"):
            m = (
                torch.from_numpy(clip["mv"][t])
                .to(device)
                .permute(2, 0, 1)
                .float()[None]
            )
            v = torch.from_numpy(clip["valid"][t]).to(device).float()[None, None]
            pts = S.pixel_grid(1, h, w, device, seq[t : t + 1])
            mv_lr.append(
                F.grid_sample(
                    m, pts, mode="bilinear", padding_mode="border", align_corners=False
                )
                / scale
            )
            va_lr.append(
                F.grid_sample(
                    v, pts, mode="nearest", padding_mode="border", align_corners=False
                )
            )
            mv_hr.append(m)
            va_hr.append(v)
            quality = 1.0
        elif raft is not None and t > 0:
            m, v = M.raft_mv_and_valid(raft, lr_all[:, t], lr_all[:, t - 1])
            mv_lr.append(m)
            va_lr.append(v)
            mv_hr.append(
                F.interpolate(
                    m, scale_factor=scale, mode="bilinear", align_corners=False
                )
                * scale
            )
            va_hr.append(F.interpolate(v, scale_factor=scale, mode="nearest"))
            quality = 0.5
        else:
            mv_lr.append(torch.zeros(1, 2, h, w, device=device))
            va_lr.append(torch.zeros(1, 1, h, w, device=device))
            mv_hr.append(torch.zeros(1, 2, H, W, device=device))
            va_hr.append(torch.zeros(1, 1, H, W, device=device))
        if "depth" in clip:
            d = torch.from_numpy(clip["depth"][t]).to(device).float()[None, None]
            d_lr.append(
                F.interpolate(d, size=(h, w), mode="bilinear", align_corners=False)
            )
        else:
            d_lr.append(torch.zeros(1, 1, h, w, device=device))
    st = lambda xs: torch.stack(xs, 1)  # noqa: E731
    return {
        "lr": lr_all,
        "hr": hr,
        "mv_lr": st(mv_lr),
        "valid_lr": st(va_lr),
        "mv_hr": st(mv_hr),
        "valid_hr": st(va_hr),
        "depth_lr": st(d_lr),
        "reactive_lr": torch.zeros(1, T, 1, h, w, device=device),
        "jitter": seq[None],
        "exposure": torch.zeros(1, T, device=device),
        "has_depth": torch.tensor([1.0 if "depth" in clip else 0.0], device=device),
        "mv_quality": torch.tensor([quality], device=device),
        "speed_class": torch.full((1,), -1, device=device),
    }


def synthetic_suite(stills, scale, T, P, motion_class, seed, device, text_bank=None):
    """Synthetic game-like test sequences with every sample in one motion class (static / slow / medium / fast).
    The camera path is only kept inside the texture for 6 frames so long sequences keep their speed class.
    """
    prof = dict(S.MOTION_PROFILE)
    prof["probs"] = [
        1.0 if i == motion_class else 0.0 for i in range(len(prof["probs"]))
    ]
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    return S.sr_batch(
        stills, scale, T, P, g, profile=prof, text_bank=text_bank, room_frames=min(T, 6)
    )


# ---------------------------------------------------------------------------------------------------------------
# Runners
# ---------------------------------------------------------------------------------------------------------------
def frame_inputs(b, t):
    return {
        "color": b["lr"][:, t],
        "depth": b["depth_lr"][:, t],
        "mv": b["mv_lr"][:, t],
        "reactive": b["reactive_lr"][:, t],
        "exposure": b["exposure"][:, t],
        "jitter": b["jitter"][:, t],
        "has_depth": b["has_depth"],
        "mv_quality": b["mv_quality"],
    }


@torch.no_grad()
def run_tsr(
    net, b, use_history=True, ablate_gbuffers=False, keep_alpha=False, amp=True
):
    outs, alphas, state = [], [], None
    for t in range(b["lr"].shape[1]):
        f = frame_inputs(b, t)
        if ablate_gbuffers:
            f.update(
                depth=torch.zeros_like(f["depth"]),
                mv=torch.zeros_like(f["mv"]),
                has_depth=torch.zeros_like(f["has_depth"]),
                mv_quality=torch.zeros_like(f["mv_quality"]),
            )
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            out, state = net(f, state, use_history=use_history)
        outs.append(out.float().clamp(0, 1))
        if keep_alpha:
            alphas.append(state["alpha"].float())
    return (
        (torch.stack(outs, 1), torch.stack(alphas, 1))
        if keep_alpha
        else torch.stack(outs, 1)
    )


@torch.no_grad()
def run_bilinear(net, b):
    return torch.stack(
        [
            net.base_upsample(b["lr"][:, t], b["jitter"][:, t]).clamp(0, 1)
            for t in range(b["lr"].shape[1])
        ],
        1,
    )


@torch.no_grad()
def run_bicubic(b, scale):
    n, T, _, h, w = b["lr"].shape
    x = F.interpolate(
        b["lr"].flatten(0, 1), scale_factor=scale, mode="bicubic", align_corners=False
    ).clamp(0, 1)
    return x.view(n, T, 3, h * scale, w * scale)


@torch.no_grad()
def run_taau(net, b, scale):
    taau = M.AnalyticTAAU(net.base_upsample, scale)
    outs, state = [], None
    for t in range(b["lr"].shape[1]):
        out, state = taau(b["lr"][:, t], b["jitter"][:, t], b["mv_lr"][:, t], state)
        outs.append(out.clamp(0, 1))
    return torch.stack(outs, 1)


def hf_mask(hr, frac=0.15):
    """Fine-detail mask: pixels in the top frac of ground-truth gradient magnitude."""
    y = M.rgb_to_y(hr)
    gx = F.pad(y[..., :, 2:] - y[..., :, :-2], (1, 1, 0, 0))
    gy = F.pad(y[..., 2:, :] - y[..., :-2, :], (0, 0, 1, 1))
    mag = (gx**2 + gy**2).sqrt().flatten(1)
    thr = torch.quantile(mag.float(), 1 - frac, dim=1)
    return (mag >= thr[:, None]).view_as(y).float()


@torch.no_grad()
def sr_frame_metrics(
    out, b, scale, lpips_fn=None, raft=None, lpips_every=4, skip_first=1
):
    """Per (sample, frame) metric rows: PSNR / SSIM on Y, PSNR in disoccluded, fine-detail and particle regions,
    LPIPS (every lpips_every frames), warping and temporal-change errors along ground-truth motion when exact,
    tOF with RAFT."""
    hr = b["hr"]
    n, T = out.shape[:2]
    rows = []
    exact = b["mv_quality"] >= 1.0
    for t in range(skip_first, T):
        o, g = out[:, t], hr[:, t]
        yo, yg = M.rgb_to_y(o), M.rgb_to_y(g)
        p = M.psnr(yo, yg, border=scale)
        sm = M.ssim(yo, yg, border=scale)
        disocc = (1 - b["valid_hr"][:, t]) if t > 0 else torch.zeros_like(yo)
        hfm = hf_mask(g)
        rea = (
            F.interpolate(b["reactive_lr"][:, t], scale_factor=scale, mode="nearest")
            > 0.1
        )
        p_dis = M.psnr(yo, yg, mask=disocc) if disocc.sum() > 50 else None
        p_hf = M.psnr(yo, yg, mask=hfm)
        p_rea = M.psnr(yo, yg, mask=rea.float()) if rea.sum() > 50 else None
        lp = lpips_fn(o, g) if (lpips_fn is not None and t % lpips_every == 0) else None
        mv_px = (
            b["mv_hr"][:, t].norm(dim=1).mean(dim=(1, 2))
        )  # actual motion magnitude, output pixels per frame
        we = tce = tof = None
        if t > 0:
            we = M.warping_error(
                o, out[:, t - 1], b["mv_hr"][:, t], b["valid_hr"][:, t]
            )
            tce = M.temporal_change_error(
                o, out[:, t - 1], g, hr[:, t - 1], b["mv_hr"][:, t], b["valid_hr"][:, t]
            )
            if raft is not None and t % lpips_every == 0:
                tof = M.tof(raft, o, out[:, t - 1], g, hr[:, t - 1])
        for i in range(n):
            rows.append(
                {
                    "sample": i,
                    "frame": t,
                    "psnr_y": p[i].item(),
                    "ssim_y": sm[i].item(),
                    "psnr_disocc": None if p_dis is None else p_dis[i].item(),
                    "psnr_detail": p_hf[i].item(),
                    "psnr_particles": None if p_rea is None else p_rea[i].item(),
                    "lpips": None if lp is None else lp[i].item(),
                    "warp_err": None if (we is None or not exact[i]) else we[i].item(),
                    "temporal_err": (
                        None
                        if (tce is None or b["mv_quality"][i] <= 0)
                        else tce[i].item()
                    ),
                    "tof": None if tof is None else tof[i].item(),
                    "mv_px": mv_px[i].item(),
                    "speed_class": int(b["speed_class"][i].item()),
                }
            )
    return rows


# ---------------------------------------------------------------------------------------------------------------
# Frame generation
# ---------------------------------------------------------------------------------------------------------------
def fg_suite(
    stills,
    P,
    seed,
    device,
    tiles=None,
    extrap=False,
    motion_class=None,
    p_ui=0.0,
    p_cut=0.0,
    p_menu=0.0,
    p_flip=0.0,
):
    prof = dict(S.MOTION_PROFILE)
    if motion_class is not None:
        prof["probs"] = [
            1.0 if i == motion_class else 0.0 for i in range(len(prof["probs"]))
        ]
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    b = S.fg_batch(
        stills,
        P,
        g,
        tiles=tiles,
        p_extrap=1.0 if extrap else 0.0,
        p_ui=p_ui,
        p_cut=p_cut,
        p_menu=p_menu,
        p_flip=p_flip,
        profile=prof,
    )
    n = b["i0"].shape[0]
    b["mv_avail"] = torch.ones(n, device=device)
    b["depth_avail"] = torch.ones(n, device=device)
    b["cam_avail"] = b["extrap"].clone()
    return b


@torch.no_grad()
def run_fg(net, b, hudless=False):
    f = dict(b)
    if hudless and "hudless_i0" in b:
        f["i0"], f["i1"] = b["hudless_i0"], b["hudless_i1"]
    with torch.autocast("cuda", dtype=torch.float16):
        out, aux = net(f)
    out = out.float().clamp(0, 1)
    if (
        hudless and "ui_rgb" in b
    ):  # engine path: generate the scene without UI, composite the latest UI on top
        out = out * (1 - b["ui_alpha"]) + b["ui_rgb"]
    return out, aux


def fg_baselines(b):
    """Frame repeat (latest real frame), plain average, and motion-vector reprojection."""
    n = b["i0"].shape[0]
    tau = b["tau"].view(n, 1, 1, 1)
    m1 = b["mv1"] * b["mv_avail"].view(n, 1, 1, 1)
    return {
        "repeat": b["i1"],
        "average": 0.5 * (b["i0"] + b["i1"]),
        "mv_reprojection": M.fg_analytic(
            b["i0"], b["i1"], tau * m1, (tau - 1) * m1
        ).clamp(0, 1),
    }


@torch.no_grad()
def fg_metrics(out, b, lpips_fn=None):
    tgt = b["target"]
    n = tgt.shape[0]
    p = M.psnr(out, tgt)
    sm = M.ssim(out, tgt)
    ie = M.interpolation_error(out, tgt)
    lp = lpips_fn(out, tgt) if lpips_fn is not None else None
    ui = (b["ui_mask"] > 0.5).float()
    has_ui = ui.flatten(1).amax(1) > 0
    p_ui = M.psnr(out, tgt, mask=ui)
    occl = (1 - b["valid_t1"]) if "valid_t1" in b else torch.zeros_like(ui)
    has_occ = occl.flatten(1).sum(1) > 50
    p_occ = M.psnr(out, tgt, mask=occl)
    # UI ghosting: fraction of UI pixels whose error exceeds 10 percent (half-interpolated text / HUD)
    ghost = (((out - tgt).abs().amax(1, keepdim=True) > 0.1).float() * ui).flatten(
        1
    ).sum(1) / ui.flatten(1).sum(1).clamp_min(1)
    mv_px = (
        b["mv1"].norm(dim=1).mean(dim=(1, 2))
    )  # motion between the real frames, pixels
    rows = []
    for i in range(n):
        rows.append(
            {
                "psnr": p[i].item(),
                "ssim": sm[i].item(),
                "ie": ie[i].item(),
                "lpips": None if lp is None else lp[i].item(),
                "psnr_ui": p_ui[i].item() if has_ui[i] else None,
                "ui_ghost_rate": ghost[i].item() if has_ui[i] else None,
                "psnr_occluded": p_occ[i].item() if has_occ[i] else None,
                "mv_px": mv_px[i].item(),
                "extrap": bool(b["extrap"][i] > 0.5),
                "speed_class": int(b["speed_class"][i].item()),
                "cut": bool(b["cut"][i] > 0.5),
                "menu": bool(b["menu"][i] > 0.5),
                "flip": bool(b["flip"][i] > 0.5),
            }
        )
    return rows


@torch.no_grad()
def cut_score(i0, i1, mv1=None):
    """Scene-cut score: luma-histogram distance plus motion-compensated difference (higher = more likely a cut)."""
    y0, y1 = M.rgb_to_y(i0), M.rgb_to_y(i1)
    h0 = torch.stack([torch.histc(y, bins=32, min=0, max=1) for y in y0.flatten(1)])
    h1 = torch.stack([torch.histc(y, bins=32, min=0, max=1) for y in y1.flatten(1)])
    h0, h1 = h0 / h0.sum(1, keepdim=True), h1 / h1.sum(1, keepdim=True)
    hist = 0.5 * (h0 - h1).abs().sum(1)
    w = backward_warp(i0, mv1) if mv1 is not None else i0
    diff = (i1 - w).abs().mean(dim=(1, 2, 3))
    return hist + 2.0 * diff

**`train_tsr.py`**: DDP training entry point of the temporal super-resolution network.

In [ ]:
%%writefile /kaggle/working/code/train_tsr.py
"""DDP training of the temporal super-resolution network (one process per GPU, launched by the notebook with
torch.distributed.run; ablations run as single-GPU processes, one per GPU in parallel).

Each step mixes synthetic layered game-like sequences rendered on the GPU from cached stills (exact motion,
disocclusion, thin geometry, particles) with cached real sequences (exact motion from TartanAir / Sintel,
teacher-estimated motion from video). Loss per frame: Charbonnier + gradient + Fourier-magnitude terms,
LPIPS on the last frames, a temporal-change term along exact motion, and optional distillation toward a frozen
teacher network unrolled on the same inputs."""

import argparse
import json
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from nss_metrics import (
    charbonnier,
    fft_loss,
    gradient_loss,
    load_lpips,
    psnr,
    rgb_to_y,
    temporal_change_error,
    temporal_loss,
)  # noqa: E402
from nss_models import build_tsr  # noqa: E402
from nss_synth import make_text_bank, sr_batch, sr_batch_from_frames  # noqa: E402
from nss_train import emit, make_loader, setup_ddp, train_loop  # noqa: E402


def frame_inputs(b, t, ablate_gbuffers=False):
    n = b["lr"].shape[0]
    f = {
        "color": b["lr"][:, t],
        "depth": b["depth_lr"][:, t],
        "mv": b["mv_lr"][:, t],
        "reactive": b["reactive_lr"][:, t],
        "exposure": b["exposure"][:, t],
        "jitter": b["jitter"][:, t],
        "has_depth": b["has_depth"],
        "mv_quality": b["mv_quality"],
    }
    if (
        ablate_gbuffers
    ):  # colour-only input: no depth, no motion vectors (history warped with zero motion)
        f["depth"], f["mv"] = torch.zeros_like(f["depth"]), torch.zeros_like(f["mv"])
        f["has_depth"], f["mv_quality"] = torch.zeros(
            n, device=f["color"].device
        ), torch.zeros(n, device=f["color"].device)
    return f


def unroll(model, b, use_history=True, ablate_gbuffers=False):
    outs, state = [], None
    for t in range(b["lr"].shape[1]):
        out, state = model(
            frame_inputs(b, t, ablate_gbuffers), state, use_history=use_history
        )
        outs.append(out)
    return torch.stack(outs, 1)


class Sequence(torch.nn.Module):
    """Wraps the recurrent unroll in one module call, so DDP sees one forward per backward."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, b, use_history=True, ablate_gbuffers=False):
        return unroll(self.net, b, use_history, ablate_gbuffers)


def bicubic_inputs(b, scale):
    """Ablation: classic image-SR degradation (anti-aliased bicubic, no jitter) instead of render-like sampling."""
    n, T, _, H, W = b["hr"].shape
    hr = b["hr"].flatten(0, 1)
    lr = F.interpolate(
        hr,
        size=(H // scale, W // scale),
        mode="bicubic",
        antialias=True,
        align_corners=False,
    ).clamp(0, 1)
    b = dict(b)
    b["lr"] = lr.view(n, T, 3, H // scale, W // scale)
    b["jitter"] = torch.zeros_like(b["jitter"])
    return b


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    cfg = json.load(open(parser.parse_args().config))
    rank, world, device = setup_ddp()
    torch.manual_seed(cfg["seed"] + rank)
    gen = torch.Generator(device=device)
    gen.manual_seed(cfg["seed"] * 100 + rank)
    s, T, P = cfg["scale"], cfg["seq_len"], cfg["patch"]
    model = Sequence(build_tsr(cfg["tier"], s)).to(device)
    teacher = None
    if cfg.get("teacher_ckpt"):
        teacher = Sequence(build_tsr(cfg["teacher_tier"], s)).to(device)
        teacher.load_state_dict(
            torch.load(cfg["teacher_ckpt"], map_location=device)["ema"]
        )
        teacher.eval().requires_grad_(False)
    lpips_fn = (
        load_lpips(device, log=lambda m: emit("LOG", {"msg": m}))
        if cfg["w_lpips"] > 0
        else None
    )
    tiles = (
        torch.from_numpy(make_text_bank(64, cfg["seed"], height=24, width=128)).to(
            device
        )
        if cfg.get("world_text", True)
        else None
    )
    caches = {k: v for k, v in cfg["caches"].items() if v["count"] > 0}
    b_real = int(round(cfg["batch_per_gpu"] * cfg["p_real"])) if "seq" in caches else 0
    sizes = {"still": cfg["batch_per_gpu"] - b_real, "seq": b_real}
    loader = make_loader(
        caches, sizes, cfg["seed"] * 1000 + rank * 10**7, cfg["workers_per_rank"]
    )

    def make_batch(raw, g, n_synth_override=None):
        parts = []
        if "still/rgb" in raw and raw["still/rgb"].shape[0] > 0:
            parts.append(sr_batch(raw["still/rgb"], s, T, P, g, text_bank=tiles))
        if "seq/rgb" in raw and raw["seq/rgb"].shape[0] > 0:
            meta = raw["seq/meta"].float()
            parts.append(
                sr_batch_from_frames(
                    raw["seq/rgb"],
                    raw["seq/mv"],
                    raw["seq/valid"],
                    raw["seq/depth"],
                    meta[:, 0],
                    meta[:, 1],
                    s,
                    g,
                )
            )
        b = {k: torch.cat([p[k] for p in parts], 0) for k in parts[0]}
        return bicubic_inputs(b, s) if cfg.get("degradation") == "bicubic" else b

    def losses(out, b, teacher_out=None):
        n, T_ = out.shape[:2]
        hr = b["hr"]
        w_t = torch.ones(T_, device=out.device)
        w_t[0] = 0.25  # the first frame has no history
        o, g = out.flatten(0, 1), hr.flatten(0, 1)
        per = torch.stack([charbonnier(out[:, t], hr[:, t]) for t in range(T_)])
        l_pix = (per * w_t).sum() / w_t.sum()
        l_grad = gradient_loss(o, g)
        l_fft = fft_loss(o, g)
        l_tmp = out.new_zeros(())
        q = b["mv_quality"].view(n, 1, 1, 1)
        for t in range(1, T_):
            l_tmp = l_tmp + temporal_loss(
                out[:, t],
                out[:, t - 1],
                hr[:, t],
                hr[:, t - 1],
                b["mv_hr"][:, t],
                b["valid_hr"][:, t] * q,
            )
        l_tmp = l_tmp / max(T_ - 1, 1)
        l_lp = (
            lpips_fn(out[:, -1].clamp(0, 1), hr[:, -1]).mean()
            if lpips_fn is not None
            else out.new_zeros(())
        )
        l_kd = (
            (out[:, 1:] - teacher_out[:, 1:]).abs().mean()
            if teacher_out is not None
            else out.new_zeros(())
        )
        total = (
            l_pix
            + cfg["w_grad"] * l_grad
            + cfg["w_fft"] * l_fft
            + cfg["w_temporal"] * l_tmp
            + cfg["w_lpips"] * l_lp
            + cfg["w_distill"] * l_kd
        )
        return total, {
            "loss": total.detach(),
            "pix": l_pix.detach(),
            "temporal": l_tmp.detach(),
            "lpips": l_lp.detach(),
            "distill": l_kd.detach(),
        }

    def step_fn(net, raw, step):
        b = make_batch(raw, gen)
        with torch.autocast("cuda", dtype=torch.float16):
            out = net(
                b, cfg.get("use_history", True), cfg.get("ablate_gbuffers", False)
            )
            t_out = None
            if teacher is not None:
                with torch.no_grad():
                    t_out = teacher(b)
        loss, logs = losses(
            out.float(), b, t_out.float() if t_out is not None else None
        )
        return loss, logs, b["lr"].shape[0]

    # fixed validation batches (same for every round): synthetic from validation stills + real validation windows
    val_batches = []
    if rank == 0:
        vg = torch.Generator(device=device)
        vg.manual_seed(12345)
        vc = cfg["val_caches"]
        for k in range(cfg["val_batches"]):
            raw = {}
            for name in ("still", "seq"):
                if vc.get(name, {}).get("count", 0) > 0:
                    arrs = {
                        key: np.load(p, mmap_mode="r")
                        for key, p in vc[name]["arrays"].items()
                    }
                    idx = (
                        np.arange(k * cfg["val_bs"], (k + 1) * cfg["val_bs"])
                        % vc[name]["count"]
                    )
                    for key, a in arrs.items():
                        raw[f"{name}/{key}"] = torch.from_numpy(
                            np.ascontiguousarray(a[np.sort(idx)])
                        ).to(device)
            if raw:
                val_batches.append(
                    {k2: v.cpu() for k2, v in make_batch(raw, vg).items()}
                )

    @torch.no_grad()
    def val_fn(ema_model):
        rows = {"psnr_y": [], "psnr_y_last": [], "tce": []}
        for vb in val_batches:
            b = {k: v.to(device) for k, v in vb.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                out = (
                    ema_model(
                        b,
                        cfg.get("use_history", True),
                        cfg.get("ablate_gbuffers", False),
                    )
                    .float()
                    .clamp(0, 1)
                )
            hr = b["hr"]
            T_ = out.shape[1]
            ps = torch.stack(
                [
                    psnr(rgb_to_y(out[:, t]), rgb_to_y(hr[:, t]), border=s)
                    for t in range(1, T_)
                ],
                1,
            )
            rows["psnr_y"].append(ps.mean(1))
            rows["psnr_y_last"].append(ps[:, -1])
            tce = torch.stack(
                [
                    temporal_change_error(
                        out[:, t],
                        out[:, t - 1],
                        hr[:, t],
                        hr[:, t - 1],
                        b["mv_hr"][:, t],
                        b["valid_hr"][:, t],
                    )
                    for t in range(1, T_)
                ],
                1,
            ).mean(1)
            rows["tce"].append(tce)
        return {k: torch.cat(v).mean().item() for k, v in rows.items() if v}

    train_loop(cfg, model, step_fn, val_fn, rank, world, device, loader)
    if world > 1:
        torch.distributed.destroy_process_group()


if __name__ == "__main__":
    main()

## 2. Setup and dependencies

Installs the two packages Kaggle does not ship (`lpips` for the perceptual loss and metric, `flip-evaluator` for NVIDIA FLIP) plus `onnxruntime` for the export check; each is optional and its absence is logged. Imports the library and creates the run folders and logger. Expected output: install status per package.

In [ ]:
import copy
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, "/kaggle/working/code")
import nss_common as C

RUN = C.Run("nss_sr")
log = RUN.log.info
PKGS = C.pip_install(
    {
        "lpips": "lpips",
        "flip-evaluator": "flip_evaluator",
        "onnxruntime": "onnxruntime",
    },
    log=RUN.log,
)
print(PKGS)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

import nss_cache as K
import nss_data as D
import nss_eval as E
import nss_metrics as M
import nss_models as NM
import nss_pipeline as PL
import nss_synth as S

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

## 3. Hardware check

GPUs, CPU cores, RAM and disk detected at run time; stops early unless two GPUs are visible, because training uses one process per GPU. Expected output: two Tesla T4 rows.

In [ ]:
HW = C.hardware_report()
RUN.save_json(HW, "hardware.json")
print(pd.Series({k: v for k, v in HW.items() if k != "gpu_list"}).to_string())
print(pd.DataFrame(HW["gpu_list"]).to_string(index=False))
assert HW["gpus"] >= 2, "Select the GPU T4 x2 accelerator (Settings > Accelerator)."
DEVICES = [torch.device("cuda", i) for i in range(HW["gpus"])]

## 4. Configuration

Every tunable value. Key choices:

* `scale` 3 is the primary ratio (360p to 1080p, the extreme case); `scale_alt` 2 is the less aggressive ratio (540p to 1080p) trained for the `low` tier.
* `patch` 192 output pixels (64 low-resolution pixels at 3x) and `seq_len` 6 frames per training sequence (truncated back-propagation through the recurrence); `still_size` 512 texture crops leave room for fast camera paths.
* `p_real` is the share of each batch drawn from real sequences; the rest is synthetic.
* Loss weights follow the research plan: temporal 0.5, LPIPS 0.05, distillation 0.5.
* `minutes` are wall-clock training budgets per stage; they are scaled down automatically if the run budget is short.

In [ ]:
QUICK_RUN = False
RUN_BUDGET_H = 9.5
CFG = {
    "seed": 11,
    "scale": 3,
    "scale_alt": 2,
    "patch": 192,
    "seq_len": 6,
    "still_size": 512,
    "student_tier": "low",
    "small_tier": "igpu",
    "teacher_tier": "teacher",
    "cache": {
        "still_train": 600 if QUICK_RUN else 6500,
        "still_val": 64 if QUICK_RUN else 256,
        "win_train": 300 if QUICK_RUN else 3600,
        "win_val": 48 if QUICK_RUN else 192,
        "build_min": 6 if QUICK_RUN else 30,
    },
    "p_real": 0.4,
    "lr": 2e-3,
    "lr_teacher": 1e-3,
    "min_lr": 1e-6,
    "warmup_frac": 0.03,
    "weight_decay": 1e-4,
    "ema_decay": 0.999,
    "grad_clip": 1.0,
    "w_grad": 0.1,
    "w_fft": 0.05,
    "w_temporal": 0.5,
    "w_lpips": 0.05,
    "w_distill": 0.5,
    "val_bs": 8,
    "val_batches": 2 if QUICK_RUN else 4,
    "val_interval_min": 8,
    "minutes": {
        "teacher": 4 if QUICK_RUN else 100,
        "student": 4 if QUICK_RUN else 80,
        "pair": 3 if QUICK_RUN else 50,
        "ablation": 3 if QUICK_RUN else 20,
    },
    "eval": {
        "synthetic_per_class": 2 if QUICK_RUN else 6,
        "synthetic_T": 8 if QUICK_RUN else 16,
        "synthetic_P": 384,
        "real_clips": 2 if QUICK_RUN else 8,
        "real_T": 8 if QUICK_RUN else 16,
        "eval_reserve_h": 1.0,
    },
    "pipeline": {
        "list_cap_files": 3000 if QUICK_RUN else 20000,
        "seed": 7,
        "workers": os.cpu_count(),
        "hf_caps": {
            "game_stills": (120, 30) if QUICK_RUN else (600, 120),
            "tartanair": 30 if QUICK_RUN else 80,
            "tartanair_seq": 24,
            "gameir": (3, 3) if QUICK_RUN else (20, 12),
            "vimeo1080p": (6, 3) if QUICK_RUN else (60, 16),
        },
        "calibration_pairs": 300,
        "val_fraction": 0.05,
    },
}
ABLATIONS = {
    "reference": {},
    "no_history": {"use_history": False, "find_unused": True},
    "no_gbuffers": {"ablate_gbuffers": True},
    "bicubic_degradation": {"degradation": "bicubic"},
    "no_temporal_loss": {"w_temporal": 0.0},
    "no_distillation": {"teacher_ckpt": None},
}
RUN.save_json(
    {
        "quick_run": QUICK_RUN,
        "budget_h": RUN_BUDGET_H,
        "cfg": CFG,
        "ablations": ABLATIONS,
    },
    "config.json",
)
print(json.dumps(CFG["minutes"]), json.dumps(CFG["cache"]))

## 5. Dataset manifest

Looks for `nss_manifest.parquet` in an attached `neural-data-pipeline` output. If it is missing, the same pipeline runs inline with smaller caps (Kaggle discovery, Hugging Face subsets in `/tmp/nss_data`, quality filters, pHash + DINOv2 near-duplicate search, leakage-safe split), which adds roughly 30 to 60 minutes. Expected output: where the manifest came from and the item counts per split.

In [ ]:
with RUN.stage("manifest"):
    DATA_ROOT = D.locate_data_root(C.INPUT_ROOT)
    if DATA_ROOT is None:
        log("no attached pipeline output: building the manifest inline (smaller caps)")
        DATA_ROOT = Path("/tmp/nss_data")
        PL.build_manifest(DATA_ROOT, C.INPUT_ROOT, CFG["pipeline"], DEVICES, log=log)
    MANIFEST = D.load_manifest(DATA_ROOT / "nss_manifest.parquet")
log(f"data root: {DATA_ROOT}")
print(pd.crosstab(MANIFEST["source"], MANIFEST["split"]).to_string())

## 6. Training and validation item selection

Textures for the synthetic renderer come from stills and from frames of high-resolution sequences (at least 512 px on the shorter side); real training windows come from every sequence source. Test items are never touched until the evaluation section. Expected output: item counts per role.

In [ ]:
TEX_SOURCES = ["div2k", "flickr2k", "game_stills", "gameir", "vimeo1080p", "reds"]
SEQ_SOURCES = ["tartanair", "sintel", "reds", "vimeo_septuplet", "vimeo1080p", "gameir"]
T, P = CFG["seq_len"], CFG["patch"]
ITEMS = {}
for split in ("train", "val"):
    ITEMS[f"tex_{split}"] = K.items_for(MANIFEST, split, TEX_SOURCES)
    ITEMS[f"seq_{split}"] = K.items_for(
        MANIFEST, split, SEQ_SOURCES, kinds=["seq"], min_frames=T
    )
print(pd.Series({k: len(v) for k, v in ITEMS.items()}).to_string())
assert ITEMS[
    "tex_train"
], "No texture sources found: attach the pipeline output or the Kaggle datasets."

## 7. Training caches

Decodes once, with all CPU cores, into memory-mapped arrays in `/kaggle/working/cache` (shared by both training processes through the page cache):

* still crops of 512x512 for the synthetic renderer (random down-scale per source for varied detail density, flat crops rejected);
* windows of 6 frames at 192x192 with motion vectors, validity and depth at half resolution.

Windows from video sources get RAFT motion (frame to previous frame) with a forward-backward consistency mask, computed on both GPUs. Expected output: progress bars, counts per source and the cache size.

In [ ]:
CACHE = C.WORK / "cache"
CACHE.mkdir(exist_ok=True)
CACHES = {}
with RUN.stage("caches"):
    deadline = CFG["cache"]["build_min"] * 60
    for split in ("train", "val"):
        n = CFG["cache"][f"still_{split}"]
        tasks = K.still_tasks(
            ITEMS[f"tex_{split}"],
            DATA_ROOT,
            n,
            CFG["still_size"],
            2 if split == "train" else 1,
            CFG["seed"],
        )
        cnt = K.build_still_cache(
            tasks,
            CACHE / f"still_{split}.npy",
            n,
            CFG["still_size"],
            os.cpu_count(),
            deadline,
            log=log,
        )
        CACHES[f"still_{split}"] = {
            "arrays": {"rgb": str(CACHE / f"still_{split}.npy")},
            "count": cnt,
        }
    RAFTS = [M.load_raft(d, log=log) for d in DEVICES]
    RAFTS = RAFTS if all(r is not None for r in RAFTS) else []
    WIN_COUNTS = {}
    for split in ("train", "val"):
        prefix = CACHE / f"win_{split}"
        cnt, counts, need = K.build_window_cache(
            ITEMS[f"seq_{split}"],
            DATA_ROOT,
            prefix,
            CFG["cache"][f"win_{split}"],
            T,
            P,
            os.cpu_count(),
            deadline,
            CFG["seed"] + (split == "val"),
            log=log,
        )
        K.fill_estimated_motion(prefix, need, RAFTS, DEVICES[: len(RAFTS)], log=log)
        CACHES[f"seq_{split}"] = K.cache_spec(
            prefix, cnt, ["rgb", "mv", "valid", "depth", "meta"]
        )
        WIN_COUNTS[split] = counts
cache_gb = sum(p.stat().st_size for p in CACHE.glob("*.npy")) / 1e9
RUN.save_json(
    {"caches": CACHES, "window_sources": WIN_COUNTS, "cache_gb": round(cache_gb, 2)},
    "caches.json",
)
print(
    pd.DataFrame(WIN_COUNTS).fillna(0).astype(int).to_string(),
    f"\ncache size {cache_gb:.2f} GB",
)

## 8. Cache inspection

Random still crops, and for real windows: first and last frame, the motion-vector field of the last frame (colour wheel: hue = direction, brightness = magnitude) and its validity (black = disoccluded or out of view). Expected output: two image grids and per-window statistics (share with exact motion, depth, mean motion magnitude).

In [ ]:
def flow_to_rgb(f):
    mag = np.linalg.norm(f, axis=-1)
    ang = np.arctan2(f[..., 1], f[..., 0])
    hsv = np.stack(
        [
            (ang + np.pi) / (2 * np.pi),
            np.ones_like(mag),
            np.clip(mag / (np.percentile(mag, 99) + 1e-6), 0, 1),
        ],
        -1,
    )
    import matplotlib.colors as mcolors

    return mcolors.hsv_to_rgb(hsv)


st = np.load(CACHES["still_train"]["arrays"]["rgb"], mmap_mode="r")
fig, axes = plt.subplots(2, 8, figsize=(16, 4.2))
for ax, i in zip(
    axes.flat,
    np.random.default_rng(0).choice(CACHES["still_train"]["count"], 16, replace=False),
):
    ax.imshow(st[i])
    ax.axis("off")
fig.suptitle("Still crops (textures for the synthetic renderer)")
RUN.show(fig, "cache_stills")
if CACHES["seq_train"]["count"]:
    arr = {
        k: np.load(p, mmap_mode="r") for k, p in CACHES["seq_train"]["arrays"].items()
    }
    pick = np.random.default_rng(1).choice(
        CACHES["seq_train"]["count"],
        min(4, CACHES["seq_train"]["count"]),
        replace=False,
    )
    fig, axes = plt.subplots(len(pick), 4, figsize=(12, 3 * len(pick)), squeeze=False)
    for r, i in enumerate(pick):
        for c, (img, title) in enumerate(
            [
                (arr["rgb"][i, 0], "frame 0"),
                (arr["rgb"][i, -1], f"frame {T - 1}"),
                (flow_to_rgb(arr["mv"][i, -1].astype(np.float32)), "motion vectors"),
                (arr["valid"][i, -1], "validity"),
            ]
        ):
            axes[r, c].imshow(img, cmap="gray" if c == 3 else None)
            axes[r, c].set_title(
                f"{title} (mv quality {arr['meta'][i, 1]:.1f})", fontsize=8
            )
            axes[r, c].axis("off")
    RUN.show(fig, "cache_windows")
    meta = arr["meta"][: CACHES["seq_train"]["count"]]
    print(
        pd.Series(
            {
                "exact_motion_share": float((meta[:, 1] >= 1).mean()),
                "estimated_motion_share": float((meta[:, 1] == 0.5).mean()),
                "depth_share": float(meta[:, 0].mean()),
                "mean_mv_px": float(
                    np.abs(arr["mv"][: min(256, len(meta))].astype(np.float32)).mean()
                ),
            }
        )
        .round(3)
        .to_string()
    )

## 9. Synthetic renderer preview

One synthetic training sequence: jittered low-resolution input (nearest-neighbour enlarged, showing aliasing), the anti-aliased target, exact motion vectors, disocclusion mask, depth and particle (reactive) mask. Expected output: a 6-column grid for three frames.

In [ ]:
g = torch.Generator(device=DEVICES[0])
g.manual_seed(3)
TEXT_BANK = torch.from_numpy(
    S.make_text_bank(64, CFG["seed"], height=24, width=128)
).to(DEVICES[0])
stills_t = torch.from_numpy(np.ascontiguousarray(st[:4])).to(DEVICES[0])
demo = S.sr_batch(stills_t, CFG["scale"], T, P, g, text_bank=TEXT_BANK)
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for r, t in enumerate([1, 3, 5]):
    panels = [
        (
            F.interpolate(
                demo["lr"][0, t : t + 1], scale_factor=CFG["scale"], mode="nearest"
            )[0],
            "LR input (jittered)",
        ),
        (demo["hr"][0, t], "target (4x SSAA)"),
        (
            torch.from_numpy(
                flow_to_rgb(demo["mv_hr"][0, t].permute(1, 2, 0).cpu().numpy())
            ).permute(2, 0, 1),
            "motion vectors",
        ),
        (demo["valid_hr"][0, t].expand(3, -1, -1), "valid history"),
        (demo["depth_lr"][0, t].expand(3, -1, -1), "depth (1 = near)"),
        (demo["reactive_lr"][0, t].expand(3, -1, -1), "reactive (particles)"),
    ]
    for c, (img, title) in enumerate(panels):
        axes[r, c].imshow(img.permute(1, 2, 0).float().cpu().clamp(0, 1).numpy())
        axes[r, c].set_title(f"t={t}: {title}", fontsize=8)
        axes[r, c].axis("off")
RUN.show(fig, "synthetic_preview")

## 10. Model tiers, cost and fusion parity

For each tier: trainable parameters, deployed (fused) parameters, multiply-accumulates per frame at 640x360 to 1920x1080 (3x) and 960x540 to 1920x1080 (2x), and the maximum difference between the multi-branch and fused networks on random input (should be about 1e-5 or less). Expected output: a tier table and a bar chart.

In [ ]:
tier_rows = []
for tier in NM.TSR_TIERS:
    for s in (CFG["scale"], CFG["scale_alt"]):
        net = NM.build_tsr(tier, s).eval()
        f = {
            "color": torch.rand(1, 3, 32, 32),
            "depth": torch.rand(1, 1, 32, 32),
            "mv": torch.randn(1, 2, 32, 32),
            "reactive": torch.zeros(1, 1, 32, 32),
            "exposure": torch.zeros(1),
            "jitter": torch.rand(1, 2) - 0.5,
            "has_depth": torch.ones(1),
            "mv_quality": torch.ones(1),
        }
        with torch.no_grad():
            for p in net.out.parameters():
                p.normal_(
                    0, 0.01
                )  # non-zero head so parity is tested on a non-trivial output
            fused = NM.fuse_model(net)
            o1, st1 = net(f, None)
            o2, _ = fused(f, None)
            o1b, _ = net(f, st1)
            o2b, _ = fused(f, st1)
        lr_px = (1920 // s) * (1080 // s)
        macs = NM.tsr_macs_per_lr_pixel(net)
        tier_rows.append(
            {
                "tier": tier,
                "scale": s,
                "train_params": NM.count_params(net),
                "fused_params": NM.count_params(fused),
                "gmac_per_frame_1080p": round(macs * lr_px / 1e9, 2),
                "parity_max_abs": float(
                    max((o1 - o2).abs().max(), (o1b - o2b).abs().max())
                ),
            }
        )
TIERS = pd.DataFrame(tier_rows)
RUN.save_csv(TIERS, "model_tiers.csv", index=False)
print(TIERS.to_string(index=False))
assert TIERS["parity_max_abs"].max() < 1e-3, "fusion parity failed"
fig, ax = plt.subplots(figsize=(9, 3.5))
TIERS.pivot(index="tier", columns="scale", values="gmac_per_frame_1080p").plot.bar(
    ax=ax, rot=0
)
ax.set_ylabel("GMAC per 1080p output frame")
RUN.show(fig, "model_tiers")

## 11. Batch-size probe

Runs full training steps (synthetic batch generation, recurrent unroll, losses, backward) on GPU 0 for increasing batch sizes, for the teacher alone and for the student with the frozen teacher alongside (distillation), and keeps the largest batch below 85 percent of VRAM. Expected output: a table of batch size, peak VRAM share and sequences per second.

In [ ]:
def probe(student_tier, teacher_tier, scale, sizes):
    rows = []
    dev = DEVICES[0]
    total = torch.cuda.get_device_properties(dev).total_memory
    stills = torch.from_numpy(np.ascontiguousarray(st[: max(sizes)])).to(dev)
    for bs in sizes:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(dev)
        try:
            net = NM.build_tsr(student_tier, scale).to(dev)
            teacher = (
                NM.build_tsr(teacher_tier, scale).to(dev).eval().requires_grad_(False)
                if teacher_tier
                else None
            )
            opt = torch.optim.AdamW(net.parameters(), 1e-4)
            scaler = torch.amp.GradScaler("cuda")
            times = []
            for _ in range(3):
                torch.cuda.synchronize(dev)
                t0 = time.time()
                b = S.sr_batch(stills[:bs], scale, T, P, g)
                with torch.autocast("cuda", dtype=torch.float16):
                    outs, state = [], None
                    for t in range(T):
                        o, state = net(E.frame_inputs(b, t), state)
                        outs.append(o)
                    if teacher is not None:
                        with torch.no_grad():
                            st_ = None
                            for t in range(T):
                                _, st_ = teacher(E.frame_inputs(b, t), st_)
                    loss = sum(
                        M.charbonnier(o.float(), b["hr"][:, t])
                        for t, o in enumerate(outs)
                    )
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                torch.cuda.synchronize(dev)
                times.append(time.time() - t0)
            rows.append(
                {
                    "batch": bs,
                    "fits": True,
                    "peak_vram_pct": round(
                        100 * torch.cuda.max_memory_allocated(dev) / total, 1
                    ),
                    "seq_per_s": round(bs / np.mean(times[1:]), 1),
                }
            )
            del net, teacher, opt, outs, state, loss, b
        except torch.cuda.OutOfMemoryError:
            rows.append({"batch": bs, "fits": False})
            break
    torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    ok = df[df["fits"] & (df["peak_vram_pct"] <= 85)]
    return df, int(ok["batch"].max()) if len(ok) else sizes[0]


with RUN.stage("batch_probe"):
    sizes = [4, 8] if QUICK_RUN else [4, 8, 12, 16, 24, 32, 48]
    PROBE_T, BS_TEACHER = probe(CFG["teacher_tier"], None, CFG["scale"], sizes)
    PROBE_S, BS_STUDENT = probe(
        CFG["student_tier"], CFG["teacher_tier"], CFG["scale"], sizes
    )
print(
    "teacher\n",
    PROBE_T.to_string(index=False),
    "\nstudent + teacher\n",
    PROBE_S.to_string(index=False),
)
print(f"batch per GPU: teacher {BS_TEACHER}, student {BS_STUDENT}")
RUN.save_csv(
    pd.concat([PROBE_T.assign(run="teacher"), PROBE_S.assign(run="student")]),
    "batch_probe.csv",
    index=False,
)

## 12. Training launcher and budget

Builds one JSON config per run and launches `code/train_tsr.py`: DDP on both GPUs for the main stages, or two single-GPU runs in parallel threads for the paired stages and ablations. Each run streams `PROGRESS` lines into a progress bar (percent of its time budget, loss, sequences per second, learning rate, per-GPU memory) and `ROUND` lines (validation PSNR-Y, temporal-change error) into a live chart. Stage budgets are scaled down if the run budget is short. Expected output: the planned minutes per stage.

In [ ]:
remaining_h = RUN_BUDGET_H - RUN.hours() - CFG["eval"]["eval_reserve_h"]
plan_min = (
    CFG["minutes"]["teacher"]
    + CFG["minutes"]["student"]
    + CFG["minutes"]["pair"]
    + 3 * CFG["minutes"]["ablation"]
)
SCALE_T = min(1.0, max(0.1, remaining_h * 60 / plan_min))
MINUTES = {k: v * SCALE_T for k, v in CFG["minutes"].items()}
print(
    pd.Series(MINUTES).round(1).to_string(),
    f"\n(scale {SCALE_T:.2f}, {remaining_h:.2f} h available for training)",
)
HISTORY = {}
PORT = [29500]


def run_cfg(name, tier, scale, minutes, batch, teacher_ckpt=None, **over):
    cfg = {
        "name": name,
        "tier": tier,
        "scale": scale,
        "seq_len": T,
        "patch": P,
        "seed": CFG["seed"],
        "batch_per_gpu": batch,
        "p_real": CFG["p_real"],
        "lr": CFG["lr_teacher"] if tier == "teacher" else CFG["lr"],
        "min_lr": CFG["min_lr"],
        "warmup_frac": CFG["warmup_frac"],
        "weight_decay": CFG["weight_decay"],
        "ema_decay": CFG["ema_decay"],
        "grad_clip": CFG["grad_clip"],
        "w_grad": CFG["w_grad"],
        "w_fft": CFG["w_fft"],
        "w_temporal": CFG["w_temporal"],
        "w_lpips": CFG["w_lpips"] if PKGS.get("lpips") else 0.0,
        "w_distill": CFG["w_distill"],
        "teacher_ckpt": teacher_ckpt,
        "teacher_tier": CFG["teacher_tier"],
        "train_seconds": minutes * 60,
        "val_interval_s": min(CFG["val_interval_min"] * 60, max(minutes * 60 / 6, 60)),
        "progress_interval_s": 15,
        "sync_every": 10,
        "select_metric": "psnr_y",
        "caches": {"still": CACHES["still_train"], "seq": CACHES["seq_train"]},
        "val_caches": {"still": CACHES["still_val"], "seq": CACHES["seq_val"]},
        "val_bs": CFG["val_bs"],
        "val_batches": CFG["val_batches"],
        "workers_per_rank": max(1, os.cpu_count() // 2 - 0),
        "weights_dir": str(RUN.dirs["weights"]),
        "metrics_dir": str(RUN.dirs["metrics"]),
    }
    cfg.update(over)
    return cfg


def live_plot(title):
    handle = display(plt.figure(), display_id=True)

    def update(rounds):
        h = pd.DataFrame(rounds)
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
        axes[0].plot(h["elapsed_min"], h["train_loss"])
        axes[0].set_title("train loss")
        axes[1].plot(h["elapsed_min"], h["val_psnr_y"])
        axes[1].set_title("val PSNR-Y (dB)")
        axes[2].plot(h["elapsed_min"], h["val_tce"])
        axes[2].set_title("val temporal-change error")
        for ax in axes:
            ax.set_xlabel("minutes")
        fig.suptitle(title)
        handle.update(fig)
        plt.close(fig)

    return update


def launch(cfg, gpus, live=True):
    PORT[0] += 1
    cfg_path = RUN.dirs["code"] / f"cfg_{cfg['name']}.json"
    rounds, done = C.launch_training(
        RUN.work / "code" / "train_tsr.py",
        cfg,
        cfg_path,
        RUN.dirs["logs"] / f"{cfg['name']}.log",
        gpus,
        cfg["name"],
        port=PORT[0],
        on_round=live_plot(cfg["name"]) if live else None,
    )
    HISTORY[cfg["name"]] = rounds
    return done


def launch_pair(cfg_a, cfg_b):
    with ThreadPoolExecutor(2) as pool:
        fa = pool.submit(launch, cfg_a, [0], False)
        fb = pool.submit(launch, cfg_b, [1], False)
        return fa.result(), fb.result()

## 13. Stage A: teacher (both GPUs)

The `teacher` tier (64 channels, 8 blocks, 6 U-Net blocks) at 3x on both GPUs. It is too large for deployment; it only provides distillation targets for the students. Expected output: a live training chart and the final validation scores.

In [ ]:
with RUN.stage("train_teacher"):
    S3 = CFG["scale"]
    done_teacher = launch(
        run_cfg(
            "tsr_teacher_s3", CFG["teacher_tier"], S3, MINUTES["teacher"], BS_TEACHER
        ),
        [0, 1],
    )
TEACHER_CKPT = str(RUN.dirs["weights"] / "tsr_teacher_s3_best.pt")
print(done_teacher)

## 14. Stage B: main student with distillation (both GPUs)

The `low` tier at 3x, trained with the full loss plus distillation toward the frozen teacher. Expected output: live chart and final scores.

In [ ]:
with RUN.stage("train_student"):
    done_student = launch(
        run_cfg(
            f"tsr_{CFG['student_tier']}_s3",
            CFG["student_tier"],
            S3,
            MINUTES["student"],
            BS_STUDENT,
            teacher_ckpt=TEACHER_CKPT,
        ),
        [0, 1],
    )
print(done_student)

## 15. Stage C: integrated-GPU student (3x) and 2x student, in parallel

GPU 0 trains the smallest `igpu` tier at 3x with distillation; GPU 1 trains the `low` tier at 2x (the teacher is 3x only, so no distillation). Expected output: both runs' final scores.

In [ ]:
with RUN.stage("train_pair"):
    S2 = CFG["scale_alt"]
    done_pair = launch_pair(
        run_cfg(
            f"tsr_{CFG['small_tier']}_s3",
            CFG["small_tier"],
            S3,
            MINUTES["pair"],
            BS_STUDENT,
            teacher_ckpt=TEACHER_CKPT,
        ),
        run_cfg(
            f"tsr_{CFG['student_tier']}_s2",
            CFG["student_tier"],
            S2,
            MINUTES["pair"],
            BS_STUDENT,
        ),
    )
print(done_pair)

## 16. Ablations (three parallel pairs)

Short runs of the `low` 3x student with one change each, against a `reference` run with the full recipe and the same short budget: no history (single-frame upscaling), no G-buffers (colour only: no depth or motion vectors), bicubic degradation (classic image super-resolution inputs instead of render-like jittered sampling), no temporal loss, and no distillation. Expected output: final scores of the six runs.

In [ ]:
with RUN.stage("ablations"):
    names = list(ABLATIONS)
    for a, b in zip(names[0::2], names[1::2]):
        cfgs = []
        for name in (a, b):
            over = dict(ABLATIONS[name])
            ckpt = over.pop("teacher_ckpt", TEACHER_CKPT)
            cfgs.append(
                run_cfg(
                    f"abl_{name}",
                    CFG["student_tier"],
                    S3,
                    MINUTES["ablation"],
                    BS_STUDENT,
                    teacher_ckpt=ckpt,
                    **over,
                )
            )
        print(launch_pair(*cfgs))

## 17. Training curves

Validation PSNR-Y and temporal-change error against time for every run, and the train loss of the main stages. Expected output: three charts and a table of the best validation round per run.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
best_rows = []
for name, rounds in HISTORY.items():
    if not rounds:
        continue
    h = pd.DataFrame(rounds)
    RUN.save_csv(h, f"history_{name}.csv", index=False)
    style = "-" if not name.startswith("abl_") else "--"
    axes[0].plot(h["elapsed_min"], h["val_psnr_y"], style, label=name)
    axes[1].plot(h["elapsed_min"], h["val_tce"], style, label=name)
    if not name.startswith("abl_"):
        axes[2].plot(h["elapsed_min"], h["train_loss"], label=name)
    b = h.loc[h["val_psnr_y"].idxmax()]
    best_rows.append(
        {
            "run": name,
            "best_round": int(b["round"]),
            "val_psnr_y": b["val_psnr_y"],
            "val_psnr_y_last": b["val_psnr_y_last"],
            "val_tce": b["val_tce"],
            "samples_per_s": h["samples_per_s"].mean(),
            "peak_mem_gb": max(max(m) for m in h["mem_gb"]),
        }
    )
for ax, t in zip(axes, ["val PSNR-Y (dB)", "val temporal-change error", "train loss"]):
    ax.set_title(t)
    ax.set_xlabel("minutes")
axes[0].legend(fontsize=7)
RUN.show(fig, "training_curves")
BEST = pd.DataFrame(best_rows)
RUN.save_csv(BEST, "best_validation.csv", index=False)
print(BEST.round(4).to_string(index=False))

## 18. Load, fuse and place models

Loads the best EMA weights of every run, folds RepConv branches into plain convolutions, checks fused against unfused output on a real validation batch, and keeps FP16 fused replicas on GPU 0 for evaluation. Expected output: a table of runs with fused parameter count and parity.

In [ ]:
def load_tsr(name, tier, scale):
    path = RUN.dirs["weights"] / f"{name}_best.pt"
    if not path.exists():
        return None
    net = NM.build_tsr(tier, scale)
    sd = torch.load(path, map_location="cpu")["ema"]
    net.load_state_dict(
        {k[4:] if k.startswith("net.") else k: v for k, v in sd.items()}
    )
    return net.eval()


RUNS = {
    "teacher_s3": ("tsr_teacher_s3", CFG["teacher_tier"], S3),
    "student_s3": (f"tsr_{CFG['student_tier']}_s3", CFG["student_tier"], S3),
    "igpu_s3": (f"tsr_{CFG['small_tier']}_s3", CFG["small_tier"], S3),
    "student_s2": (f"tsr_{CFG['student_tier']}_s2", CFG["student_tier"], S2),
}
RUNS.update({f"abl_{a}": (f"abl_{a}", CFG["student_tier"], S3) for a in ABLATIONS})
MODELS, rows = {}, []
vb = {k: v.to(DEVICES[0]) for k, v in S.sr_batch(stills_t[:2], S3, 4, P, g).items()}
for key, (name, tier, scale) in RUNS.items():
    net = load_tsr(name, tier, scale)
    if net is None:
        continue
    fused = NM.fuse_model(net).to(DEVICES[0]).half()
    torch.save(
        {"state_dict": fused.state_dict(), "tier": tier, "scale": scale},
        RUN.dirs["weights"] / f"{name}_fused.pt",
    )
    if scale == S3:  # FP32 parity on a synthetic validation batch
        a = E.run_tsr(net.to(DEVICES[0]).float(), vb, amp=False)
        b2 = E.run_tsr(NM.fuse_model(net).to(DEVICES[0]).float(), vb, amp=False)
        parity = float((a - b2).abs().max())
    else:
        parity = None
    MODELS[key] = (fused, scale)
    net.cpu()
    rows.append(
        {
            "model": key,
            "tier": tier,
            "scale": scale,
            "fused_params": NM.count_params(fused),
            "parity_max_abs": parity,
        }
    )
print(pd.DataFrame(rows).to_string(index=False))
LPIPS = M.load_lpips(DEVICES[0], log=log)
RAFT = RAFTS[0] if RAFTS else M.load_raft(DEVICES[0], log=log)

## 19. Test suites

Built only from the test split:

* **synthetic-game**: layered scenes from held-out game frames and DIV2K validation images at 384x384 output, one set per motion class (static, slow 0.3 to 3 px/frame, medium 3 to 10, fast 10 to 28), with exact motion, disocclusions, thin geometry and particles;
* **tartanair**: held-out environments (Japanese alley, seaside town) with exact motion and depth, 576x480;
* **sintel**: held-out `market` and `cave` scene families with exact motion, 1008x432;
* **video**: Vid4 and held-out Vimeo-1080p clips with RAFT motion;
* **gameir-native**: held-out town 05, native 720p renders to 1440p ground truth (2x, a real renderer's low-resolution image instead of a simulated one).

Expected output: the number of sequences per suite.

In [ ]:
ev = CFG["eval"]
TEST_ITEMS = D.items_from_frame(MANIFEST[MANIFEST["split"] == "test"])
test_still_items = [i for i in TEST_ITEMS if i["source"] in ("game_stills", "div2k")]
SUITES = {}
with RUN.stage("test_suites"):
    S_EV = 1024  # texture crop size for 384-px test views
    tasks = K.still_tasks(
        test_still_items, DATA_ROOT, 4 * ev["synthetic_per_class"], S_EV, 1, 99
    )
    crops = []
    for task in tasks:
        _, c = D.still_crop(task)
        if c is not None:
            crops.append(c[0])
        if len(crops) >= 4 * ev["synthetic_per_class"]:
            break
    TEST_STILLS = torch.from_numpy(np.stack(crops)).to(DEVICES[0]) if crops else None

    def clips(source, crop, n, Tn, max_side=None):
        out = []
        for it in [
            i for i in TEST_ITEMS if i["source"] == source and i["n_frames"] >= Tn
        ][: n * 3]:
            c = E.load_clip(DATA_ROOT, it, 0, Tn, crop, max_side=max_side)
            if c is not None:
                out.append((it["item_id"], c))
            if len(out) >= n:
                break
        return out

    REAL = {
        "tartanair": clips("tartanair", (480, 576), ev["real_clips"], ev["real_T"]),
        "sintel": clips("sintel", (432, 1008), ev["real_clips"], ev["real_T"]),
        "vid4": clips("vid4", (480, 696), ev["real_clips"], ev["real_T"]),
        "vimeo1080p": clips(
            "vimeo1080p", (540, 960), ev["real_clips"], ev["real_T"], max_side=540
        ),
        "gameir_native": clips(
            "gameir", (1440, 1920), max(2, ev["real_clips"] // 2), 6
        ),
    }
print(
    pd.Series(
        {
            "synthetic stills": 0 if TEST_STILLS is None else len(TEST_STILLS),
            **{k: len(v) for k, v in REAL.items()},
        }
    ).to_string()
)

## 20. Evaluation

Runs every method on every suite at 3x (and the 2x student on 2x inputs) and records per-frame metrics, skipping the first frame (no history yet): PSNR-Y, SSIM-Y, PSNR inside disoccluded regions, fine-detail regions and particles, LPIPS and tOF every fourth frame, warping error (exact motion only) and temporal-change error. Methods: bicubic, jitter-aware bilinear, analytic TAA upscaler, the teacher, the students and the ablations. Expected output: progress logs and the number of metric rows.

In [ ]:
ROWS = []


def evaluate(suite, b, scale, methods, extra=None):
    ref = (
        MODELS["student_s3"][0] if scale == S3 else MODELS.get("student_s2", (None,))[0]
    )
    for mname in methods:
        if mname == "bicubic":
            out = E.run_bicubic(b, scale)
        elif (
            mname == "bilinear"
        ):  # the reference network only provides its parameter-free upsampler
            out = E.run_bilinear(ref, b)
        elif mname == "analytic_taau":
            out = E.run_taau(ref, b, scale)
        else:
            net, s_ = MODELS[mname]
            if s_ != scale:
                continue
            kw = {"use_history": False} if mname == "abl_no_history" else {}
            kw.update({"ablate_gbuffers": True} if mname == "abl_no_gbuffers" else {})
            out = E.run_tsr(net, b, **kw)
        for r in E.sr_frame_metrics(out, b, scale, LPIPS, RAFT):
            ROWS.append(
                {"suite": suite, "method": mname, "scale": scale, **(extra or {}), **r}
            )


METHODS_S3 = ["bicubic", "bilinear", "analytic_taau"] + [
    k for k, v in MODELS.items() if v[1] == S3
]
with RUN.stage("evaluation"):
    if TEST_STILLS is not None:
        for cls, name in enumerate(S.MOTION_PROFILE["names"]):
            sub = TEST_STILLS[
                cls * ev["synthetic_per_class"] : (cls + 1) * ev["synthetic_per_class"]
            ]
            if len(sub):
                b = E.synthetic_suite(
                    sub,
                    S3,
                    ev["synthetic_T"],
                    ev["synthetic_P"],
                    cls,
                    1000 + cls,
                    DEVICES[0],
                    text_bank=TEXT_BANK,
                )
                evaluate("synthetic-game", b, S3, METHODS_S3, {"motion": name})
                if "student_s2" in MODELS:
                    b2 = E.synthetic_suite(
                        sub,
                        S2,
                        ev["synthetic_T"],
                        ev["synthetic_P"],
                        cls,
                        1000 + cls,
                        DEVICES[0],
                        text_bank=TEXT_BANK,
                    )
                    evaluate(
                        "synthetic-game",
                        b2,
                        S2,
                        ["bicubic", "bilinear", "analytic_taau", "student_s2"],
                        {"motion": name},
                    )
            log(f"synthetic {name} done")
    for suite in ("tartanair", "sintel", "vid4", "vimeo1080p"):
        for cid, clip in REAL[suite]:
            b = E.sr_inputs(clip, S3, DEVICES[0], raft=RAFT)
            evaluate(suite, b, S3, METHODS_S3, {"clip": cid, "motion": "real"})
            if "student_s2" in MODELS:
                b2 = E.sr_inputs(clip, S2, DEVICES[0], raft=RAFT)
                evaluate(
                    suite,
                    b2,
                    S2,
                    ["bicubic", "bilinear", "analytic_taau", "student_s2"],
                    {"clip": cid, "motion": "real"},
                )
        log(f"{suite} done")
    for cid, clip in REAL["gameir_native"]:
        b2 = E.sr_inputs(clip, S2, DEVICES[0], raft=RAFT, native=True)
        evaluate(
            "gameir-native",
            b2,
            S2,
            ["bicubic", "bilinear", "analytic_taau", "student_s2"],
            {"clip": cid, "motion": "real"},
        )
RES = pd.DataFrame(ROWS)
METRIC_COLS = [
    "psnr_y",
    "ssim_y",
    "psnr_disocc",
    "psnr_detail",
    "psnr_particles",
    "lpips",
    "warp_err",
    "temporal_err",
    "tof",
    "mv_px",
]
for (
    col
) in (
    METRIC_COLS
):  # metrics that were not computable are None: make every metric column numeric (NaN)
    RES[col] = pd.to_numeric(RES[col], errors="coerce")
RUN.save_csv(RES, "test_per_frame.csv", index=False)
print(len(RES), "metric rows")

## 21. Results by suite and method

Mean of every metric per suite, scale and method (lower is better for LPIPS, warping, temporal-change error and tOF). Expected output: the results table and a chart of PSNR-Y against temporal-change error (top-left is better: sharp and stable).

In [ ]:
AGG = (
    RES.groupby(["suite", "scale", "method"])
    .mean(numeric_only=True)
    .drop(columns=["sample", "frame", "speed_class"], errors="ignore")
)
RUN.save_csv(AGG.round(4), "test_results.csv")
print(AGG.round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, scale in zip(axes, (S3, S2)):
    sub = (
        AGG.xs(scale, level="scale")
        if scale in AGG.index.get_level_values("scale")
        else None
    )
    if sub is None:
        continue
    for (suite, method), r in sub.iterrows():
        ax.scatter(r["temporal_err"], r["psnr_y"], s=30)
        ax.annotate(f"{method}\n{suite}", (r["temporal_err"], r["psnr_y"]), fontsize=6)
    ax.set_xlabel("temporal-change error (lower is more stable)")
    ax.set_ylabel("PSNR-Y (dB)")
    ax.set_title(f"{scale}x")
RUN.show(fig, "quality_vs_stability")

## 22. Motion classes, regions and perceptual metrics (synthetic-game, 3x)

How quality changes from static to fast motion, and quality inside the hardest regions: disocclusions (no valid history), fine detail (top 15 percent gradients: thin geometry, foliage, text) and particles (transparent effects without motion vectors). Expected output: three grouped bar charts.

In [ ]:
syn = RES[(RES["suite"] == "synthetic-game") & (RES["scale"] == S3)]
if len(syn):
    print(
        "measured motion per class (output px / frame):",
        syn.groupby("motion")["mv_px"].mean().round(2).to_dict(),
    )
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
    syn.groupby(["motion", "method"])["psnr_y"].mean().unstack().reindex(
        S.MOTION_PROFILE["names"]
    ).plot.bar(ax=axes[0], rot=0)
    axes[0].set_title("PSNR-Y by motion class")
    syn.groupby("method")[
        ["psnr_disocc", "psnr_detail", "psnr_particles"]
    ].mean().plot.bar(ax=axes[1], rot=30)
    axes[1].set_title("PSNR-Y in hard regions")
    syn.groupby("method")[["lpips", "tof"]].mean().plot.bar(ax=axes[2], rot=30)
    axes[2].set_title("LPIPS and tOF (lower is better)")
    for ax in axes:
        ax.legend(fontsize=7)
    RUN.show(fig, "motion_regions")

## 23. Static-scene convergence

A static camera with only sub-pixel jitter: a temporal upscaler should accumulate detail frame after frame and approach the native image, while single-frame methods stay flat. Expected output: PSNR-Y against frame index per method, and FLIP for the last frame.

In [ ]:
if TEST_STILLS is not None:
    sub = TEST_STILLS[: ev["synthetic_per_class"]]
    b = E.synthetic_suite(
        sub, S3, max(ev["synthetic_T"], 16), ev["synthetic_P"], 0, 4242, DEVICES[0]
    )
    conv, flip_rows = {}, []
    for mname in [
        "bilinear",
        "analytic_taau",
        "student_s3",
        "igpu_s3",
        "teacher_s3",
        "abl_no_history",
    ]:
        if mname in ("bilinear", "analytic_taau"):
            ref_net = MODELS["student_s3"][
                0
            ]  # only its parameter-free jitter-aware upsampler is used
            out = (
                E.run_bilinear(ref_net, b)
                if mname == "bilinear"
                else E.run_taau(ref_net, b, S3)
            )
        elif mname in MODELS:
            out = E.run_tsr(
                MODELS[mname][0], b, use_history=(mname != "abl_no_history")
            )
        else:
            continue
        conv[mname] = [
            M.psnr(M.rgb_to_y(out[:, t]), M.rgb_to_y(b["hr"][:, t]), border=S3)
            .mean()
            .item()
            for t in range(out.shape[1])
        ]
        fe = M.flip_error(b["hr"][0, -1], out[0, -1])
        flip_rows.append(
            {"method": mname, "flip_last_frame": fe, "psnr_y_last": conv[mname][-1]}
        )
    fig, ax = plt.subplots(figsize=(9, 4))
    for k, v in conv.items():
        ax.plot(v, label=k)
    ax.set_xlabel("frame")
    ax.set_ylabel("PSNR-Y (dB)")
    ax.set_title("Static camera with jitter: convergence toward native")
    ax.legend()
    RUN.show(fig, "static_convergence")
    FLIP_STATIC = pd.DataFrame(flip_rows)
    RUN.save_csv(FLIP_STATIC, "static_convergence.csv", index=False)
    print(FLIP_STATIC.round(4).to_string(index=False))

## 24. Visual comparisons, history weights and temporal profiles

For one fast-motion synthetic sequence and one TartanAir clip: crops of the last frame for each method, the student's learned history weight `a` (white = history kept, black = history rejected, which should light up at disocclusions and particles), and an x-t slice (one image row over time) where flicker and ghosting appear as streaks. Expected output: two figures; full frames are saved to `predictions/`.

In [ ]:
def visual(b, title, fname, crop=160):
    outs = {
        "bilinear": E.run_bilinear(MODELS["student_s3"][0], b),
        "analytic_taau": E.run_taau(MODELS["student_s3"][0], b, S3),
    }
    for k in ("igpu_s3", "student_s3", "teacher_s3"):
        if k in MODELS:
            outs[k] = E.run_tsr(MODELS[k][0], b)
    _, alpha = E.run_tsr(MODELS["student_s3"][0], b, keep_alpha=True)
    outs["ground truth"] = b["hr"]
    H, W = b["hr"].shape[-2:]
    y0, x0 = (H - crop) // 2, (W - crop) // 2
    fig, axes = plt.subplots(2, len(outs) + 1, figsize=(3 * (len(outs) + 1), 6.4))
    row = H // 2
    for c, (k, o) in enumerate(outs.items()):
        axes[0, c].imshow(
            o[0, -1, :, y0 : y0 + crop, x0 : x0 + crop].permute(1, 2, 0).cpu().numpy()
        )
        axes[0, c].set_title(k, fontsize=8)
        axes[1, c].imshow(
            o[0, :, :, row, :].permute(0, 2, 1).cpu().numpy(), aspect="auto"
        )  # time down, x across
        axes[1, c].set_title("x-t slice", fontsize=8)
        import PIL.Image as PImage

        PImage.fromarray(
            (o[0, -1].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        ).save(RUN.dirs["predictions"] / f"{fname}_{k.replace(' ', '_')}.png")
    axes[0, -1].imshow(
        alpha[0, -1, 0, y0 : y0 + crop, x0 : x0 + crop].cpu().numpy(),
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[0, -1].set_title("history weight a", fontsize=8)
    axes[1, -1].imshow(
        1 - b["valid_hr"][0, -1, 0, y0 : y0 + crop, x0 : x0 + crop].cpu().numpy(),
        cmap="gray",
    )
    axes[1, -1].set_title("true disocclusion", fontsize=8)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(title)
    RUN.show(fig, fname)


if TEST_STILLS is not None:
    visual(
        E.synthetic_suite(
            TEST_STILLS[-2:],
            S3,
            12,
            ev["synthetic_P"],
            3,
            77,
            DEVICES[0],
            text_bank=TEXT_BANK,
        ),
        "synthetic, fast motion",
        "visual_synthetic_fast",
    )
if REAL["tartanair"]:
    visual(
        E.sr_inputs(REAL["tartanair"][0][1], S3, DEVICES[0]),
        "TartanAir held-out environment",
        "visual_tartanair",
    )

## 25. Ablations

Same short budget, one change each, on the synthetic-game and TartanAir suites. The difference to `abl_reference` isolates the value of history, G-buffers, render-like training degradation, the temporal loss and distillation. Expected output: a table of deltas and a chart.

In [ ]:
abl = RES[
    RES["method"].str.startswith("abl_")
    & RES["suite"].isin(["synthetic-game", "tartanair"])
    & (RES["scale"] == S3)
]
if len(abl):
    ABL = abl.groupby(["suite", "method"])[
        ["psnr_y", "ssim_y", "lpips", "temporal_err", "psnr_disocc"]
    ].mean()
    ref = ABL.xs("abl_reference", level="method")
    delta = ABL.sub(ref, level="suite")
    RUN.save_csv(ABL.round(4), "ablations.csv")
    print(
        ABL.round(3).to_string(), "\n\ndelta to reference\n", delta.round(3).to_string()
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    delta["psnr_y"].unstack(0).plot.bar(ax=axes[0], rot=30)
    axes[0].set_title("PSNR-Y change vs reference (dB)")
    delta["temporal_err"].unstack(0).plot.bar(ax=axes[1], rot=30)
    axes[1].set_title("temporal-change error change (negative is better)")
    RUN.show(fig, "ablations")

## 26. Latency of fused models

Times one fused FP16 step (including warps, jitter-aware upsampling and the final blend) on one T4 with CUDA events after warm-up, batch 1, for 640x360 to 1920x1080 (3x), 960x540 to 1920x1080 (2x) and 1280x720 to 3840x2160 (3x). PyTorch eager on a T4 ranks variants; in-engine latency on target GPUs needs the ONNX / DirectML path. The arithmetic estimate column converts multiply-accumulates to milliseconds for an assumed sustained FP16 throughput (a parameter, not a measurement). Expected output: latency table and chart.

In [ ]:
@torch.no_grad()
def time_step(net, scale, out_hw, iters=100):
    dev = DEVICES[0]
    h, w = out_hw[0] // scale, out_hw[1] // scale
    f = {
        "color": torch.rand(1, 3, h, w, device=dev).half(),
        "depth": torch.rand(1, 1, h, w, device=dev).half(),
        "mv": torch.zeros(1, 2, h, w, device=dev).half(),
        "reactive": torch.zeros(1, 1, h, w, device=dev).half(),
        "exposure": torch.zeros(1, device=dev),
        "jitter": torch.zeros(1, 2, device=dev),
        "has_depth": torch.ones(1, device=dev),
        "mv_quality": torch.ones(1, device=dev),
    }
    _, state = net(f, None)
    for _ in range(10):
        _, state = net(f, state)
    torch.cuda.synchronize(dev)
    times = []
    for _ in range(iters):
        a, b_ = torch.cuda.Event(enable_timing=True), torch.cuda.Event(
            enable_timing=True
        )
        a.record()
        _, state = net(f, state)
        b_.record()
        torch.cuda.synchronize(dev)
        times.append(a.elapsed_time(b_))
    return np.array(times)


lat_rows = []
SUSTAINED_TFLOPS = {
    "assumed iGPU (4 TFLOPS FP16 sustained)": 4.0,
    "assumed low-end dGPU (8 TFLOPS)": 8.0,
}
for key in ("igpu_s3", "student_s3", "student_s2", "teacher_s3"):
    if key not in MODELS:
        continue
    net, scale = MODELS[key]
    for out_hw in ((1080, 1920), (2160, 3840)):
        if scale == S2 and out_hw[0] > 1080:
            continue
        t_ms = time_step(net, scale, out_hw, iters=30 if QUICK_RUN else 100)
        tier = RUNS[key][1]
        gmac = (
            TIERS[(TIERS["tier"] == tier) & (TIERS["scale"] == scale)][
                "gmac_per_frame_1080p"
            ].iloc[0]
            * (out_hw[0] * out_hw[1])
            / (1920 * 1080)
        )
        lat_rows.append(
            {
                "model": key,
                "scale": scale,
                "output": f"{out_hw[1]}x{out_hw[0]}",
                "t4_mean_ms": t_ms.mean(),
                "t4_p95_ms": np.percentile(t_ms, 95),
                "gmac": gmac,
                **{
                    f"est_ms {k}": M.estimate_ms(gmac, v)
                    for k, v in SUSTAINED_TFLOPS.items()
                },
            }
        )
LAT = pd.DataFrame(lat_rows)
RUN.save_csv(LAT, "latency.csv", index=False)
print(LAT.round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 3.8))
LAT.pivot_table(index="model", columns="output", values="t4_mean_ms").plot.bar(
    ax=ax, rot=0
)
ax.set_ylabel("ms per frame (T4, PyTorch FP16)")
RUN.show(fig, "latency")

## 27. ONNX export

Exports one fused recurrent step of the main 3x student and the integrated-GPU student (explicit state tensors: previous output, hidden state, previous depth), opset 17, with static shapes (a deployment builds one optimised graph per render resolution; re-export with the target size), and compares ONNX Runtime (CPU) with PyTorch FP32. Expected output: export status and maximum difference.

In [ ]:
onnx_rows = []
for key in ("student_s3", "igpu_s3"):
    if key not in MODELS:
        continue
    net = copy.deepcopy(MODELS[key][0]).float().cpu().eval()
    wrap = NM.TSRExport(net)
    h, w = 36, 64
    s = MODELS[key][1]
    args = (
        torch.rand(1, 3, h, w),
        torch.rand(1, 1, h, w),
        torch.zeros(1, 2, h, w),
        torch.zeros(1, 1, h, w),
        torch.zeros(1),
        torch.zeros(1, 2),
        torch.ones(1, 2),
        torch.rand(1, 3, h * s, w * s),
        torch.zeros(1, net.hidden, h, w),
        torch.rand(1, 1, h, w),
    )
    names = [
        "color",
        "depth",
        "mv",
        "reactive",
        "exposure",
        "jitter",
        "flags",
        "prev_out",
        "prev_hidden",
        "prev_depth",
    ]
    path = RUN.dirs["weights"] / f"tsr_{key}_fused.onnx"
    row = {"model": key, "exported": False, "ort_max_abs": None}
    try:  # static shapes: engines build one optimised graph per render resolution anyway
        torch.onnx.export(
            wrap,
            args,
            str(path),
            input_names=names,
            output_names=["out", "hidden", "depth_out"],
            opset_version=17,
            dynamo=False,
        )
        row["exported"] = True
        row["onnx_mb"] = round(path.stat().st_size / 1e6, 3)
        import onnxruntime as ort

        sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
        ort_out = sess.run(None, {n: a.numpy() for n, a in zip(names, args)})[0]
        with torch.no_grad():
            ref_out = wrap(*args)[0].numpy()
        row["ort_max_abs"] = float(np.abs(ort_out - ref_out).max())
    except Exception as exc:
        row["error"] = repr(exc)[:300]
    onnx_rows.append(row)
print(pd.DataFrame(onnx_rows).to_string(index=False))
RUN.save_json(onnx_rows, "onnx_export.json")

## 28. Summary

Headline results of the main 3x student against the analytic TAA upscaler on the game-like suites, the integrated-GPU tier, the 2x student on native GameIR renders, latency, and stage times against the budget. Expected output: the summary table and a stage-time chart.

In [ ]:
def pick(suite, method, col, scale=S3):
    try:
        return round(float(AGG.loc[(suite, scale, method), col]), 4)
    except KeyError:
        return None


summary = {"run_hours": round(RUN.hours(), 2)}
for suite in ("synthetic-game", "tartanair", "sintel", "vid4", "vimeo1080p"):
    for m in ("analytic_taau", "student_s3", "igpu_s3"):
        summary[f"{suite}/{m}/psnr_y"] = pick(suite, m, "psnr_y")
        summary[f"{suite}/{m}/temporal_err"] = pick(suite, m, "temporal_err")
        summary[f"{suite}/{m}/lpips"] = pick(suite, m, "lpips")
for m in ("analytic_taau", "student_s2"):
    summary[f"gameir-native/{m}/psnr_y"] = pick("gameir-native", m, "psnr_y", S2)
if len(LAT):
    for _, r in LAT.iterrows():
        summary[f"latency_ms/{r['model']}/{r['output']}"] = round(r["t4_mean_ms"], 2)
RUN.save_json(summary, "summary.json")
print(pd.Series(summary).to_string())
st_t = pd.Series(RUN.stage_times) / 60
RUN.save_csv(st_t.rename("minutes").to_frame(), "stage_times.csv")
fig, ax = plt.subplots(figsize=(8, 3.8))
st_t.plot.barh(ax=ax)
ax.set_xlabel("minutes")
RUN.show(fig, "stage_times")

## 29. Remove training caches

The memory-mapped caches are intermediates; deleting them keeps the saved output small and frees disk for the zip. Expected output: the space freed.

In [ ]:
import shutil

freed = sum(p.stat().st_size for p in CACHE.rglob("*") if p.is_file())
shutil.rmtree(CACHE, ignore_errors=True)
print(f"removed caches: {freed / 1e9:.2f} GB")

## 30. Package outputs

Zips everything in `/kaggle/working` (weights, ONNX models, plots, metrics, logs, predictions, code and configs) into `outputs.zip`, storing already-compressed files without recompression, after a disk-space check. After **Save Version > Save & Run All**, the zip and all files also appear in the notebook's **Output** tab.

In [ ]:
import shutil
import zipfile

from IPython.display import FileLink
from tqdm.auto import tqdm

WORK = C.WORK
ZIP_PATH = WORK / "outputs.zip"
EXCLUDE_PARTS = {"__pycache__", ".ipynb_checkpoints", ".cache", "cache", "tmp", "wandb"}
STORED_SUFFIXES = {
    ".zip",
    ".png",
    ".jpg",
    ".jpeg",
    ".pt",
    ".pth",
    ".bin",
    ".safetensors",
    ".gz",
    ".npz",
    ".onnx",
    ".mp4",
    ".parquet",
}
files = [
    p
    for p in WORK.rglob("*")
    if p.is_file()
    and p != ZIP_PATH
    and not EXCLUDE_PARTS.intersection(p.relative_to(WORK).parts)
]
total_bytes = sum(p.stat().st_size for p in files)
free_bytes = shutil.disk_usage(WORK).free
assert (
    free_bytes > total_bytes * 1.05
), f"Not enough disk space to zip {total_bytes / 1e9:.2f} GB with {free_bytes / 1e9:.2f} GB free."
with zipfile.ZipFile(ZIP_PATH, "w", allowZip64=True) as archive:
    for p in tqdm(files, desc="Zipping outputs", unit="file"):
        method = (
            zipfile.ZIP_STORED
            if p.suffix.lower() in STORED_SUFFIXES
            else zipfile.ZIP_DEFLATED
        )
        archive.write(p, p.relative_to(WORK), compress_type=method)
print(
    f"Zipped {len(files)} files ({total_bytes / 1e9:.2f} GB) into {ZIP_PATH}",
    flush=True,
)
display(FileLink(os.path.relpath(ZIP_PATH)))